# 06 — Geração de Documentos Semânticos do SINAN

## Objetivo

Este notebook transforma os produtos analíticos consolidados no Notebook 05 em documentos semânticos destinados à recuperação de informação por mecanismos de RAG (*Retrieval-Augmented Generation*).

As análises epidemiológicas não são recalculadas nesta etapa. O notebook organiza os resultados previamente produzidos em um contrato documental comum, preservando proveniência, escopo temporal e geográfico, indicadores, evidências quantitativas, interpretação descritiva, observações sobre os dados e conceitos semânticos.

## Entrada

Produtos analíticos do SINAN gerados pelo Notebook 05.

## Saída

Documentos semânticos em JSON e Markdown, organizados nos domínios:

- clínico;
- geográfico;
- temporal;
- virológico;
- desfechos.

A etapa também produz um inventário final para verificação estrutural e rastreabilidade dos documentos gerados.


## Estrutura do notebook

1. Montagem do Google Drive e imports.
2. Configuração de caminhos e parâmetros.
3. Carregamento dos produtos analíticos do Notebook 05.
4. Funções auxiliares e contrato do documento semântico.
5. Geração dos documentos do domínio virológico.
6. Geração dos documentos do domínio temporal.
7. Geração dos documentos do domínio geográfico.
8. Geração dos documentos do domínio clínico.
9. Geração dos documentos de desfechos.
10. Verificação estrutural, inventário e auditoria final.


## 0 - Montagem do drive

In [51]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Imports


In [52]:
import json
import re

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)


## 2. Configuração


In [53]:
# ============================================================
# PARÂMETROS GERAIS
# ============================================================

ANO = 2026

PASTA_BASE = Path(
    "/content/drive/MyDrive/Doutorado/arbovirus_rag"
)

PASTA_ANALYTICS = (
    PASTA_BASE
    / "data_analytics"
    / "SINAN"
    / str(ANO)
)

PASTA_DOCS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
)

PASTA_DOCS.mkdir(
    parents=True,
    exist_ok=True
)

# As chaves abaixo refletem os nomes físicos das pastas
# produzidas pelo Notebook 05.
PASTAS_ANALISES = {
    "clinicas": PASTA_ANALYTICS / "clinicas",
    "desfechos": PASTA_ANALYTICS / "desfechos",
    "geograficos": PASTA_ANALYTICS / "geograficos",
    "temporais": PASTA_ANALYTICS / "temporais",
    "virologicas": PASTA_ANALYTICS / "virologicas",
}

# Vocabulário controlado usado nos documentos semânticos.
DOMINIOS_VALIDOS = {
    "clinico",
    "desfechos",
    "geografico",
    "temporal",
    "virologico",
}

TIPOS_DOCUMENTO_VALIDOS = {
    "panorama_clinico_nacional",
    "perfil_clinico_uf",
    "desfechos_uf",
    "panorama_desfechos_nacional",
    "panorama_obitos_nacional",
    "perfil_obitos_uf",
    "distribuicao_geografica_uf",
    "panorama_geografico_nacional",
    "panorama_temporal_nacional",
    "perfil_temporal_uf",
    "panorama_virologico_nacional",
    "perfil_virologico_uf",
}

print(f"Ano analisado: {ANO}")
print(f"Produtos analíticos: {PASTA_ANALYTICS}")
print(f"Documentos semânticos: {PASTA_DOCS}")

for nome, caminho in PASTAS_ANALISES.items():
    status = "OK" if caminho.exists() else "NÃO ENCONTRADA"
    print(f"{nome:<15} -> {status}")


Ano analisado: 2026
Produtos analíticos: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026
Documentos semânticos: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026
clinicas        -> OK
desfechos       -> OK
geograficos     -> OK
temporais       -> OK
virologicas     -> OK


## 3. Carregamento dos resultados analíticos

As análises epidemiológicas não são recalculadas neste notebook. Os produtos do Notebook 05 são apenas carregados e transformados em documentos semânticos.


### 3.1 Carregar os produtos analíticos


In [54]:
# ============================================================
# FUNÇÃO PARA CARREGAR RESULTADOS ANALÍTICOS
# ============================================================

def carregar_resultados_analiticos(pastas_analises):

    resultados = {}

    for dominio, pasta in pastas_analises.items():

        resultados[dominio] = {}

        if not pasta.exists():
            print(
                f"[AVISO] Pasta não encontrada: {pasta}"
            )
            continue

        arquivos = sorted(
            arquivo
            for arquivo in pasta.iterdir()
            if arquivo.is_file()
        )

        for arquivo in arquivos:

            try:

                if arquivo.suffix.lower() == ".parquet":

                    df = pd.read_parquet(
                        arquivo
                    )

                elif arquivo.suffix.lower() == ".csv":

                    df = pd.read_csv(
                        arquivo
                    )

                else:
                    continue

                resultados[dominio][
                    arquivo.stem
                ] = df

                print(
                    f"[OK] {dominio}/{arquivo.name} "
                    f"-> {df.shape[0]:,} linhas | "
                    f"{df.shape[1]} colunas"
                )

            except Exception as erro:

                print(
                    f"[ERRO] {arquivo.name}: "
                    f"{erro}"
                )

    return resultados

In [55]:
# ============================================================
# CARREGAR RESULTADOS DO NOTEBOOK 05
# ============================================================

resultados_analytics = (
    carregar_resultados_analiticos(
        PASTAS_ANALISES
    )
)

# ============================================================
# TOTAL DE RESULTADOS CARREGADOS
# ============================================================

total_carregados = sum(
    len(arquivos)
    for arquivos in resultados_analytics.values()
)

print("\n" + "=" * 70)
print("RESUMO DO CARREGAMENTO")
print("=" * 70)

print(
    f"Total de arquivos analíticos carregados: "
    f"{total_carregados}"
)

[OK] clinicas/doencas_preexistentes_brasil_2026.parquet -> 7 linhas | 9 colunas
[OK] clinicas/doencas_preexistentes_uf_2026.parquet -> 189 linhas | 11 colunas
[OK] clinicas/inventario_produtos_clinicos_2026.csv -> 4 linhas | 4 colunas
[OK] clinicas/sintomas_brasil_2026.parquet -> 14 linhas | 9 colunas
[OK] clinicas/sintomas_uf_2026.parquet -> 378 linhas | 11 colunas
[OK] desfechos/doencas_preexistentes_obitos_brasil.parquet -> 7 linhas | 9 colunas
[OK] desfechos/doencas_preexistentes_obitos_uf.parquet -> 182 linhas | 11 colunas
[OK] desfechos/evolucao_brasil.parquet -> 6 linhas | 4 colunas
[OK] desfechos/evolucao_uf.parquet -> 142 linhas | 9 colunas
[OK] desfechos/hospitalizacao_brasil.parquet -> 1 linhas | 11 colunas
[OK] desfechos/hospitalizacao_uf.parquet -> 27 linhas | 12 colunas
[OK] desfechos/inventario_produtos_desfechos.csv -> 13 linhas | 6 colunas
[OK] desfechos/obitos_brasil.parquet -> 3 linhas | 3 colunas
[OK] desfechos/obitos_sorotipo_brasil.parquet -> 4 linhas | 4 colunas


In [56]:
# ============================================================
# CRIAR DATAFRAMES ANALÍTICOS COM NOMES INDEPENDENTES DO ANO
# ============================================================

import re

dataframes_analytics = {}

for dominio, resultados in resultados_analytics.items():

    for nome_arquivo, df in resultados.items():

        if nome_arquivo.startswith(
            "inventario_produtos"
        ):
            continue

        # ----------------------------------------------------
        # Remove o ano apenas do final do nome
        #
        # Exemplo:
        # sintomas_brasil_2026
        # ->
        # sintomas_brasil
        # ----------------------------------------------------

        nome_logico = re.sub(
            rf"_{ANO}$",
            "",
            nome_arquivo
        )

        nome_variavel = (
            f"df_{nome_logico}"
        )

        globals()[nome_variavel] = df

        dataframes_analytics[
            nome_variavel
        ] = {
            "ano": ANO,
            "dominio": dominio,
            "produto": nome_logico,
            "arquivo_origem": nome_arquivo,
            "dataframe": df
        }

print(
    f"DataFrames criados para {ANO}: "
    f"{len(dataframes_analytics)}"
)

DataFrames criados para 2026: 36


### 3.2 Verificar os produtos carregados


In [57]:
# ============================================================
# RESUMO DOS RESULTADOS CARREGADOS
# ============================================================

total_dataframes = 0

for dominio, arquivos in resultados_analytics.items():

    print("\n" + "=" * 70)
    print(dominio.upper())
    print("=" * 70)

    for nome_df, df in arquivos.items():

        print(
            f"{nome_df:<40} "
            f"{df.shape[0]:>10,} linhas | "
            f"{df.shape[1]:>3} colunas"
        )

        total_dataframes += 1

print("\n" + "=" * 70)
print("RESUMO")
print("=" * 70)

print(
    f"Total de DataFrames carregados: "
    f"{total_dataframes}"
)


CLINICAS
doencas_preexistentes_brasil_2026                 7 linhas |   9 colunas
doencas_preexistentes_uf_2026                   189 linhas |  11 colunas
inventario_produtos_clinicos_2026                 4 linhas |   4 colunas
sintomas_brasil_2026                             14 linhas |   9 colunas
sintomas_uf_2026                                378 linhas |  11 colunas

DESFECHOS
doencas_preexistentes_obitos_brasil               7 linhas |   9 colunas
doencas_preexistentes_obitos_uf                 182 linhas |  11 colunas
evolucao_brasil                                   6 linhas |   4 colunas
evolucao_uf                                     142 linhas |   9 colunas
hospitalizacao_brasil                             1 linhas |  11 colunas
hospitalizacao_uf                                27 linhas |  12 colunas
inventario_produtos_desfechos                    13 linhas |   6 colunas
obitos_brasil                                     3 linhas |   3 colunas
obitos_sorotipo_brasil        

## 4. Funções auxiliares e contrato documental


Essas funções devem cuidar de tarefas genéricas e repetitivas, por exemplo:

converter tipos numpy para tipos serializáveis em JSON;
tratar NaN, NaT e valores ausentes;
normalizar listas e dicionários;
gerar IDs padronizados para os documentos;
converter o objeto semântico para Markdown;
salvar JSON;
salvar Markdown.

Começar com estas duas funções básicas:

In [58]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def converter_json(obj):
    """
    Converte tipos NumPy/Pandas para tipos compatíveis com JSON.
    """

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.bool_):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    if pd.isna(obj):
        return None

    raise TypeError(
        f"Objeto do tipo {type(obj).__name__} "
        f"não é serializável em JSON"
    )

E uma função para normalizar valores antes de montar os documentos:

In [59]:
def normalizar_valor(valor):
    """
    Converte valores ausentes e tipos NumPy/Pandas
    para tipos Python simples.
    """

    if valor is None:
        return None

    if isinstance(valor, (list, dict)):
        return valor

    try:
        if pd.isna(valor):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(valor, np.integer):
        return int(valor)

    if isinstance(valor, np.floating):
        return float(valor)

    if isinstance(valor, np.bool_):
        return bool(valor)

    if isinstance(valor, pd.Timestamp):
        return valor.isoformat()

    return valor

função para o ID dos documentos:

In [60]:
def gerar_document_id(
    fonte,
    doenca,
    ano,
    dominio,
    localizacao=None,
    complemento=None
):
    """
    Gera um identificador padronizado para documentos semânticos.
    """

    partes = [
        str(fonte),
        str(doenca),
        str(ano),
        str(dominio)
    ]

    if localizacao:
        partes.append(
            str(localizacao)
        )

    if complemento:
        partes.append(
            str(complemento)
        )

    document_id = "_".join(partes)

    document_id = (
        document_id
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
    )

    return document_id

Por exemplo:

```
gerar_document_id(
    fonte="SINAN",
    doenca="DENGUE",
    ano=2026,
    dominio="SOROTIPOS",
    localizacao="SP"
)
```
retornaria:

SINAN_DENGUE_2026_SOROTIPOS_SP

Não criar a função documento_para_markdown() neste ponto. Antes dela, iremos definir formalmente qual será a estrutura padrão do objeto semântico. A sequência fica:
```
Carregar resultados do Notebook 05
            ↓
Verificar estrutura
            ↓
Funções auxiliares básicas
            ↓
Definir estrutura padrão
do documento semântico
            ↓
Criar função
documento_para_markdown()
            ↓
Criar funções de salvamento
            ↓
Gerar documentos por domínio
```

### 4.1 Estrutura padrão dos documentos semânticos


In [61]:
# ============================================================
# ESTRUTURA PADRÃO DO DOCUMENTO SEMÂNTICO
# ============================================================

def criar_documento_semantico(
    document_id,
    titulo,
    tipo_documento,
    dominio,
    sintese,
    indicadores,
    evidencias,
    interpretacao,
    escopo,
    observacoes_dados=None,
    conceitos_semanticos=None,
    palavras_chave=None,
    fonte="SINAN",
    doenca="dengue",
    ano=ANO
):
    """
    Cria um documento semântico seguindo a estrutura
    padronizada adotada no pipeline.
    """

    documento = {

        # ----------------------------------------------------
        # IDENTIFICAÇÃO
        # ----------------------------------------------------
        "document_id": document_id,
        "titulo": titulo,
        "tipo_documento": tipo_documento,
        "dominio": dominio,

        # ----------------------------------------------------
        # PROVENIÊNCIA
        # ----------------------------------------------------
        "fonte": {
            "sistema": fonte,
            "doenca": doenca,
            "ano": ano
        },

        # ----------------------------------------------------
        # ESCOPO
        # ----------------------------------------------------
        "escopo": escopo,

        # ----------------------------------------------------
        # CONTEÚDO
        # ----------------------------------------------------
        "sintese": sintese,

        "indicadores": indicadores,

        "evidencias": evidencias,

        "interpretacao": interpretacao,

        # ----------------------------------------------------
        # QUALIDADE / LIMITAÇÕES
        # ----------------------------------------------------
        "observacoes_dados": (
            observacoes_dados
            if observacoes_dados is not None
            else []
        ),

        # ----------------------------------------------------
        # CAMADA SEMÂNTICA
        # ----------------------------------------------------
        "conceitos_semanticos": (
            conceitos_semanticos
            if conceitos_semanticos is not None
            else []
        ),

        # ----------------------------------------------------
        # RECUPERAÇÃO TEXTUAL
        # ----------------------------------------------------
        "palavras_chave": (
            palavras_chave
            if palavras_chave is not None
            else []
        )
    }

    return documento

Com isso, temos quatro camadas muito claras:

```
DADOS
├── indicadores
└── evidencias

        ↓

CONTEÚDO TEXTUAL
├── sintese
└── interpretacao

        ↓

CONTEXTO
├── fonte
├── escopo
└── observacoes_dados

        ↓

SEMÂNTICA / RECUPERAÇÃO
├── conceitos_semanticos
└── palavras_chave
```

Essa separação vai ser útil mais adiante porque podemos decidir quais partes entram efetivamente no texto enviado para embeddings. Não precisamos necessariamente indexar o Markdown inteiro.

Por exemplo, poderemos construir o texto_indexacao principalmente a partir de:

Título
+ Síntese
+ Indicadores
+ Evidências
+ Interpretação
+ localização/período

enquanto document_id, proveniência e conceitos semânticos podem permanecer como metadados do documento no banco vetorial.

Isso nos dará um controle experimental interessante quando chegarmos à etapa de recuperação.

### 4.2 Conversão para Markdown


In [62]:
# ============================================================
# FORMATAR EVIDÊNCIAS PARA MARKDOWN
# ============================================================

def formatar_evidencias_markdown(
    evidencias,
    nivel=3
):
    """
    Converte estruturas de evidências em Markdown.

    Suporta:
    - listas;
    - dicionários;
    - estruturas hierárquicas;
    - valores simples.
    """

    linhas = []

    if evidencias is None:
        return [
            "Nenhuma evidência disponível."
        ]

    # --------------------------------------------------------
    # DICIONÁRIO
    # --------------------------------------------------------

    if isinstance(evidencias, dict):

        if not evidencias:
            return [
                "Nenhuma evidência disponível."
            ]

        for chave, valor in evidencias.items():

            titulo = (
                chave
                .replace("_", " ")
                .capitalize()
            )

            linhas.append(
                f"{'#' * nivel} {titulo}\n"
            )

            linhas.extend(
                formatar_evidencias_markdown(
                    valor,
                    nivel=nivel + 1
                )
            )

    # --------------------------------------------------------
    # LISTA
    # --------------------------------------------------------

    elif isinstance(evidencias, list):

        if not evidencias:
            return [
                "Nenhuma evidência disponível."
            ]

        for evidencia in evidencias:

            if isinstance(evidencia, dict):

                partes = []

                for chave, valor in evidencia.items():

                    nome_campo = (
                        chave
                        .replace("_", " ")
                        .capitalize()
                    )

                    partes.append(
                        f"**{nome_campo}:** "
                        f"{valor}"
                    )

                linhas.append(
                    "- " + " | ".join(partes)
                )

            elif isinstance(
                evidencia,
                list
            ):

                linhas.extend(
                    formatar_evidencias_markdown(
                        evidencia,
                        nivel=nivel
                    )
                )

            else:

                linhas.append(
                    f"- {evidencia}"
                )

    # --------------------------------------------------------
    # VALOR SIMPLES
    # --------------------------------------------------------

    else:

        linhas.append(
            str(evidencias)
        )

    return linhas

In [63]:
# ============================================================
# CONVERTER DOCUMENTO SEMÂNTICO PARA MARKDOWN
# ============================================================

def documento_para_markdown(documento):
    """
    Converte um documento semântico padronizado
    para uma representação textual em Markdown.
    """

    linhas = []

    # ========================================================
    # TÍTULO
    # ========================================================

    linhas.append(
        f"# {documento['titulo']}"
    )

    # ========================================================
    # IDENTIFICAÇÃO
    # ========================================================

    linhas.append("\n## Identificação\n")

    linhas.append(
        f"- **ID do documento:** "
        f"{documento['document_id']}"
    )

    linhas.append(
        f"- **Tipo de documento:** "
        f"{documento['tipo_documento']}"
    )

    linhas.append(
        f"- **Domínio:** "
        f"{documento['dominio']}"
    )

    fonte = documento.get(
        "fonte",
        {}
    )

    linhas.append(
        f"- **Fonte:** "
        f"{fonte.get('sistema', 'Não informado')}"
    )

    linhas.append(
        f"- **Doença:** "
        f"{fonte.get('doenca', 'Não informado')}"
    )

    linhas.append(
        f"- **Ano:** "
        f"{fonte.get('ano', 'Não informado')}"
    )

    # ========================================================
    # ESCOPO
    # ========================================================

    linhas.append("\n## Escopo\n")

    escopo = documento.get(
        "escopo",
        {}
    )

    if escopo:

        for chave, valor in escopo.items():

            if valor is None:
                continue

            nome_campo = (
                chave
                .replace("_", " ")
                .capitalize()
            )

            if isinstance(valor, list):

                valor_formatado = ", ".join(
                    str(item)
                    for item in valor
                )

            else:

                valor_formatado = valor

            linhas.append(
                f"- **{nome_campo}:** "
                f"{valor_formatado}"
            )

    else:

        linhas.append(
            "Escopo não informado."
        )

    # ========================================================
    # SÍNTESE
    # ========================================================

    linhas.append(
        "\n## Síntese epidemiológica\n"
    )

    linhas.append(
        str(
            documento.get(
                "sintese",
                "Não informado."
            )
        )
    )

    # ========================================================
    # INDICADORES
    # ========================================================

    linhas.append(
        "\n## Indicadores\n"
    )

    indicadores = documento.get(
        "indicadores",
        {}
    )

    if indicadores:

        for chave, valor in indicadores.items():

            nome_indicador = (
                chave
                .replace("_", " ")
                .capitalize()
            )

            linhas.append(
                f"- **{nome_indicador}:** "
                f"{valor}"
            )

    else:

        linhas.append(
            "Nenhum indicador disponível."
        )



    # ------------------------------------------------------------
    # EVIDÊNCIAS
    # ------------------------------------------------------------

    linhas.append(
        "\n## Evidências\n"
    )

    evidencias = documento.get(
        "evidencias",
        []
    )

    linhas.extend(
        formatar_evidencias_markdown(
            evidencias,
            nivel=3
        )
    )

    # ========================================================
    # INTERPRETAÇÃO
    # ========================================================

    linhas.append(
        "\n## Interpretação dos resultados\n"
    )

    interpretacao = documento.get(
        "interpretacao"
    )

    if interpretacao:

        linhas.append(
            str(interpretacao)
        )

    else:

        linhas.append(
            "Nenhuma interpretação disponível."
        )

    # ========================================================
    # OBSERVAÇÕES SOBRE OS DADOS
    # ========================================================

    linhas.append(
        "\n## Observações sobre os dados\n"
    )

    observacoes = documento.get(
        "observacoes_dados",
        []
    )

    if observacoes:

        for observacao in observacoes:

            linhas.append(
                f"- {observacao}"
            )

    else:

        linhas.append(
            "Nenhuma observação adicional."
        )

    # ========================================================
    # CONCEITOS SEMÂNTICOS
    # ========================================================

    linhas.append(
        "\n## Conceitos semânticos\n"
    )

    conceitos = documento.get(
        "conceitos_semanticos",
        []
    )

    if conceitos:

        for conceito in conceitos:

            if isinstance(
                conceito,
                dict
            ):

                nome = conceito.get(
                    "conceito",
                    ""
                )

                classe = conceito.get(
                    "classe"
                )

                if classe:

                    linhas.append(
                        f"- {nome} (`{classe}`)"
                    )

                else:

                    linhas.append(
                        f"- {nome}"
                    )

            else:

                linhas.append(
                    f"- {conceito}"
                )

    else:

        linhas.append(
            "Nenhum conceito semântico informado."
        )

    # ========================================================
    # PALAVRAS-CHAVE
    # ========================================================

    linhas.append(
        "\n## Palavras-chave\n"
    )

    palavras_chave = documento.get(
        "palavras_chave",
        []
    )

    if palavras_chave:

        linhas.append(
            ", ".join(
                str(palavra)
                for palavra in palavras_chave
            )
        )

    else:

        linhas.append(
            "Nenhuma palavra-chave informada."
        )

    # ========================================================
    # RESULTADO FINAL
    # ========================================================

    markdown = "\n".join(
        linhas
    )

    return markdown

In [64]:
# ============================================================
# SALVAR DOCUMENTO EM JSON
# ============================================================

def salvar_documento_json(
    documento,
    pasta_saida
):
    """
    Salva um documento semântico em formato JSON.
    """

    pasta_saida = Path(
        pasta_saida
    )

    pasta_saida.mkdir(
        parents=True,
        exist_ok=True
    )

    nome_arquivo = (
        documento["document_id"]
        .lower()
        + ".json"
    )

    caminho_arquivo = (
        pasta_saida
        / nome_arquivo
    )

    with open(
        caminho_arquivo,
        "w",
        encoding="utf-8"
    ) as arquivo:

        json.dump(
            documento,
            arquivo,
            ensure_ascii=False,
            indent=2,
            default=converter_json
        )

    return caminho_arquivo

E para Markdown:

In [65]:
# ============================================================
# SALVAR DOCUMENTO EM MARKDOWN
# ============================================================

def salvar_documento_markdown(
    documento,
    pasta_saida
):
    """
    Converte o documento semântico para Markdown
    e salva o arquivo.
    """

    pasta_saida = Path(
        pasta_saida
    )

    pasta_saida.mkdir(
        parents=True,
        exist_ok=True
    )

    markdown = documento_para_markdown(
        documento
    )

    nome_arquivo = (
        documento["document_id"]
        .lower()
        + ".md"
    )

    caminho_arquivo = (
        pasta_saida
        / nome_arquivo
    )

    with open(
        caminho_arquivo,
        "w",
        encoding="utf-8"
    ) as arquivo:

        arquivo.write(
            markdown
        )

    return caminho_arquivo

Depois criar uma função única que salve os dois formatos:

In [66]:
# ============================================================
# SALVAR DOCUMENTO SEMÂNTICO
# ============================================================

def salvar_documento_semantico(
    documento,
    pasta_saida
):
    """
    Salva o documento semântico nos formatos
    JSON e Markdown.
    """

    caminho_json = salvar_documento_json(
        documento,
        pasta_saida
    )

    caminho_markdown = salvar_documento_markdown(
        documento,
        pasta_saida
    )

    return {
        "json": caminho_json,
        "markdown": caminho_markdown
    }

## 5. Domínio virológico


In [67]:
# ============================================================
# ESCOPO TEMPORAL GLOBAL DOS DOCUMENTOS SEMÂNTICOS
# ============================================================

SEMANA_INICIAL = int(
    df_resumo_temporal_brasil.iloc[0][
        "SEMANA_INICIAL"
    ]
)

SEMANA_FINAL = int(
    df_resumo_temporal_brasil.iloc[0][
        "SEMANA_FINAL"
    ]
)

TOTAL_SEMANAS = (
    SEMANA_FINAL
    - SEMANA_INICIAL
    + 1
)

print(
    f"Escopo temporal dos documentos semânticos: "
    f"SE {SEMANA_INICIAL} a SE {SEMANA_FINAL} "
    f"de {ANO}"
)

Escopo temporal dos documentos semânticos: SE 1 a SE 34 de 2026


O objetivo agora é construir um documento semântico por UF, reunindo no mesmo documento:

distribuição dos sorotipos;
total de registros com sorotipo informado;
completude da variável;
evidências quantitativas;
uma interpretação descritiva;
observações sobre limitações dos dados.
```
A estrutura ficará aproximadamente assim:

UF
 ├── distribuição dos sorotipos
 ├── completude do sorotipo
 ├── indicadores
 ├── evidências
 ├── interpretação
 └── observações de qualidade
 ```

Isso é melhor do que gerar um documento apenas com a distribuição de sorotipos, porque uma UF pode aparentar ter predominância de um sorotipo enquanto, na realidade, a variável possui baixa cobertura.

Depois dessa inspeção, podemos montar a função:
```
def criar_documentos_sorotipos_por_uf(
    df_sorotipos,
    df_completude
):
    ...
```

Ela retornará uma lista:

documentos_sorotipos_uf

com um documento para cada UF.

O ponto importante é: não vamos recalcular a análise do SINAN. Vamos apenas combinar e transformar os resultados analíticos já produzidos no Notebook 05.

In [68]:
# ============================================================
# INTERPRETAÇÃO DA DISTRIBUIÇÃO DE SOROTIPOS
# ============================================================

def interpretar_sorotipos_uf(
    uf_nome,
    sorotipo_informado,
    percentual_informado,
    evidencias
):
    """
    Gera uma interpretação descritiva e determinística
    da distribuição de sorotipos de uma UF.

    A função não realiza inferências epidemiológicas causais.
    """

    # --------------------------------------------------------
    # SEM INFORMAÇÃO DE SOROTIPO
    # --------------------------------------------------------

    if sorotipo_informado == 0 or not evidencias:

        return (
            f"Não foram identificados registros com informação "
            f"de sorotipo para {uf_nome} no período analisado. "
            f"Assim, não é possível caracterizar a distribuição "
            f"dos sorotipos nessa UF."
        )

    # --------------------------------------------------------
    # IDENTIFICAR SOROTIPO MAIS FREQUENTE
    # --------------------------------------------------------

    evidencia_principal = max(
        evidencias,
        key=lambda x: x["total_casos"]
    )

    sorotipo_principal = (
        evidencia_principal["sorotipo"]
    )

    total_principal = (
        evidencia_principal["total_casos"]
    )

    # --------------------------------------------------------
    # INTERPRETAÇÃO
    # --------------------------------------------------------

    return (
        f"Entre os {sorotipo_informado:,} registros com "
        f"informação de sorotipo em {uf_nome}, "
        f"{sorotipo_principal} apresentou a maior frequência, "
        f"com {total_principal:,} registros. "
        f"A informação de sorotipo estava disponível em "
        f"{percentual_informado:.2f}% dos registros da UF."
    )

Observe que evitamos escrever algo como:

“DENV-2 predominou epidemiologicamente no estado.”

Preferimos:

“DENV-2 apresentou a maior frequência entre os registros com informação de sorotipo.”

Isso é mais defensável, especialmente diante da elevada incompletude dessa variável.

## 2. Criar a função para gerar os documentos por UF

Agora podemos usar nossa função criar_documento_semantico() já definida anteriormente.

In [69]:
def criar_documentos_sorotipos_por_uf(
    df_sorotipos,
    df_completude,
    semana_inicial,
    semana_final,
    ano=ANO
):

    documentos = []

    # ========================================================
    # UNIVERSO DE UFs
    # Usamos a tabela de completude para preservar UFs
    # mesmo quando não há sorotipo informado
    # ========================================================

    ufs = (
        df_completude[
            [
                "SG_UF_NOT",
                "UF_NAME"
            ]
        ]
        .drop_duplicates()
        .sort_values("SG_UF_NOT")
    )

    for _, linha_uf in ufs.iterrows():

        codigo_uf = str(
            linha_uf["SG_UF_NOT"]
        )

        nome_uf = linha_uf[
            "UF_NAME"
        ]

        # ====================================================
        # COMPLETUDE DO SOROTIPO
        # ====================================================

        dados_completude = (
            df_completude[
                df_completude["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .iloc[0]
        )

        total_registros = int(
            dados_completude[
                "TOTAL_REGISTROS"
            ]
        )

        sorotipo_informado = int(
            dados_completude[
                "SOROTIPO_INFORMADO"
            ]
        )

        sorotipo_nao_informado = int(
            dados_completude[
                "SOROTIPO_NAO_INFORMADO"
            ]
        )

        percentual_informado = float(
            dados_completude[
                "PERCENTUAL_SOROTIPO_INFORMADO"
            ]
        )

        percentual_nao_informado = float(
            dados_completude[
                "PERCENTUAL_SOROTIPO_NAO_INFORMADO"
            ]
        )

        # ====================================================
        # DISTRIBUIÇÃO DOS SOROTIPOS
        # ====================================================

        dados_sorotipos = (
            df_sorotipos[
                df_sorotipos["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        evidencias_sorotipos = []

        if not dados_sorotipos.empty:

            dados_sorotipos = (
                dados_sorotipos
                .sort_values(
                    "TOTAL_CASOS",
                    ascending=False
                )
            )

            for _, linha in dados_sorotipos.iterrows():

                evidencias_sorotipos.append(
                    {
                        "sorotipo":
                            linha[
                                "SOROTIPO_DECODED"
                            ],
                        "total_registros":
                            int(
                                linha[
                                    "TOTAL_CASOS"
                                ]
                            )
                    }
                )

        # ====================================================
        # SOROTIPO MAIS FREQUENTE
        # ====================================================

        if evidencias_sorotipos:

            sorotipo_mais_frequente = (
                evidencias_sorotipos[0][
                    "sorotipo"
                ]
            )

            total_sorotipo_mais_frequente = (
                evidencias_sorotipos[0][
                    "total_registros"
                ]
            )

        else:

            sorotipo_mais_frequente = None
            total_sorotipo_mais_frequente = 0

        # ====================================================
        # INTERPRETAÇÃO
        # ====================================================

        if sorotipo_informado == 0:

            interpretacao = (
                f"Na UF {nome_uf}, foram registrados "
                f"{total_registros:,} registros de dengue "
                f"entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}. "
                f"Nenhum registro apresentou informação "
                f"de sorotipo, impossibilitando caracterizar "
                f"a distribuição virológica da UF neste período."
            )

        else:

            interpretacao = (
                f"Na UF {nome_uf}, foram registrados "
                f"{total_registros:,} registros de dengue "
                f"entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}. "
                f"Desses, {sorotipo_informado:,} apresentaram "
                f"informação de sorotipo "
                f"({percentual_informado:.2f}%). "
                f"O sorotipo mais frequente entre os registros "
                f"com informação disponível foi "
                f"{sorotipo_mais_frequente}, com "
                f"{total_sorotipo_mais_frequente:,} registros."
            )

        # ====================================================
        # DOCUMENTO
        # ====================================================

        documento = criar_documento_semantico(

            document_id=gerar_document_id(
                fonte="SINAN",
                doenca="dengue",
                ano=ano,
                dominio="virologico",
                localizacao=codigo_uf
            ),

            titulo=(
                f"Sorotipos de dengue em "
                f"{nome_uf} - {ano}"
            ),

            tipo_documento=(
                "perfil_virologico_uf"
            ),

            dominio="virologico",

            sintese=(
                f"Este documento apresenta a distribuição "
                f"dos sorotipos de dengue registrados no "
                f"SINAN em {nome_uf}, no ano de {ano}, "
                f"considerando os registros disponíveis "
                f"entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final}. "
                f"Também apresenta a completude da variável "
                f"de sorotipo."
            ),

            indicadores={
                "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1,

                "total_registros":
                    total_registros,

                "sorotipo_informado":
                    sorotipo_informado,

                "sorotipo_nao_informado":
                    sorotipo_nao_informado,

                "percentual_sorotipo_informado":
                    percentual_informado,

                "percentual_sorotipo_nao_informado":
                    percentual_nao_informado,

                "sorotipo_mais_frequente":
                    sorotipo_mais_frequente,

                "total_sorotipo_mais_frequente":
                    total_sorotipo_mais_frequente
            },

            evidencias={
                "distribuicao_sorotipos":
                    evidencias_sorotipos,

                "completude_sorotipo": [
                    {
                        "total_registros":
                            total_registros,
                        "sorotipo_informado":
                            sorotipo_informado,
                        "sorotipo_nao_informado":
                            sorotipo_nao_informado,
                        "percentual_informado":
                            percentual_informado,
                        "percentual_nao_informado":
                            percentual_nao_informado
                    }
                ]
            },

            interpretacao=
                interpretacao,

            escopo={
                "codigo_uf":
                    codigo_uf,

                "uf_nome":
                    nome_uf,

                "ano":
                    ano,

                "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1
            },

            observacoes_dados=[
                (
                    f"Os resultados correspondem aos "
                    f"registros disponíveis entre as "
                    f"semanas epidemiológicas "
                    f"{semana_inicial} e "
                    f"{semana_final} de {ano}."
                ),
                (
                    "A ausência de informação de sorotipo "
                    "não representa ausência de registros "
                    "de dengue."
                ),
                (
                    "A distribuição dos sorotipos considera "
                    "somente os registros em que a variável "
                    "SOROTIPO possui informação disponível."
                )
            ],

            conceitos_semanticos=[
                "dengue",
                "sorotipo",
                "DEN-1",
                "DEN-2",
                "DEN-3",
                "DEN-4",
                "unidade federativa",
                "semana epidemiológica",
                "vigilância epidemiológica"
            ],

            palavras_chave=[
                "dengue",
                "sorotipo",
                "sorotipos dengue",
                nome_uf,
                codigo_uf,
                f"semana epidemiológica {semana_inicial}",
                f"semana epidemiológica {semana_final}",
                str(ano),
                "SINAN"
            ],

            ano=ano
        )

        documentos.append(
            documento
        )

    return documentos

##3. Gerar os documentos

Agora executamos:

In [70]:
# ============================================================
# GERAR DOCUMENTOS VIROLÓGICOS POR UF
# ============================================================

documentos_sorotipos_uf = (
    criar_documentos_sorotipos_por_uf(
        df_sorotipos=df_sorotipos_uf,
        df_completude=df_completude_sorotipo_uf,
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO
    )
)

print(
    f"Total de documentos gerados: "
    f"{len(documentos_sorotipos_uf)}"
)

Total de documentos gerados: 27


## 4. Antes de salvar, vamos inspecionar Rondônia e Amapá

Eu não salvaria ainda. Primeiro verificaria dois casos distintos.

Rondônia possui informação de sorotipo:

In [71]:
# ============================================================
# VERIFICAR DOCUMENTO DE RONDÔNIA
# ============================================================

documento_ro = next(
    documento
    for documento in documentos_sorotipos_uf
    if documento["escopo"]["codigo_uf"] == "11"
)

print(
    documento_para_markdown(
        documento_ro
    )
)

# Sorotipos de dengue em Rondônia - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_VIROLOGICO_11
- **Tipo de documento:** perfil_virologico_uf
- **Domínio:** virologico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 11
- **Uf nome:** Rondônia
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34

## Síntese epidemiológica

Este documento apresenta a distribuição dos sorotipos de dengue registrados no SINAN em Rondônia, no ano de 2026, considerando os registros disponíveis entre as semanas epidemiológicas 1 e 34. Também apresenta a completude da variável de sorotipo.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros:** 1354
- **Sorotipo informado:** 97
- **Sorotipo nao informado:** 1257
- **Percentual sorotipo informado:** 7.16
- **Percentual sorotipo nao informado:** 92.84
- **Sorotipo mais frequente:** DENV-2
- **Total sorotipo mais frequen

E o Amapá é nosso caso de controle, porque sabemos que há 443 registros, mas nenhum com sorotipo informado:

In [72]:
# ============================================================
# VERIFICAR DOCUMENTO DO AMAPÁ
# ============================================================

documento_ap = next(
    documento
    for documento in documentos_sorotipos_uf
    if documento["escopo"]["codigo_uf"] == "16"
)

print(
    documento_para_markdown(
        documento_ap
    )
)

# Sorotipos de dengue em Amapá - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_VIROLOGICO_16
- **Tipo de documento:** perfil_virologico_uf
- **Domínio:** virologico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 16
- **Uf nome:** Amapá
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34

## Síntese epidemiológica

Este documento apresenta a distribuição dos sorotipos de dengue registrados no SINAN em Amapá, no ano de 2026, considerando os registros disponíveis entre as semanas epidemiológicas 1 e 34. Também apresenta a completude da variável de sorotipo.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros:** 513
- **Sorotipo informado:** 0
- **Sorotipo nao informado:** 513
- **Percentual sorotipo informado:** 0.0
- **Percentual sorotipo nao informado:** 100.0
- **Sorotipo mais frequente:** None
- **Total sorotipo mais frequente:** 0

## Evi

Seguimos com a estratégia de dupla granularidade: documentos por UF + documento nacional consolidado.

Antes de criar o panorama nacional, eu salvaria os 27 documentos que já validamos.

##1. Definir a estrutura de saída

In [73]:
# ============================================================
# DIRETÓRIOS DOS DOCUMENTOS VIROLÓGICOS
# ============================================================

PASTA_DOCS_VIROLOGICAS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
    / "virologicas"
)

PASTA_DOCS_SOROTIPOS_UF = (
    PASTA_DOCS_VIROLOGICAS
    / "sorotipos_por_uf"
)

PASTA_DOCS_PANORAMA_SOROTIPOS = (
    PASTA_DOCS_VIROLOGICAS
    / "panorama_sorotipos"
)

PASTA_DOCS_SOROTIPOS_UF.mkdir(
    parents=True,
    exist_ok=True
)

PASTA_DOCS_PANORAMA_SOROTIPOS.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Documentos por UF:\n"
    f"{PASTA_DOCS_SOROTIPOS_UF}"
)

print(
    f"\nPanorama nacional:\n"
    f"{PASTA_DOCS_PANORAMA_SOROTIPOS}"
)

Documentos por UF:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/virologicas/sorotipos_por_uf

Panorama nacional:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/virologicas/panorama_sorotipos


##2. Salvar os 27 documentos

Como já temos salvar_documento_semantico(), podemos reutilizá-la:

In [74]:
# ============================================================
# SALVAR DOCUMENTOS DE SOROTIPOS POR UF
# ============================================================

arquivos_sorotipos_uf = []

for documento in documentos_sorotipos_uf:

    caminhos = salvar_documento_semantico(
        documento=documento,
        pasta_saida=PASTA_DOCS_SOROTIPOS_UF
    )

    arquivos_sorotipos_uf.append(
        {
            "document_id": documento["document_id"],
            "json": caminhos["json"],
            "markdown": caminhos["markdown"]
        }
    )

print("=" * 70)
print("SALVAMENTO DOS DOCUMENTOS VIROLÓGICOS")
print("=" * 70)

print(
    f"Documentos semânticos: "
    f"{len(arquivos_sorotipos_uf)}"
)

print(
    f"Arquivos JSON: "
    f"{len(arquivos_sorotipos_uf)}"
)

print(
    f"Arquivos Markdown: "
    f"{len(arquivos_sorotipos_uf)}"
)

print(
    f"Total de arquivos: "
    f"{len(arquivos_sorotipos_uf) * 2}"
)

SALVAMENTO DOS DOCUMENTOS VIROLÓGICOS
Documentos semânticos: 27
Arquivos JSON: 27
Arquivos Markdown: 27
Total de arquivos: 54


## 3. Verificação física

Conferência:

In [75]:
# ============================================================
# VERIFICAR ARQUIVOS SALVOS
# ============================================================

arquivos_json = list(
    PASTA_DOCS_SOROTIPOS_UF.glob(
        "*.json"
    )
)

arquivos_markdown = list(
    PASTA_DOCS_SOROTIPOS_UF.glob(
        "*.md"
    )
)

print(
    f"JSON encontrados: "
    f"{len(arquivos_json)}"
)

print(
    f"Markdown encontrados: "
    f"{len(arquivos_markdown)}"
)

if (
    len(arquivos_json) == 27
    and len(arquivos_markdown) == 27
):
    print(
        "✓ Salvamento concluído corretamente."
    )
else:
    print(
        "⚠ Verifique a quantidade de arquivos gerados."
    )

JSON encontrados: 27
Markdown encontrados: 27
✓ Salvamento concluído corretamente.


A ideia é calcular, a partir das tabelas agregadas:

- total nacional de registros;
- total com e sem sorotipo informado;
- percentual nacional de completude;
- distribuição nacional de DENV-1 a DENV-4;
- UFs com maior e menor percentual de informação;
- UFs sem sorotipo informado;
- uma interpretação descritiva e cautelosa.

## 1. Função para criar o panorama nacional de sorotipos

In [76]:
# ============================================================
# GERAR DOCUMENTO NACIONAL DE SOROTIPOS
# ============================================================

def criar_documento_panorama_sorotipos_brasil(
    df_sorotipos,
    df_completude,
    semana_inicial,
    semana_final,
    ano=ANO
):
    """
    Cria um documento semântico nacional consolidado
    sobre a distribuição dos sorotipos de dengue e a
    disponibilidade dessa informação nas UFs.
    """

    # --------------------------------------------------------
    # INDICADORES NACIONAIS DE COMPLETUDE
    # --------------------------------------------------------

    total_registros = int(
        df_completude["TOTAL_REGISTROS"].sum()
    )

    total_sorotipo_informado = int(
        df_completude["SOROTIPO_INFORMADO"].sum()
    )

    total_sorotipo_nao_informado = int(
        df_completude["SOROTIPO_NAO_INFORMADO"].sum()
    )

    percentual_informado = (
        total_sorotipo_informado
        / total_registros
        * 100
        if total_registros > 0
        else 0
    )

    percentual_nao_informado = (
        total_sorotipo_nao_informado
        / total_registros
        * 100
        if total_registros > 0
        else 0
    )

    # --------------------------------------------------------
    # DISTRIBUIÇÃO NACIONAL DOS SOROTIPOS
    # --------------------------------------------------------

    distribuicao_nacional = (
        df_sorotipos
        .groupby(
            "SOROTIPO_DECODED",
            as_index=False
        )["TOTAL_CASOS"]
        .sum()
        .sort_values(
            "TOTAL_CASOS",
            ascending=False
        )
    )

    evidencias_sorotipos = []

    for _, linha in distribuicao_nacional.iterrows():

        total = int(
            linha["TOTAL_CASOS"]
        )

        percentual_entre_informados = (
            total
            / total_sorotipo_informado
            * 100
            if total_sorotipo_informado > 0
            else 0
        )

        evidencias_sorotipos.append(
            {
                "sorotipo": str(
                    linha["SOROTIPO_DECODED"]
                ),
                "total_casos": total,
                "percentual_entre_sorotipos_informados": round(
                    percentual_entre_informados,
                    2
                )
            }
        )

    # --------------------------------------------------------
    # UFs COM MAIOR E MENOR COMPLETUDE
    # --------------------------------------------------------

    df_completude_ordenada = (
        df_completude
        .sort_values(
            "PERCENTUAL_SOROTIPO_INFORMADO",
            ascending=False
        )
        .copy()
    )

    maior_completude = (
        df_completude_ordenada.iloc[0]
    )

    menor_completude = (
        df_completude_ordenada.iloc[-1]
    )

    # --------------------------------------------------------
    # UFs SEM SOROTIPO INFORMADO
    # --------------------------------------------------------

    ufs_sem_sorotipo = (
        df_completude[
            df_completude[
                "SOROTIPO_INFORMADO"
            ] == 0
        ]["UF_NAME"]
        .tolist()
    )

    # --------------------------------------------------------
    # EVIDÊNCIA DE COMPLETUDE POR UF
    # --------------------------------------------------------

    evidencias_ufs = []

    for _, linha in (
        df_completude
        .sort_values("UF_NAME")
        .iterrows()
    ):

        evidencias_ufs.append(
            {
                "codigo_uf": str(
                    linha["SG_UF_NOT"]
                ),
                "uf": str(
                    linha["UF_NAME"]
                ),
                "total_registros": int(
                    linha["TOTAL_REGISTROS"]
                ),
                "sorotipo_informado": int(
                    linha["SOROTIPO_INFORMADO"]
                ),
                "percentual_sorotipo_informado": float(
                    linha[
                        "PERCENTUAL_SOROTIPO_INFORMADO"
                    ]
                )
            }
        )

    # --------------------------------------------------------
    # INTERPRETAÇÃO
    # --------------------------------------------------------

    if evidencias_sorotipos:

        sorotipo_mais_frequente = (
            evidencias_sorotipos[0]
        )

        interpretacao = (
            f"No conjunto nacional de registros com informação "
            f"de sorotipo, "
            f"{sorotipo_mais_frequente['sorotipo']} apresentou "
            f"a maior frequência, com "
            f"{sorotipo_mais_frequente['total_casos']:,} registros. "
            f"A informação de sorotipo estava disponível em "
            f"{percentual_informado:.2f}% dos registros analisados. "
            f"Entre as UFs, {maior_completude['UF_NAME']} apresentou "
            f"o maior percentual de registros com sorotipo informado "
            f"({maior_completude['PERCENTUAL_SOROTIPO_INFORMADO']:.2f}%), "
            f"enquanto {menor_completude['UF_NAME']} apresentou o menor "
            f"percentual "
            f"({menor_completude['PERCENTUAL_SOROTIPO_INFORMADO']:.2f}%)."
        )

    else:

        interpretacao = (
            "Não foram identificados registros com informação "
            "de sorotipo no conjunto analisado."
        )

    # --------------------------------------------------------
    # OBSERVAÇÕES
    # --------------------------------------------------------

    observacoes = [
        (
            f"{total_sorotipo_nao_informado:,} registros "
            f"({percentual_nao_informado:.2f}%) não possuem "
            f"informação de sorotipo."
        ),
        (
            "A distribuição nacional dos sorotipos considera "
            "somente os registros com informação de sorotipo "
            "disponível."
        ),
        (
            f"Os resultados correspondem aos registros "
            f"disponíveis entre as semanas epidemiológicas "
            f"{semana_inicial} e {semana_final} de {ano}."
        ),
        (
            "Diferenças entre UFs devem ser interpretadas levando "
            "em consideração a disponibilidade da informação "
            "de sorotipo."
        )
    ]

    if ufs_sem_sorotipo:

        observacoes.append(
            "UFs sem registros com sorotipo informado: "
            + ", ".join(ufs_sem_sorotipo)
            + "."
        )

    # --------------------------------------------------------
    # DOCUMENTO
    # --------------------------------------------------------

    documento = criar_documento_semantico(

        document_id=gerar_document_id(
            fonte="SINAN",
            doenca="DENGUE",
            ano=ano,
            dominio="SOROTIPOS",
            localizacao="BRASIL"
        ),

        titulo=(
            f"Panorama dos sorotipos de dengue "
            f"no Brasil — {ano}"
        ),

        tipo_documento=(
            "panorama_virologico_nacional"
        ),

        dominio="virologico",


        sintese=(
            f"Este documento apresenta o panorama nacional "
            f"dos sorotipos de dengue registrados no SINAN "
            f"em {ano}, considerando os registros disponíveis "
            f"entre as semanas epidemiológicas "
            f"{semana_inicial} e {semana_final}. "
            f"São apresentadas a distribuição dos sorotipos "
            f"identificados e a completude da informação "
            f"de sorotipo entre as Unidades da Federação."
        ),

        indicadores={
            "total_registros": total_registros,
            "semana_inicial":
                semana_inicial,
            "semana_final":
                semana_final,
            "total_semanas":
                semana_final - semana_inicial + 1,
            "sorotipo_informado": total_sorotipo_informado,
            "sorotipo_nao_informado": total_sorotipo_nao_informado,
            "percentual_sorotipo_informado": round(
                percentual_informado,
                2
            ),
            "percentual_sorotipo_nao_informado": round(
                percentual_nao_informado,
                2
            ),
            "total_ufs": int(
                df_completude["SG_UF_NOT"].nunique()
            ),
            "ufs_sem_sorotipo_informado": len(
                ufs_sem_sorotipo
            )
        },

        evidencias={
            "distribuicao_nacional_sorotipos":
                evidencias_sorotipos,

            "completude_por_uf":
                evidencias_ufs
        },

        interpretacao=interpretacao,

        escopo={
            "pais": "Brasil",
            "ano": ano,
            "semana_inicial": semana_inicial,
            "semana_final": semana_final,
            "total_semanas": (
                semana_final - semana_inicial + 1
            ),
            "total_ufs": int(
                df_completude["SG_UF_NOT"].nunique()
            )
        },

        observacoes_dados=observacoes,

        conceitos_semanticos=[
            "Dengue",
            "Sorotipo",
            "Brasil",
            "Unidade Federativa",
            "Notificação epidemiológica"
        ],

        palavras_chave=[
            "dengue",
            "SINAN",
            "Brasil",
            "sorotipo",
            "DENV-1",
            "DENV-2",
            "DENV-3",
            "DENV-4",
            "completude"
        ],

        fonte="SINAN",
        doenca="dengue",
        ano=ano
    )

    return documento

Há apenas um ponto: nossa função atual documento_para_markdown() foi criada pensando em evidencias como uma lista. Nesse panorama nacional estamos usando uma estrutura mais rica:
```
"evidencias": {
    "distribuicao_nacional_sorotipos": [...],
    "completude_por_uf": [...]
}
```
Isso é melhor semanticamente, mas antes de salvar o Markdown devemos adaptar o conversor para aceitar tanto lista quanto dicionário.

## 2. Primeiro gere e inspecione o JSON

In [77]:
# ============================================================
# CRIAR PANORAMA NACIONAL DE SOROTIPOS
# ============================================================

documento_sorotipos_brasil = (
    criar_documento_panorama_sorotipos_brasil(
        df_sorotipos=df_sorotipos_uf,
        df_completude=df_completude_sorotipo_uf,
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO
    )
)

print(
    json.dumps(
        documento_sorotipos_brasil,
        indent=2,
        ensure_ascii=False,
        default=converter_json
    )
)

{
  "document_id": "SINAN_DENGUE_2026_SOROTIPOS_BRASIL",
  "titulo": "Panorama dos sorotipos de dengue no Brasil — 2026",
  "tipo_documento": "panorama_virologico_nacional",
  "dominio": "virologico",
  "fonte": {
    "sistema": "SINAN",
    "doenca": "dengue",
    "ano": 2026
  },
  "escopo": {
    "pais": "Brasil",
    "ano": 2026,
    "semana_inicial": 1,
    "semana_final": 34,
    "total_semanas": 34,
    "total_ufs": 27
  },
  "sintese": "Este documento apresenta o panorama nacional dos sorotipos de dengue registrados no SINAN em 2026, considerando os registros disponíveis entre as semanas epidemiológicas 1 e 34. São apresentadas a distribuição dos sorotipos identificados e a completude da informação de sorotipo entre as Unidades da Federação.",
  "indicadores": {
    "total_registros": 444266,
    "semana_inicial": 1,
    "semana_final": 34,
    "total_semanas": 34,
    "sorotipo_informado": 29517,
    "sorotipo_nao_informado": 414749,
    "percentual_sorotipo_informado": 6.64,


## 3. Faça também estas verificações

In [78]:
# ============================================================
# VERIFICAR PANORAMA NACIONAL
# ============================================================

print(
    "Document ID:",
    documento_sorotipos_brasil[
        "document_id"
    ]
)

print(
    "Total de UFs:",
    documento_sorotipos_brasil[
        "indicadores"
    ][
        "total_ufs"
    ]
)

print(
    "UFs sem sorotipo informado:",
    documento_sorotipos_brasil[
        "indicadores"
    ][
        "ufs_sem_sorotipo_informado"
    ]
)

print("\nDistribuição nacional:")

for evidencia in (
    documento_sorotipos_brasil[
        "evidencias"
    ][
        "distribuicao_nacional_sorotipos"
    ]
):
    print(evidencia)

Document ID: SINAN_DENGUE_2026_SOROTIPOS_BRASIL
Total de UFs: 27
UFs sem sorotipo informado: 1

Distribuição nacional:
{'sorotipo': 'DENV-2', 'total_casos': 18763, 'percentual_entre_sorotipos_informados': 63.57}
{'sorotipo': 'DENV-3', 'total_casos': 9017, 'percentual_entre_sorotipos_informados': 30.55}
{'sorotipo': 'DENV-1', 'total_casos': 1631, 'percentual_entre_sorotipos_informados': 5.53}
{'sorotipo': 'DENV-4', 'total_casos': 106, 'percentual_entre_sorotipos_informados': 0.36}


## 3. Gerar o Markdown nacional

In [79]:
# ============================================================
# GERAR MARKDOWN DO PANORAMA NACIONAL
# ============================================================

markdown_sorotipos_brasil = (
    documento_para_markdown(
        documento_sorotipos_brasil
    )
)

print(
    markdown_sorotipos_brasil
)

# Panorama dos sorotipos de dengue no Brasil — 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_SOROTIPOS_BRASIL
- **Tipo de documento:** panorama_virologico_nacional
- **Domínio:** virologico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Pais:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total ufs:** 27

## Síntese epidemiológica

Este documento apresenta o panorama nacional dos sorotipos de dengue registrados no SINAN em 2026, considerando os registros disponíveis entre as semanas epidemiológicas 1 e 34. São apresentadas a distribuição dos sorotipos identificados e a completude da informação de sorotipo entre as Unidades da Federação.

## Indicadores

- **Total registros:** 444266
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Sorotipo informado:** 29517
- **Sorotipo nao informado:** 414749
- **Percentual sorotipo informado:** 6.64
- **Percentual sorotipo nao i

Antes de salvar, confira:

In [80]:
print(
    "Período:",
    documento_sorotipos_brasil[
        "escopo"
    ]["semana_inicial"],
    "a",
    documento_sorotipos_brasil[
        "escopo"
    ]["semana_final"]
)

print(
    documento_para_markdown(
        documento_sorotipos_brasil
    )
)

Período: 1 a 34
# Panorama dos sorotipos de dengue no Brasil — 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_SOROTIPOS_BRASIL
- **Tipo de documento:** panorama_virologico_nacional
- **Domínio:** virologico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Pais:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total ufs:** 27

## Síntese epidemiológica

Este documento apresenta o panorama nacional dos sorotipos de dengue registrados no SINAN em 2026, considerando os registros disponíveis entre as semanas epidemiológicas 1 e 34. São apresentadas a distribuição dos sorotipos identificados e a completude da informação de sorotipo entre as Unidades da Federação.

## Indicadores

- **Total registros:** 444266
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Sorotipo informado:** 29517
- **Sorotipo nao informado:** 414749
- **Percentual sorotipo informado:** 6.64
- **Percentua

## 4. Salvar o 28º documento

Se o Markdown estiver correto:

In [81]:
# ============================================================
# SALVAR PANORAMA NACIONAL DE SOROTIPOS
# ============================================================

arquivos_panorama_sorotipos = (
    salvar_documento_semantico(
        documento=documento_sorotipos_brasil,
        pasta_saida=PASTA_DOCS_PANORAMA_SOROTIPOS
    )
)

print(
    "JSON:",
    arquivos_panorama_sorotipos[
        "json"
    ]
)

print(
    "Markdown:",
    arquivos_panorama_sorotipos[
        "markdown"
    ]
)

print(
    "\nJSON existe:",
    arquivos_panorama_sorotipos[
        "json"
    ].exists()
)

print(
    "Markdown existe:",
    arquivos_panorama_sorotipos[
        "markdown"
    ].exists()
)

JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/virologicas/panorama_sorotipos/sinan_dengue_2026_sorotipos_brasil.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/virologicas/panorama_sorotipos/sinan_dengue_2026_sorotipos_brasil.md

JSON existe: True
Markdown existe: True


Com isso, fechamos o primeiro conjunto:
```
VIROLÓGICAS
│
├── 27 documentos por UF
│     ├── JSON
│     └── Markdown
│
└── 1 panorama Brasil
      ├── JSON
      └── Markdown

Total:
28 documentos semânticos
56 arquivos
```

## 6. Domínio temporal


Antes de montar os documentos, eu inspecionaria as colunas reais desses DataFrames:

In [82]:
# ============================================================
# INSPECIONAR DATAFRAMES TEMPORAIS
# ============================================================

dataframes_temporais = {
    "resumo_temporal_brasil":
        df_resumo_temporal_brasil,

    "serie_temporal_brasil":
        df_serie_temporal_brasil,

    "serie_temporal_uf":
        df_serie_temporal_uf,

    "indicadores_temporais_uf":
        df_indicadores_temporais_uf,

    "variacao_temporal_uf":
        df_variacao_temporal_uf,

    "semanas_pico_uf":
        df_semanas_pico_uf,

    "pico_comportamento_uf":
        df_pico_comportamento_uf
}

for nome, df in dataframes_temporais.items():

    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    print(
        df.columns.tolist()
    )

    display(
        df.head()
    )


resumo_temporal_brasil
['ANO', 'SEMANA_INICIAL', 'SEMANA_FINAL', 'TOTAL_REGISTROS', 'MEDIA_SEMANAL', 'MEDIANA_SEMANAL', 'DESVIO_PADRAO']


,ANO,SEMANA_INICIAL,SEMANA_FINAL,TOTAL_REGISTROS,MEDIA_SEMANAL,MEDIANA_SEMANAL,DESVIO_PADRAO
0,2026,1,34,444266,13066.65,11983.5,5135.84



serie_temporal_brasil
['SEMANA_EPIDEMIOLOGICA', 'TOTAL_REGISTROS', 'PERCENTUAL_TOTAL']


,SEMANA_EPIDEMIOLOGICA,TOTAL_REGISTROS,PERCENTUAL_TOTAL
0,1,4219,0.95
1,2,7673,1.73
2,3,8532,1.92
3,4,9417,2.12
4,5,10237,2.30



serie_temporal_uf
['SG_UF_NOT', 'UF_NAME', 'SEMANA_EPIDEMIOLOGICA', 'TOTAL_REGISTROS']


...

,SG_UF_NOT,UF_NAME,SEMANA_INICIAL_ANTERIOR,SEMANA_FINAL_ANTERIOR,SEMANA_INICIAL_RECENTE,SEMANA_FINAL_RECENTE,TOTAL_ANTERIOR,MEDIA_ANTERIOR,TOTAL_RECENTE,MEDIA_RECENTE,VARIACAO_ABSOLUTA,VARIACAO_PERCENTUAL,VARIACAO_PERCENTUAL_CALCULAVEL,CLASSIFICACAO_VARIACAO,LIMIAR_ESTABILIDADE_PERCENTUAL
0,42,Santa Catarina,27,30,31,34,203,50.75,404,101.00,50.25,99.01,Sim,Aumento,10
1,41,Paraná,27,30,31,34,732,183.00,1405,351.25,168.25,91.94,Sim,Aumento,10
2,43,Rio Grande do Sul,27,30,31,34,122,30.50,196,49.00,18.50,60.66,Sim,Aumento,10
3,12,Acre,27,30,31,34,181,45.25,262,65.50,20.25,44.75,Sim,Aumento,10
4,50,Mato Grosso do Sul,27,30,31,34,332,83.00,479,119.75,36.75,44.28,Sim,Aumento,10



semanas_pico_uf
['SG_UF_NOT', 'UF_NAME', 'SEMANA_PICO', 'TOTAL_PICO']


,SG_UF_NOT,UF_NAME,SEMANA_PICO,TOTAL_PICO
0,11,Rondônia,15,71
1,12,Acre,32,134
2,13,Amazonas,31,102
3,14,Roraima,31,50
4,15,Pará,12,446



pico_comportamento_uf
['SG_UF_NOT', 'UF_NAME', 'SEMANAS_PICO', 'PRIMEIRA_SEMANA_PICO', 'ULTIMA_SEMANA_PICO', 'QUANTIDADE_SEMANAS_PICO', 'TOTAL_PICO', 'TOTAL_REGISTROS', 'PERCENTUAL_PICO_TOTAL', 'DISTANCIA_PICO_SEMANA_FINAL', 'PICO_NA_JANELA_RECENTE', 'MEDIA_ANTERIOR', 'MEDIA_RECENTE', 'VARIACAO_PERCENTUAL', 'CLASSIFICACAO_VARIACAO']


,SG_UF_NOT,UF_NAME,SEMANAS_PICO,PRIMEIRA_SEMANA_PICO,ULTIMA_SEMANA_PICO,QUANTIDADE_SEMANAS_PICO,TOTAL_PICO,TOTAL_REGISTROS,PERCENTUAL_PICO_TOTAL,DISTANCIA_PICO_SEMANA_FINAL,PICO_NA_JANELA_RECENTE,MEDIA_ANTERIOR,MEDIA_RECENTE,VARIACAO_PERCENTUAL,CLASSIFICACAO_VARIACAO
0,42,Santa Catarina,33,33,33,1,250,3070,8.14,1,Sim,50.75,101.00,99.01,Aumento
1,53,Distrito Federal,33,33,33,1,193,3841,5.02,1,Sim,90.50,111.50,23.20,Aumento
2,12,Acre,32,32,32,1,134,1832,7.31,2,Sim,45.25,65.50,44.75,Aumento
3,24,Rio Grande do Norte,31,31,31,1,444,7234,6.14,3,Sim,356.50,309.75,-13.11,Redução
4,13,Amazonas,31,31,31,1,102,1399,7.29,3,Sim,48.25,67.75,40.41,Aumento


## 1. Estrutura do documento temporal por UF

Vamos combinar:
```
df_serie_temporal_uf
        +
df_indicadores_temporais_uf
        +
df_variacao_temporal_uf
        +
df_pico_comportamento_uf
        ↓
Documento temporal da UF
```

Isso produzirá novamente 27 documentos, um por UF.

##  2. Função de interpretação temporal

In [83]:
# ============================================================
# INTERPRETAÇÃO TEMPORAL POR UF
# ============================================================

def interpretar_temporal_uf(
    uf_nome,
    semana_pico,
    total_pico,
    semana_inicial,
    semana_final,
    media_semanal,
    classificacao_variacao,
    variacao_percentual,
    semana_inicial_recente,
    semana_final_recente
):
    """
    Gera interpretação descritiva e determinística
    do comportamento temporal dos registros.
    """

    texto = (
        f"Em {uf_nome}, o maior número semanal de registros "
        f"no período analisado ocorreu na semana epidemiológica "
        f"{semana_pico}, com {total_pico:,} registros. "
        f"Entre as semanas epidemiológicas {semana_inicial} e "
        f"{semana_final}, a média foi de "
        f"{media_semanal:.2f} registros por semana."
    )

    if pd.notna(variacao_percentual):

        texto += (
            f" Na comparação entre as janelas temporais "
            f"analisadas, o período mais recente "
            f"(semanas {semana_inicial_recente} a "
            f"{semana_final_recente}) apresentou variação de "
            f"{variacao_percentual:.2f}% na média semanal, "
            f"classificada como {str(classificacao_variacao).lower()}."
        )

    return texto

Aqui mantemos a mesma regra adotada nos sorotipos: descrever o que os dados mostram **sem inferir causas.**

## 3. Gerar os documentos temporais

In [84]:
# ============================================================
# GERAR DOCUMENTOS TEMPORAIS POR UF
# ============================================================

def criar_documentos_temporais_por_uf(
    df_serie,
    df_indicadores,
    df_variacao,
    df_pico,
    ano=ANO
):
    """
    Cria um documento semântico temporal por UF.
    """

    documentos = []

    # --------------------------------------------------------
    # INDICADORES COMO TABELA-BASE
    # --------------------------------------------------------

    for _, linha in df_indicadores.iterrows():

        codigo_uf = str(
            linha["SG_UF_NOT"]
        )

        uf_nome = str(
            linha["UF_NAME"]
        )

        # ----------------------------------------------------
        # INDICADORES GERAIS
        # ----------------------------------------------------

        semana_inicial = int(
            linha["SEMANA_INICIAL"]
        )

        semana_final = int(
            linha["SEMANA_FINAL"]
        )

        total_semanas = int(
            linha["TOTAL_SEMANAS"]
        )

        total_registros = int(
            linha["TOTAL_REGISTROS"]
        )

        media_semanal = float(
            linha["MEDIA_SEMANAL"]
        )

        mediana_semanal = float(
            linha["MEDIANA_SEMANAL"]
        )

        minimo_semanal = int(
            linha["MINIMO_SEMANAL"]
        )

        maximo_semanal = int(
            linha["MAXIMO_SEMANAL"]
        )

        semana_pico = int(
            linha["SEMANA_PICO"]
        )

        total_pico = int(
            linha["TOTAL_PICO"]
        )

        desvio_padrao = float(
            linha["DESVIO_PADRAO"]
        )

        coeficiente_variacao = float(
            linha["COEFICIENTE_VARIACAO"]
        )

        # ----------------------------------------------------
        # SÉRIE TEMPORAL DA UF
        # ----------------------------------------------------

        df_serie_uf = (
            df_serie[
                df_serie["SG_UF_NOT"].astype(str)
                == codigo_uf
            ]
            .sort_values(
                "SEMANA_EPIDEMIOLOGICA"
            )
        )

        serie_semanal = []

        for _, registro in df_serie_uf.iterrows():

            serie_semanal.append(
                {
                    "semana_epidemiologica": int(
                        registro[
                            "SEMANA_EPIDEMIOLOGICA"
                        ]
                    ),
                    "total_registros": int(
                        registro[
                            "TOTAL_REGISTROS"
                        ]
                    )
                }
            )

        # ----------------------------------------------------
        # VARIAÇÃO TEMPORAL
        # ----------------------------------------------------

        df_variacao_uf = df_variacao[
            df_variacao["SG_UF_NOT"].astype(str)
            == codigo_uf
        ]

        if not df_variacao_uf.empty:

            variacao = (
                df_variacao_uf.iloc[0]
            )

            semana_inicial_anterior = int(
                variacao[
                    "SEMANA_INICIAL_ANTERIOR"
                ]
            )

            semana_final_anterior = int(
                variacao[
                    "SEMANA_FINAL_ANTERIOR"
                ]
            )

            semana_inicial_recente = int(
                variacao[
                    "SEMANA_INICIAL_RECENTE"
                ]
            )

            semana_final_recente = int(
                variacao[
                    "SEMANA_FINAL_RECENTE"
                ]
            )

            media_anterior = float(
                variacao["MEDIA_ANTERIOR"]
            )

            media_recente = float(
                variacao["MEDIA_RECENTE"]
            )

            variacao_absoluta = float(
                variacao["VARIACAO_ABSOLUTA"]
            )

            variacao_percentual = (
                normalizar_valor(
                    variacao[
                        "VARIACAO_PERCENTUAL"
                    ]
                )
            )

            classificacao_variacao = str(
                variacao[
                    "CLASSIFICACAO_VARIACAO"
                ]
            )

        else:

            semana_inicial_anterior = None
            semana_final_anterior = None
            semana_inicial_recente = None
            semana_final_recente = None

            media_anterior = None
            media_recente = None
            variacao_absoluta = None
            variacao_percentual = None
            classificacao_variacao = None

        # ----------------------------------------------------
        # COMPORTAMENTO DO PICO
        # ----------------------------------------------------

        df_pico_uf = df_pico[
            df_pico["SG_UF_NOT"].astype(str)
            == codigo_uf
        ]

        if not df_pico_uf.empty:

            pico = df_pico_uf.iloc[0]

            semanas_pico = normalizar_valor(
                pico["SEMANAS_PICO"]
            )

            quantidade_semanas_pico = int(
                pico["QUANTIDADE_SEMANAS_PICO"]
            )

            percentual_pico_total = float(
                pico["PERCENTUAL_PICO_TOTAL"]
            )

            distancia_pico_final = int(
                pico[
                    "DISTANCIA_PICO_SEMANA_FINAL"
                ]
            )

            pico_janela_recente = str(
                pico["PICO_NA_JANELA_RECENTE"]
            )

        else:

            semanas_pico = None
            quantidade_semanas_pico = None
            percentual_pico_total = None
            distancia_pico_final = None
            pico_janela_recente = None

        # ----------------------------------------------------
        # INTERPRETAÇÃO
        # ----------------------------------------------------

        interpretacao = interpretar_temporal_uf(
            uf_nome=uf_nome,
            semana_pico=semana_pico,
            total_pico=total_pico,
            semana_inicial=semana_inicial,
            semana_final=semana_final,
            media_semanal=media_semanal,
            classificacao_variacao=classificacao_variacao,
            variacao_percentual=variacao_percentual,
            semana_inicial_recente=semana_inicial_recente,
            semana_final_recente=semana_final_recente
        )

        # ----------------------------------------------------
        # DOCUMENTO
        # ----------------------------------------------------

        documento = criar_documento_semantico(

            document_id=gerar_document_id(
                fonte="SINAN",
                doenca="DENGUE",
                ano=ano,
                dominio="TEMPORAL",
                localizacao=codigo_uf
            ),

            titulo=(
                f"Comportamento temporal dos registros "
                f"de dengue em {uf_nome} — {ano}"
            ),

            tipo_documento=(
                "perfil_temporal_uf"
            ),

            dominio="temporal",

            sintese=(
                f"Este documento apresenta o comportamento "
                f"temporal dos registros de dengue em {uf_nome} "
                f"entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}, "
                f"incluindo distribuição semanal, pico de "
                f"registros e variação temporal recente."
            ),

            indicadores={
                "total_registros": total_registros,
                "total_semanas": total_semanas,
                "media_semanal": media_semanal,
                "mediana_semanal": mediana_semanal,
                "minimo_semanal": minimo_semanal,
                "maximo_semanal": maximo_semanal,
                "semana_pico": semana_pico,
                "total_pico": total_pico,
                "desvio_padrao": desvio_padrao,
                "coeficiente_variacao": coeficiente_variacao,
                "media_periodo_anterior": media_anterior,
                "media_periodo_recente": media_recente,
                "variacao_absoluta": variacao_absoluta,
                "variacao_percentual": variacao_percentual,
                "classificacao_variacao":
                    classificacao_variacao
            },

            evidencias={
                "serie_semanal": serie_semanal,

                "comparacao_temporal": {
                    "semana_inicial_anterior":
                        semana_inicial_anterior,
                    "semana_final_anterior":
                        semana_final_anterior,
                    "semana_inicial_recente":
                        semana_inicial_recente,
                    "semana_final_recente":
                        semana_final_recente,
                    "media_anterior":
                        media_anterior,
                    "media_recente":
                        media_recente,
                    "variacao_percentual":
                        variacao_percentual,
                    "classificacao":
                        classificacao_variacao
                },

                "comportamento_pico": {
                    "semanas_pico":
                        semanas_pico,
                    "quantidade_semanas_pico":
                        quantidade_semanas_pico,
                    "percentual_pico_total":
                        percentual_pico_total,
                    "distancia_pico_semana_final":
                        distancia_pico_final,
                    "pico_na_janela_recente":
                        pico_janela_recente
                }
            },

            interpretacao=interpretacao,

            escopo={
                "codigo_uf": codigo_uf,
                "uf_nome": uf_nome,
                "ano": ano,
                "semana_inicial":
                    semana_inicial,
                "semana_final":
                    semana_final
            },

            observacoes_dados=[
                (
                    f"Os resultados correspondem ao período "
                    f"disponível entre as semanas epidemiológicas "
                    f"{semana_inicial} e {semana_final} de {ano}."
                ),
                (
                    "A classificação da variação compara médias "
                    "semanais entre duas janelas temporais e não "
                    "representa, isoladamente, uma tendência "
                    "epidemiológica de longo prazo."
                )
            ],

            conceitos_semanticos=[
                "Dengue",
                "Semana Epidemiológica",
                "Série Temporal",
                "Pico de registros",
                "Variação temporal",
                "Unidade Federativa"
            ],

            palavras_chave=[
                "dengue",
                "SINAN",
                uf_nome,
                "semana epidemiológica",
                "série temporal",
                "pico",
                "variação temporal",
                "registros de dengue"
            ],

            fonte="SINAN",
            doenca="dengue",
            ano=ano
        )

        documentos.append(
            documento
        )

    return documentos

## 4. Gerar e validar

In [85]:
# ============================================================
# GERAR DOCUMENTOS TEMPORAIS POR UF
# ============================================================

documentos_temporais_uf = (
    criar_documentos_temporais_por_uf(
        df_serie=df_serie_temporal_uf,
        df_indicadores=df_indicadores_temporais_uf,
        df_variacao=df_variacao_temporal_uf,
        df_pico=df_pico_comportamento_uf,
        ano=ANO
    )
)

print(
    f"Total de documentos temporais gerados: "
    f"{len(documentos_temporais_uf)}"
)

Total de documentos temporais gerados: 27


## 5. Antes de salvar, inspecione uma UF

Goiás é um bom caso de teste porque, na tabela fornecida, possui 105.364 registros, média semanal de 3.398,84 e pico na SE 15 com 5.651 registros.

In [86]:
# ============================================================
# INSPECIONAR DOCUMENTO TEMPORAL DE GOIÁS
# ============================================================

documento_go = next(
    documento
    for documento in documentos_temporais_uf
    if documento["escopo"]["codigo_uf"] == "52"
)

print(
    documento_para_markdown(
        documento_go
    )
)

# Comportamento temporal dos registros de dengue em Goiás — 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_TEMPORAL_52
- **Tipo de documento:** perfil_temporal_uf
- **Domínio:** temporal
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 52
- **Uf nome:** Goiás
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34

## Síntese epidemiológica

Este documento apresenta o comportamento temporal dos registros de dengue em Goiás entre as semanas epidemiológicas 1 e 34 de 2026, incluindo distribuição semanal, pico de registros e variação temporal recente.

## Indicadores

- **Total registros:** 108884
- **Total semanas:** 34
- **Media semanal:** 3202.47
- **Mediana semanal:** 3136.5
- **Minimo semanal:** 0
- **Maximo semanal:** 5568
- **Semana pico:** 15
- **Total pico:** 5568
- **Desvio padrao:** 1621.69
- **Coeficiente variacao:** 50.64
- **Media periodo anterior:** 1478.0
- **Media periodo recente:** 857.75
- **Variacao absoluta:

## 1. Criar as pastas do domínio temporal

Mantendo o mesmo padrão usado em virologicas:

In [87]:
# ============================================================
# DIRETÓRIOS DOS DOCUMENTOS TEMPORAIS
# ============================================================

PASTA_DOCS_TEMPORAIS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
    / "temporais"
)

PASTA_DOCS_TEMPORAIS_UF = (
    PASTA_DOCS_TEMPORAIS
    / "comportamento_por_uf"
)

PASTA_DOCS_PANORAMA_TEMPORAL = (
    PASTA_DOCS_TEMPORAIS
    / "panorama_temporal"
)

PASTA_DOCS_TEMPORAIS_UF.mkdir(
    parents=True,
    exist_ok=True
)

PASTA_DOCS_PANORAMA_TEMPORAL.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Documentos por UF:\n"
    f"{PASTA_DOCS_TEMPORAIS_UF}"
)

print(
    f"\nPanorama nacional:\n"
    f"{PASTA_DOCS_PANORAMA_TEMPORAL}"
)

Documentos por UF:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/temporais/comportamento_por_uf

Panorama nacional:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/temporais/panorama_temporal


## 2. Salvar os 27 documentos

In [88]:
# ============================================================
# SALVAR DOCUMENTOS TEMPORAIS POR UF
# ============================================================

arquivos_temporais_uf = []

for documento in documentos_temporais_uf:

    caminhos = salvar_documento_semantico(
        documento=documento,
        pasta_saida=PASTA_DOCS_TEMPORAIS_UF
    )

    arquivos_temporais_uf.append(
        {
            "document_id": documento["document_id"],
            "json": caminhos["json"],
            "markdown": caminhos["markdown"]
        }
    )

print("=" * 70)
print("SALVAMENTO DOS DOCUMENTOS TEMPORAIS")
print("=" * 70)

print(
    f"Documentos semânticos: "
    f"{len(arquivos_temporais_uf)}"
)

print(
    f"Arquivos JSON: "
    f"{len(arquivos_temporais_uf)}"
)

print(
    f"Arquivos Markdown: "
    f"{len(arquivos_temporais_uf)}"
)

print(
    f"Total de arquivos: "
    f"{len(arquivos_temporais_uf) * 2}"
)

SALVAMENTO DOS DOCUMENTOS TEMPORAIS
Documentos semânticos: 27
Arquivos JSON: 27
Arquivos Markdown: 27
Total de arquivos: 54


## 3. Agora o panorama temporal nacional

Para o documento nacional, temos duas fontes particularmente adequadas:

**df_resumo_temporal_brasil:** SE 1–31, 425.929 registros, média semanal de 13.739,65;

**df_serie_temporal_brasil:** distribuição dos registros pelas 31 semanas epidemiológicas.

Não precisamos somar novamente as 27 UFs para obter esses números, pois o Notebook 05 já produziu o agregado nacional.

## 4. Função do panorama nacional

In [89]:
# ============================================================
# GERAR PANORAMA TEMPORAL NACIONAL
# ============================================================

def criar_documento_panorama_temporal_brasil(
    df_resumo,
    df_serie,
    ano=ANO
):
    """
    Cria documento semântico nacional sobre o comportamento
    temporal dos registros de dengue no período analisado.
    """

    # --------------------------------------------------------
    # RESUMO NACIONAL
    # --------------------------------------------------------

    resumo = df_resumo.iloc[0]

    semana_inicial = int(
        resumo["SEMANA_INICIAL"]
    )

    semana_final = int(
        resumo["SEMANA_FINAL"]
    )

    total_registros = int(
        resumo["TOTAL_REGISTROS"]
    )

    media_semanal = float(
        resumo["MEDIA_SEMANAL"]
    )

    mediana_semanal = float(
        resumo["MEDIANA_SEMANAL"]
    )

    desvio_padrao = float(
        resumo["DESVIO_PADRAO"]
    )

    # --------------------------------------------------------
    # IDENTIFICAR PICO NACIONAL
    # --------------------------------------------------------

    linha_pico = df_serie.loc[
        df_serie["TOTAL_REGISTROS"].idxmax()
    ]

    semana_pico = int(
        linha_pico["SEMANA_EPIDEMIOLOGICA"]
    )

    total_pico = int(
        linha_pico["TOTAL_REGISTROS"]
    )

    percentual_pico = float(
        linha_pico["PERCENTUAL_TOTAL"]
    )

    # --------------------------------------------------------
    # SÉRIE SEMANAL
    # --------------------------------------------------------

    serie_semanal = []

    for _, linha in (
        df_serie
        .sort_values("SEMANA_EPIDEMIOLOGICA")
        .iterrows()
    ):

        serie_semanal.append(
            {
                "semana_epidemiologica": int(
                    linha["SEMANA_EPIDEMIOLOGICA"]
                ),
                "total_registros": int(
                    linha["TOTAL_REGISTROS"]
                ),
                "percentual_total": float(
                    linha["PERCENTUAL_TOTAL"]
                )
            }
        )

    # --------------------------------------------------------
    # INTERPRETAÇÃO
    # --------------------------------------------------------

    interpretacao = (
        f"No Brasil, foram registrados "
        f"{total_registros:,} registros de dengue entre as "
        f"semanas epidemiológicas {semana_inicial} e "
        f"{semana_final} de {ano}. "
        f"A média semanal foi de {media_semanal:,.2f} registros. "
        f"O maior número semanal ocorreu na semana epidemiológica "
        f"{semana_pico}, com {total_pico:,} registros, "
        f"correspondendo a {percentual_pico:.2f}% dos registros "
        f"do período analisado."
    )

    # --------------------------------------------------------
    # CRIAR DOCUMENTO
    # --------------------------------------------------------

    documento = criar_documento_semantico(

        document_id=gerar_document_id(
            fonte="SINAN",
            doenca="DENGUE",
            ano=ano,
            dominio="TEMPORAL",
            localizacao="BRASIL"
        ),

        titulo=(
            f"Panorama temporal dos registros "
            f"de dengue no Brasil — {ano}"
        ),

        tipo_documento=(
            "panorama_temporal_nacional"
        ),

        dominio="temporal",

        sintese=(
            f"Este documento apresenta o comportamento temporal "
            f"dos registros de dengue no Brasil entre as semanas "
            f"epidemiológicas {semana_inicial} e {semana_final} "
            f"de {ano}, incluindo a distribuição semanal e o "
            f"pico de registros no período."
        ),

        indicadores={
            "total_registros": total_registros,
            "semana_inicial": semana_inicial,
            "semana_final": semana_final,
            "total_semanas": (
                semana_final
                - semana_inicial
                + 1
            ),
            "media_semanal": media_semanal,
            "mediana_semanal": mediana_semanal,
            "desvio_padrao": desvio_padrao,
            "semana_pico": semana_pico,
            "total_pico": total_pico,
            "percentual_pico_total": percentual_pico
        },

        evidencias={
            "serie_temporal_nacional":
                serie_semanal,

            "pico_nacional": {
                "semana_epidemiologica":
                    semana_pico,
                "total_registros":
                    total_pico,
                "percentual_total":
                    percentual_pico
            }
        },

        interpretacao=interpretacao,

        escopo={
            "pais": "Brasil",
            "ano": ano,
            "semana_inicial": semana_inicial,
            "semana_final": semana_final
        },

        observacoes_dados=[
            (
                f"Os resultados correspondem ao período "
                f"disponível entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}."
            ),
            (
                "Os valores representam registros de dengue "
                "disponíveis no SINAN no período analisado."
            ),
            (
                "Como o período analisado não compreende "
                "necessariamente o ano epidemiológico completo, "
                "os resultados devem ser interpretados dentro "
                "da janela temporal informada."
            )
        ],

        conceitos_semanticos=[
            "Dengue",
            "Brasil",
            "Semana Epidemiológica",
            "Série Temporal",
            "Pico de registros",
            "Notificação epidemiológica"
        ],

        palavras_chave=[
            "dengue",
            "SINAN",
            "Brasil",
            "semana epidemiológica",
            "série temporal",
            "pico",
            "registros de dengue"
        ],

        fonte="SINAN",
        doenca="dengue",
        ano=ano
    )

    return documento

## 5. Gerar e conferir

In [90]:
# ============================================================
# CRIAR PANORAMA TEMPORAL DO BRASIL
# ============================================================

documento_temporal_brasil = (
    criar_documento_panorama_temporal_brasil(
        df_resumo=df_resumo_temporal_brasil,
        df_serie=df_serie_temporal_brasil,
        ano=ANO
    )
)

print(
    documento_para_markdown(
        documento_temporal_brasil
    )
)

# Panorama temporal dos registros de dengue no Brasil — 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_TEMPORAL_BRASIL
- **Tipo de documento:** panorama_temporal_nacional
- **Domínio:** temporal
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Pais:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34

## Síntese epidemiológica

Este documento apresenta o comportamento temporal dos registros de dengue no Brasil entre as semanas epidemiológicas 1 e 34 de 2026, incluindo a distribuição semanal e o pico de registros no período.

## Indicadores

- **Total registros:** 444266
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Media semanal:** 13066.65
- **Mediana semanal:** 11983.5
- **Desvio padrao:** 5135.84
- **Semana pico:** 15
- **Total pico:** 21470
- **Percentual pico total:** 4.83

## Evidências

### Serie temporal nacional

- **Semana epidemiologica:** 1 | **Total registros:** 4219 | **Percentual to

## Salvar

In [91]:
# ============================================================
# SALVAR PANORAMA TEMPORAL NACIONAL
# ============================================================

arquivos_panorama_temporal = (
    salvar_documento_semantico(
        documento=documento_temporal_brasil,
        pasta_saida=PASTA_DOCS_PANORAMA_TEMPORAL
    )
)

print("=" * 70)
print("PANORAMA TEMPORAL NACIONAL SALVO")
print("=" * 70)

print(
    f"JSON:\n"
    f"{arquivos_panorama_temporal['json']}"
)

print(
    f"\nMarkdown:\n"
    f"{arquivos_panorama_temporal['markdown']}"
)

print("\n" + "=" * 70)

print(
    "JSON existe:",
    arquivos_panorama_temporal[
        "json"
    ].exists()
)

print(
    "Markdown existe:",
    arquivos_panorama_temporal[
        "markdown"
    ].exists()
)

PANORAMA TEMPORAL NACIONAL SALVO
JSON:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/temporais/panorama_temporal/sinan_dengue_2026_temporal_brasil.json

Markdown:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/temporais/panorama_temporal/sinan_dengue_2026_temporal_brasil.md

JSON existe: True
Markdown existe: True


In [92]:
# ============================================================
# VERIFICAR DOCUMENTOS DO DOMÍNIO TEMPORAL
# ============================================================

json_uf = list(
    PASTA_DOCS_TEMPORAIS_UF.glob("*.json")
)

markdown_uf = list(
    PASTA_DOCS_TEMPORAIS_UF.glob("*.md")
)

json_panorama = list(
    PASTA_DOCS_PANORAMA_TEMPORAL.glob("*.json")
)

markdown_panorama = list(
    PASTA_DOCS_PANORAMA_TEMPORAL.glob("*.md")
)

total_documentos = (
    len(json_uf)
    + len(json_panorama)
)

total_arquivos = (
    len(json_uf)
    + len(markdown_uf)
    + len(json_panorama)
    + len(markdown_panorama)
)

print("=" * 70)
print("RESUMO DO DOMÍNIO TEMPORAL")
print("=" * 70)

print(
    f"Documentos por UF: {len(json_uf)}"
)

print(
    f"Panoramas nacionais: {len(json_panorama)}"
)

print(
    f"Total de documentos semânticos: {total_documentos}"
)

print(
    f"Arquivos JSON: "
    f"{len(json_uf) + len(json_panorama)}"
)

print(
    f"Arquivos Markdown: "
    f"{len(markdown_uf) + len(markdown_panorama)}"
)

print(
    f"Total de arquivos físicos: {total_arquivos}"
)

if (
    len(json_uf) == 27
    and len(markdown_uf) == 27
    and len(json_panorama) == 1
    and len(markdown_panorama) == 1
):
    print(
        "\n✓ Domínio temporal concluído corretamente."
    )
else:
    print(
        "\n⚠ Verifique os arquivos do domínio temporal."
    )

RESUMO DO DOMÍNIO TEMPORAL
Documentos por UF: 27
Panoramas nacionais: 1
Total de documentos semânticos: 28
Arquivos JSON: 28
Arquivos Markdown: 28
Total de arquivos físicos: 56

✓ Domínio temporal concluído corretamente.


## 7. Domínio geográfico


In [93]:
# ============================================================
# INSPECIONAR DATAFRAMES GEOGRÁFICOS
# ============================================================

dataframes_geograficos = {
    "casos_municipio_2026":
        df_casos_municipio,

    "casos_uf_notificacao_2026":
        df_casos_uf_notificacao,

    "casos_uf_residencia_2026":
        df_casos_uf_residencia,

    "criticidade_uf_2026":
        df_criticidade_uf,

    "incidencia_municipio_2026":
        df_incidencia_municipio
}

for nome, df in dataframes_geograficos.items():

    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    print(
        df.columns.tolist()
    )

    display(
        df.head()
    )


casos_municipio_2026
['CODIGO_MUNICIPIO_6', 'CODIGO_MUNICIPIO_7', 'MUNICIPIO_NAME', 'UF_RESIDENCIA', 'REGIAO', 'POPULACAO_2022', 'CASOS']


,CODIGO_MUNICIPIO_6,CODIGO_MUNICIPIO_7,MUNICIPIO_NAME,UF_RESIDENCIA,REGIAO,POPULACAO_2022,CASOS
0,110000,<NA>,None,None,None,NaN,1
1,110001,1100015,Alta Floresta D'Oeste,RO,Norte,21494.0,102
2,110002,1100023,Ariquemes,RO,Norte,96833.0,103
3,110003,1100031,Cabixi,RO,Norte,5351.0,11
4,110004,1100049,Cacoal,RO,Norte,86887.0,25



casos_uf_notificacao_2026
['SG_UF_NOT', 'UF_NAME', 'TOTAL_CASOS']


,SG_UF_NOT,UF_NAME,TOTAL_CASOS
0,52,Goiás,108884
1,35,São Paulo,65195
2,31,Minas Gerais,61248
3,26,Pernambuco,22036
4,29,Bahia,21609



casos_uf_residencia_2026
['SG_UF', 'UF_RESIDENCIA', 'TOTAL_CASOS']


,SG_UF,UF_RESIDENCIA,TOTAL_CASOS
0,52,GO,108912
1,35,SP,65032
2,31,MG,61668
3,26,PE,22008
4,29,BA,21749



criticidade_uf_2026
['UF_RESIDENCIA', 'GRAU_CRITICIDADE', 'TOTAL_MUNICIPIOS']


,UF_RESIDENCIA,GRAU_CRITICIDADE,TOTAL_MUNICIPIOS
0,AC,Alta,6
1,AC,Baixa,6
2,AC,Média,10
3,AL,Alta,10
4,AL,Baixa,54



incidencia_municipio_2026
['CODIGO_MUNICIPIO_6', 'CODIGO_MUNICIPIO_7', 'MUNICIPIO_NAME', 'UF_RESIDENCIA', 'REGIAO', 'POPULACAO_2022', 'CASOS', 'INCIDENCIA_100MIL', 'GRAU_CRITICIDADE']


,CODIGO_MUNICIPIO_6,CODIGO_MUNICIPIO_7,MUNICIPIO_NAME,UF_RESIDENCIA,REGIAO,POPULACAO_2022,CASOS,INCIDENCIA_100MIL,GRAU_CRITICIDADE
0,110001,1100015,Alta Floresta D'Oeste,RO,Norte,21494.0,102,474.55,Alta
1,110002,1100023,Ariquemes,RO,Norte,96833.0,103,106.37,Média
2,110003,1100031,Cabixi,RO,Norte,5351.0,11,205.57,Média
3,110004,1100049,Cacoal,RO,Norte,86887.0,25,28.77,Baixa
4,110005,1100056,Cerejeiras,RO,Norte,15890.0,11,69.23,Baixa


## 1. Estratégia para os documentos geográficos

```
df_incidencia_municipio_2026
        │
        ├── casos municipais
        ├── população
        ├── incidência
        └── criticidade
        │
        +
df_criticidade_uf_2026
        │
        └── distribuição da criticidade
        │
        +
df_casos_uf_residencia_2026
        │
        └── total da UF por residência
        │
        +
df_casos_uf_notificacao_2026
        │
        └── total da UF por notificação
        ↓
Documento geográfico da UF
```

## 2. Precisamos relacionar sigla e código da UF

Temos um pequeno detalhe: os dados municipais usam RO, GO, SP etc., enquanto os totais por notificação usam códigos como 11, 52, 35.

Podemos obter a relação diretamente de df_casos_uf_residencia_2026:

In [94]:
# ============================================================
# DIAGNÓSTICO - DATAFRAMES CARREGADOS
# ============================================================

nomes_df = sorted(
    nome
    for nome in globals()
    if nome.startswith("df_")
)

for nome in nomes_df:
    print(nome)

df_casos_municipio
df_casos_uf_notificacao
df_casos_uf_residencia
df_completude_sorotipo_uf
df_criticidade_uf
df_distribuicao_picos_uf
df_distribuicao_ultimo_pico_uf
df_doencas_preexistentes_brasil
df_doencas_preexistentes_obitos_brasil
df_doencas_preexistentes_obitos_uf
df_doencas_preexistentes_uf
df_evolucao_brasil
df_evolucao_uf
df_hospitalizacao_brasil
df_hospitalizacao_uf
df_incidencia_municipio
df_indicadores_temporais_uf
df_obitos_brasil
df_obitos_sorotipo_brasil
df_obitos_sorotipo_uf
df_obitos_tipo_uf
df_obitos_uf
df_pico_comportamento_uf
df_resumo_pico_recente
df_resumo_temporal_brasil
df_resumo_variacao_temporal
df_semanas_pico_uf
df_serie_temporal_brasil
df_serie_temporal_uf
df_serie_temporal_uf_normalizada
df_sintomas_brasil
df_sintomas_obitos_brasil
df_sintomas_obitos_uf
df_sintomas_uf
df_sorotipos_uf
df_variacao_temporal_uf


In [95]:
print(
    "df_incidencia_municipio existe:",
    "df_incidencia_municipio" in globals()
)

print(
    "resultados_analytics existe:",
    "resultados_analytics" in globals()
)

print(
    "dataframes_analytics existe:",
    "dataframes_analytics" in globals()
)

df_incidencia_municipio existe: True
resultados_analytics existe: True
dataframes_analytics existe: True


In [96]:
# ============================================================
# DIAGNÓSTICO DOS PRODUTOS GEOGRÁFICOS
# ============================================================

print("Domínios carregados:")
print(resultados_analytics.keys())

print("\nProdutos geográficos carregados:")

if "geograficos" in resultados_analytics:

    for nome in sorted(
        resultados_analytics["geograficos"].keys()
    ):
        print("-", nome)

else:
    print("Domínio 'geograficos' não encontrado.")

Domínios carregados:
dict_keys(['clinicas', 'desfechos', 'geograficos', 'temporais', 'virologicas'])

Produtos geográficos carregados:
- casos_municipio_2026
- casos_uf_notificacao_2026
- casos_uf_residencia_2026
- criticidade_uf_2026
- incidencia_municipio_2026
- inventario_produtos_geograficos_2026


In [97]:
# ============================================================
# DIAGNÓSTICO DOS ALIASES CRIADOS
# ============================================================

print("Aliases relacionados a incidência:")

for nome in sorted(
    dataframes_analytics.keys()
):
    if "incidencia" in nome.lower():
        print("-", nome)

Aliases relacionados a incidência:
- df_incidencia_municipio


In [98]:
# ============================================================
# VERIFICAR PASTA GEOGRÁFICA
# ============================================================

pasta_geograficos = (
    PASTA_ANALYTICS
    / "geograficos"
)

print(
    "Pasta geográficos:",
    pasta_geograficos
)

print(
    "Existe:",
    pasta_geograficos.exists()
)

print("\nArquivos encontrados:")

if pasta_geograficos.exists():

    arquivos = sorted(
        pasta_geograficos.iterdir()
    )

    print(
        "Quantidade:",
        len(arquivos)
    )

    for arquivo in arquivos:
        print(
            "-",
            arquivo.name
        )

else:
    print(
        "Pasta não encontrada."
    )

Pasta geográficos: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/geograficos
Existe: True

Arquivos encontrados:
Quantidade: 6
- casos_municipio_2026.parquet
- casos_uf_notificacao_2026.parquet
- casos_uf_residencia_2026.parquet
- criticidade_uf_2026.parquet
- incidencia_municipio_2026.parquet
- inventario_produtos_geograficos_2026.csv


In [99]:
# ============================================================
# MAPA SIGLA UF -> CÓDIGO IBGE DA UF
# A PARTIR DOS CÓDIGOS MUNICIPAIS
# ============================================================

df_mapa_uf = (
    df_incidencia_municipio[
        [
            "UF_RESIDENCIA",
            "CODIGO_MUNICIPIO_7"
        ]
    ]
    .dropna()
    .copy()
)

df_mapa_uf["CODIGO_UF"] = (
    df_mapa_uf[
        "CODIGO_MUNICIPIO_7"
    ]
    .astype("Int64")
    .astype(str)
    .str[:2]
)

mapa_uf_codigo = (
    df_mapa_uf[
        [
            "UF_RESIDENCIA",
            "CODIGO_UF"
        ]
    ]
    .drop_duplicates()
    .set_index(
        "UF_RESIDENCIA"
    )["CODIGO_UF"]
    .to_dict()
)

print(
    "Quantidade de UFs no mapa:",
    len(mapa_uf_codigo)
)

print(
    mapa_uf_codigo
)

Quantidade de UFs no mapa: 27
{'RO': '11', 'AC': '12', 'AM': '13', 'RR': '14', 'PA': '15', 'AP': '16', 'TO': '17', 'MA': '21', 'PI': '22', 'CE': '23', 'RN': '24', 'PB': '25', 'PE': '26', 'AL': '27', 'SE': '28', 'BA': '29', 'MG': '31', 'ES': '32', 'RJ': '33', 'SP': '35', 'PR': '41', 'SC': '42', 'RS': '43', 'MS': '50', 'MT': '51', 'GO': '52', 'DF': '53'}


In [100]:
# ============================================================
# VALIDAR MAPA DAS UFs
# ============================================================

for sigla in ["RJ", "SP", "MT"]:
    print(
        sigla,
        "->",
        mapa_uf_codigo.get(sigla)
    )

RJ -> 33
SP -> 35
MT -> 51


## 3. Função de interpretação geográfica

Aqui precisamos tomar bastante cuidado para não transformar incidência em sinônimo de número de casos.

In [101]:
# ============================================================
# INTERPRETAÇÃO GEOGRÁFICA POR UF
# ============================================================

def interpretar_geografico_uf(
    uf_nome,
    total_casos_residencia,
    municipio_mais_casos,
    total_maior_casos,
    municipio_maior_incidencia,
    maior_incidencia,
    grau_maior_incidencia,
    total_municipios,
    semana_inicial,
    semana_final,
    ano
):
    """
    Gera interpretação descritiva da distribuição geográfica.
    """

    return (
        f"Na UF {uf_nome}, foram contabilizados "
        f"{total_casos_residencia:,} registros segundo o local "
        f"de residência entre as semanas epidemiológicas "
        f"{semana_inicial} e {semana_final} de {ano}. "
        f"Entre os {total_municipios:,} municípios "
        f"com informações disponíveis para análise, "
        f"{municipio_mais_casos} apresentou o maior número "
        f"absoluto de registros, com {total_maior_casos:,}. "
        f"A maior incidência foi observada em "
        f"{municipio_maior_incidencia}, com "
        f"{maior_incidencia:,.2f} registros por 100 mil "
        f"habitantes, classificada como "
        f"{grau_maior_incidencia}."
    )

Observe que propositalmente dizemos:

município com maior número absoluto

e

município com maior incidência

São conceitos diferentes.

## 4. Criar os documentos geográficos por UF

In [102]:
def criar_documentos_geograficos_por_uf(
    df_incidencia,
    df_criticidade,
    df_casos_residencia,
    df_casos_notificacao,
    mapa_uf_codigo,
    semana_inicial,
    semana_final,
    ano=ANO
):
    """
    Cria um documento semântico geográfico por UF.
    """

    documentos = []

    # --------------------------------------------------------
    # PERCORRER UFs PRESENTES NOS DADOS MUNICIPAIS
    # --------------------------------------------------------

    ufs = sorted(
        df_incidencia[
            "UF_RESIDENCIA"
        ]
        .dropna()
        .unique()
    )

    for sigla_uf in ufs:

        codigo_uf = mapa_uf_codigo.get(
            sigla_uf
        )

        if codigo_uf is None:
            print(
                f"[AVISO] Código não encontrado: {sigla_uf}"
            )
            continue

        # ----------------------------------------------------
        # MUNICÍPIOS DA UF
        # ----------------------------------------------------

        df_uf = (
            df_incidencia[
                df_incidencia["UF_RESIDENCIA"]
                == sigla_uf
            ]
            .copy()
        )

        # ----------------------------------------------------
        # NOME DA UF
        # Recuperado da tabela de notificação
        # ----------------------------------------------------

        registro_nome = (
            df_casos_notificacao[
                df_casos_notificacao[
                    "SG_UF_NOT"
                ].astype(str)
                == str(codigo_uf)
            ]
        )

        if not registro_nome.empty:
            uf_nome = str(
                registro_nome.iloc[0]["UF_NAME"]
            )
        else:
            uf_nome = sigla_uf

        # ----------------------------------------------------
        # TOTAL POR RESIDÊNCIA
        # ----------------------------------------------------

        registro_residencia = (
            df_casos_residencia[
                df_casos_residencia[
                    "UF_RESIDENCIA"
                ] == sigla_uf
            ]
        )

        if not registro_residencia.empty:
            total_casos_residencia = int(
                registro_residencia.iloc[0][
                    "TOTAL_CASOS"
                ]
            )
        else:
            total_casos_residencia = int(
                df_uf["CASOS"].sum()
            )

        # ----------------------------------------------------
        # TOTAL POR NOTIFICAÇÃO
        # ----------------------------------------------------

        registro_notificacao = (
            df_casos_notificacao[
                df_casos_notificacao[
                    "SG_UF_NOT"
                ].astype(str)
                == str(codigo_uf)
            ]
        )

        if not registro_notificacao.empty:
            total_casos_notificacao = int(
                registro_notificacao.iloc[0][
                    "TOTAL_CASOS"
                ]
            )
        else:
            total_casos_notificacao = None

        # ----------------------------------------------------
        # MUNICÍPIO COM MAIS CASOS
        # ----------------------------------------------------

        linha_mais_casos = df_uf.loc[
            df_uf["CASOS"].idxmax()
        ]

        municipio_mais_casos = str(
            linha_mais_casos["MUNICIPIO_NAME"]
        )

        total_maior_casos = int(
            linha_mais_casos["CASOS"]
        )

        # ----------------------------------------------------
        # MUNICÍPIO COM MAIOR INCIDÊNCIA
        # ----------------------------------------------------

        df_incidencia_valida = (
            df_uf[
                df_uf["INCIDENCIA_100MIL"].notna()
            ]
        )

        linha_maior_incidencia = (
            df_incidencia_valida.loc[
                df_incidencia_valida[
                    "INCIDENCIA_100MIL"
                ].idxmax()
            ]
        )

        municipio_maior_incidencia = str(
            linha_maior_incidencia[
                "MUNICIPIO_NAME"
            ]
        )

        maior_incidencia = float(
            linha_maior_incidencia[
                "INCIDENCIA_100MIL"
            ]
        )

        grau_maior_incidencia = str(
            linha_maior_incidencia[
                "GRAU_CRITICIDADE"
            ]
        )

        # ----------------------------------------------------
        # DISTRIBUIÇÃO DA CRITICIDADE
        # ----------------------------------------------------

        df_criticidade_uf = (
            df_criticidade[
                df_criticidade[
                    "UF_RESIDENCIA"
                ] == sigla_uf
            ]
        )

        distribuicao_criticidade = []

        for _, linha in df_criticidade_uf.iterrows():

            distribuicao_criticidade.append(
                {
                    "grau_criticidade": str(
                        linha[
                            "GRAU_CRITICIDADE"
                        ]
                    ),
                    "total_municipios": int(
                        linha[
                            "TOTAL_MUNICIPIOS"
                        ]
                    )
                }
            )

        # ----------------------------------------------------
        # EVIDÊNCIAS MUNICIPAIS
        # ----------------------------------------------------

        evidencias_municipios = []

        for _, municipio in (
            df_uf
            .sort_values(
                "CASOS",
                ascending=False
            )
            .iterrows()
        ):

            evidencias_municipios.append(
                {
                    "codigo_municipio": str(
                        municipio[
                            "CODIGO_MUNICIPIO_7"
                        ]
                    ),
                    "municipio": str(
                        municipio[
                            "MUNICIPIO_NAME"
                        ]
                    ),
                    "populacao_2022":
                        normalizar_valor(
                            municipio[
                                "POPULACAO_2022"
                            ]
                        ),
                    "casos": int(
                        municipio["CASOS"]
                    ),
                    "incidencia_100mil":
                        normalizar_valor(
                            municipio[
                                "INCIDENCIA_100MIL"
                            ]
                        ),
                    "grau_criticidade": str(
                        municipio[
                            "GRAU_CRITICIDADE"
                        ]
                    )
                }
            )

        total_municipios = int(
            df_uf[
                "CODIGO_MUNICIPIO_7"
            ].nunique()
        )

        # ----------------------------------------------------
        # INTERPRETAÇÃO
        # ----------------------------------------------------

        interpretacao = interpretar_geografico_uf(
            uf_nome=uf_nome,
            total_casos_residencia=(
                total_casos_residencia
            ),
            municipio_mais_casos=(
                municipio_mais_casos
            ),
            total_maior_casos=(
                total_maior_casos
            ),
            municipio_maior_incidencia=(
                municipio_maior_incidencia
            ),
            maior_incidencia=(
                maior_incidencia
            ),
            grau_maior_incidencia=(
                grau_maior_incidencia
            ),
            total_municipios=(
                total_municipios
            ),
            semana_inicial=semana_inicial,
            semana_final=semana_final,
            ano=ano
        )

        # ----------------------------------------------------
        # DOCUMENTO SEMÂNTICO
        # ----------------------------------------------------

        documento = criar_documento_semantico(

            document_id=gerar_document_id(
                fonte="SINAN",
                doenca="DENGUE",
                ano=ano,
                dominio="GEOGRAFICO",
                localizacao=codigo_uf
            ),

            titulo=(
                f"Distribuição geográfica dos registros "
                f"de dengue em {uf_nome} — {ano}"
            ),

            tipo_documento=(
                "distribuicao_geografica_uf"
            ),

            dominio="geografico",


            sintese=(
                f"Este documento apresenta a distribuição geográfica "
                f"dos registros de dengue no SINAN em {uf_nome}, "
                f"no ano de {ano}, considerando os registros disponíveis "
                f"entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final}. "
                f"São apresentados indicadores segundo município "
                f"de residência, incidência por 100 mil habitantes "
                f"e classificação de criticidade."
            ),

            indicadores={
                 "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1,

                "total_casos_residencia":
                    total_casos_residencia,

                "total_casos_notificacao":
                    total_casos_notificacao,

                "total_municipios_analisados":
                    total_municipios,

                "municipio_maior_numero_casos":
                    municipio_mais_casos,

                "maior_numero_casos_municipio":
                    total_maior_casos,

                "municipio_maior_incidencia":
                    municipio_maior_incidencia,

                "maior_incidencia_100mil":
                    round(
                        maior_incidencia,
                        2
                    ),

                "criticidade_maior_incidencia":
                    grau_maior_incidencia
            },

            evidencias={
                "distribuicao_criticidade":
                    distribuicao_criticidade,

                "municipios":
                    evidencias_municipios
            },

            interpretacao=interpretacao,

            escopo={
                "codigo_uf": codigo_uf,
                "sigla_uf": sigla_uf,
                "uf_nome": uf_nome,
                "semana_inicial": semana_inicial,
                "semana_final": semana_final,
                "total_semanas": (
                    semana_final
                    - semana_inicial
                    + 1),
                "ano": ano
            },

            observacoes_dados=[
                (
                    "Os totais por residência e por notificação "
                    "representam perspectivas geográficas "
                    "distintas e não devem ser interpretados "
                    "como medidas equivalentes."
                ),
                (
                    f"Os resultados correspondem aos registros "
                    f"disponíveis entre as semanas epidemiológicas "
                    f"{semana_inicial} e {semana_final} de {ano}."
                ),
                (
                    "A incidência por 100 mil habitantes considera "
                    "a população municipal de referência utilizada "
                    "no processo de enriquecimento dos dados."
                ),
                (
                    "Municípios com populações menores podem "
                    "apresentar incidências elevadas mesmo com "
                    "números absolutos de registros relativamente "
                    "baixos."
                )
            ],

            conceitos_semanticos=[
                "Dengue",
                "Município",
                "Unidade Federativa",
                "Residência",
                "Local de notificação",
                "Incidência",
                "População",
                "Criticidade"
            ],

            palavras_chave=[
                "dengue",
                "SINAN",
                uf_nome,
                sigla_uf,
                "município",
                "incidência",
                "casos",
                "residência",
                "notificação",
                "criticidade"
            ],

            fonte="SINAN",
            doenca="dengue",
            ano=ano
        )

        documentos.append(
            documento
        )

    return documentos

## 5. Gerar os documentos

In [103]:
documentos_geograficos_uf = (
    criar_documentos_geograficos_por_uf(
        df_incidencia=(
            df_incidencia_municipio
        ),
        df_criticidade=(
            df_criticidade_uf
        ),
        df_casos_residencia=(
            df_casos_uf_residencia
        ),
        df_casos_notificacao=(
            df_casos_uf_notificacao
        ),
        mapa_uf_codigo=mapa_uf_codigo,
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO
    )
)

print(
    f"Total de documentos geográficos gerados: "
    f"{len(documentos_geograficos_uf)}"
)

Total de documentos geográficos gerados: 27


## 6. Validação importante antes de salvar

Como Goiás já chamou sua atenção em análises anteriores, ele é um ótimo teste.

In [104]:
documento_go_geo = next(
    documento
    for documento in documentos_geograficos_uf
    if documento["escopo"]["codigo_uf"] == "52"
)

print(
    documento_para_markdown(
        documento_go_geo
    )
)

# Distribuição geográfica dos registros de dengue em Goiás — 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_GEOGRAFICO_52
- **Tipo de documento:** distribuicao_geografica_uf
- **Domínio:** geografico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 52
- **Sigla uf:** GO
- **Uf nome:** Goiás
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Ano:** 2026

## Síntese epidemiológica

Este documento apresenta a distribuição geográfica dos registros de dengue no SINAN em Goiás, no ano de 2026, considerando os registros disponíveis entre as semanas epidemiológicas 1 e 34. São apresentados indicadores segundo município de residência, incidência por 100 mil habitantes e classificação de criticidade.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total casos residencia:** 108912
- **Total casos notificacao:** 108884
- **Total municipios analisados:** 243
- **Municipio maior 

Eu verificaria cinco coisas em Goiás: o total por residência deve estar próximo do valor de 105.401 mostrado no resultado do Notebook 05, enquanto o total por notificação é 105.364; além disso, confira se o município com mais casos, o município com maior incidência e a distribuição Alta/Média/Baixa fazem sentido.

Há uma vantagem metodológica importante nessa estrutura: não estamos confundindo o local de residência com o local de notificação. Essa distinção deve permanecer explícita nos documentos, porque depois o RAG poderá responder corretamente perguntas diferentes como “quantos casos foram notificados em Goiás?” e “quantos registros correspondem a residentes de Goiás?”.

## 1. Criar as pastas

In [105]:
# ============================================================
# DIRETÓRIOS DOS DOCUMENTOS GEOGRÁFICOS
# ============================================================

PASTA_DOCS_GEOGRAFICOS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
    / "geograficos"
)

PASTA_DOCS_GEOGRAFICOS_UF = (
    PASTA_DOCS_GEOGRAFICOS
    / "distribuicao_por_uf"
)

PASTA_DOCS_PANORAMA_GEOGRAFICO = (
    PASTA_DOCS_GEOGRAFICOS
    / "panorama_geografico"
)

PASTA_DOCS_GEOGRAFICOS_UF.mkdir(
    parents=True,
    exist_ok=True
)

PASTA_DOCS_PANORAMA_GEOGRAFICO.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Documentos por UF:\n"
    f"{PASTA_DOCS_GEOGRAFICOS_UF}"
)

print(
    f"\nPanorama nacional:\n"
    f"{PASTA_DOCS_PANORAMA_GEOGRAFICO}"
)

Documentos por UF:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/geograficos/distribuicao_por_uf

Panorama nacional:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/geograficos/panorama_geografico


## 2. Salvar os 27 documentos geográficos

In [106]:
# ============================================================
# SALVAR DOCUMENTOS GEOGRÁFICOS POR UF
# ============================================================

arquivos_geograficos_uf = []

for documento in documentos_geograficos_uf:

    caminhos = salvar_documento_semantico(
        documento=documento,
        pasta_saida=PASTA_DOCS_GEOGRAFICOS_UF
    )

    arquivos_geograficos_uf.append(
        {
            "document_id": documento["document_id"],
            "json": caminhos["json"],
            "markdown": caminhos["markdown"]
        }
    )

print("=" * 70)
print("SALVAMENTO DOS DOCUMENTOS GEOGRÁFICOS")
print("=" * 70)

print(
    f"Documentos semânticos: "
    f"{len(arquivos_geograficos_uf)}"
)

print(
    f"Arquivos JSON: "
    f"{len(arquivos_geograficos_uf)}"
)

print(
    f"Arquivos Markdown: "
    f"{len(arquivos_geograficos_uf)}"
)

print(
    f"Total de arquivos físicos: "
    f"{len(arquivos_geograficos_uf) * 2}"
)

SALVAMENTO DOS DOCUMENTOS GEOGRÁFICOS
Documentos semânticos: 27
Arquivos JSON: 27
Arquivos Markdown: 27
Total de arquivos físicos: 54


A estrutura ficará:
```
data_docs/
└── SINAN/
    └── 2026/
        └── geograficos/
            ├── distribuicao_por_uf/
            │   ├── sinan_dengue_2026_geografico_11.json
            │   ├── sinan_dengue_2026_geografico_11.md
            │   ├── ...
            │   ├── sinan_dengue_2026_geografico_53.json
            │   └── sinan_dengue_2026_geografico_53.md
            │
            └── panorama_geografico/
```

## 3. O panorama nacional merece um pouco mais de cuidado

Aqui eu não faria simplesmente uma concatenação dos 27 documentos.

Temos dados suficientes para produzir um documento nacional realmente útil:

distribuição de registros pelas UFs de residência;
distribuição pelas UFs de notificação;
municípios com maior número absoluto de registros;
municípios com maior incidência;
distribuição nacional dos municípios por criticidade;
diferenças entre residência e notificação.

Os dados municipais já contêm população, casos, incidência e criticidade. E as duas tabelas estaduais permitem manter separadas as perspectivas de residência e notificação.

```
Panorama geográfico da dengue no Brasil — 2026

Indicadores
├── total de registros
├── número de UFs
├── municípios analisados
├── UF com maior número por residência
├── UF com maior número por notificação
├── município com maior número de registros
└── município com maior incidência

Evidências
├── registros por UF de residência
├── registros por UF de notificação
├── distribuição nacional da criticidade
├── municípios com maior número de registros
└── municípios com maior incidência

Interpretação
└── síntese descritiva nacional
```


Para as listas municipais nacionais, eu não colocaria os 4.664 municípios novamente. Isso deixaria o documento nacional enorme e repetiria informações que já estão nos documentos das UFs.

Usaria, por exemplo, os 10 municípios com mais registros e os 10 com maior incidência. Os dados completos continuam preservados nos 27 documentos estaduais.

Isso cria uma hierarquia interessante para o RAG:

```
Pergunta nacional
       ↓
Panorama geográfico Brasil

Pergunta estadual
       ↓
Documento geográfico da UF

Pergunta municipal
       ↓
Evidências municipais contidas
no documento da respectiva UF
```

## 1. Função para criar o panorama geográfico nacional

In [107]:
# ============================================================
# CRIAR PANORAMA GEOGRÁFICO NACIONAL
# ============================================================
def criar_documento_panorama_geografico_brasil(
    df_incidencia,
    df_criticidade,
    df_casos_residencia,
    df_casos_notificacao,
    semana_inicial,
    semana_final,
    ano=ANO,
    top_n=10
):
    """
    Cria um documento semântico nacional com o panorama
    geográfico dos registros de dengue.

    Inclui:
    - distribuição por UF de residência;
    - distribuição por UF de notificação;
    - municípios com mais registros;
    - municípios com maior incidência;
    - distribuição nacional da criticidade.
    """

    # ========================================================
    # TOTAIS NACIONAIS
    # ========================================================

    total_casos_residencia = int(
        df_casos_residencia[
            "TOTAL_CASOS"
        ].sum()
    )

    total_casos_notificacao = int(
        df_casos_notificacao[
            "TOTAL_CASOS"
        ].sum()
    )

    total_ufs_residencia = int(
        df_casos_residencia[
            "UF_RESIDENCIA"
        ]
        .dropna()
        .nunique()
    )

    total_ufs_notificacao = int(
        df_casos_notificacao[
            "SG_UF_NOT"
        ]
        .dropna()
        .nunique()
    )

    total_municipios_analisados = int(
        df_incidencia[
            "CODIGO_MUNICIPIO_7"
        ]
        .dropna()
        .nunique()
    )

    # ========================================================
    # UF COM MAIS REGISTROS POR RESIDÊNCIA
    # ========================================================

    linha_uf_residencia = (
        df_casos_residencia.loc[
            df_casos_residencia[
                "TOTAL_CASOS"
            ].idxmax()
        ]
    )

    uf_maior_residencia = str(
        linha_uf_residencia[
            "UF_RESIDENCIA"
        ]
    )

    codigo_uf_maior_residencia = str(
        linha_uf_residencia[
            "SG_UF"
        ]
    )

    total_uf_maior_residencia = int(
        linha_uf_residencia[
            "TOTAL_CASOS"
        ]
    )

    # ========================================================
    # UF COM MAIS REGISTROS POR NOTIFICAÇÃO
    # ========================================================

    linha_uf_notificacao = (
        df_casos_notificacao.loc[
            df_casos_notificacao[
                "TOTAL_CASOS"
            ].idxmax()
        ]
    )

    uf_maior_notificacao = str(
        linha_uf_notificacao[
            "UF_NAME"
        ]
    )

    codigo_uf_maior_notificacao = str(
        linha_uf_notificacao[
            "SG_UF_NOT"
        ]
    )

    total_uf_maior_notificacao = int(
        linha_uf_notificacao[
            "TOTAL_CASOS"
        ]
    )

    # ========================================================
    # TOP MUNICÍPIOS POR NÚMERO DE REGISTROS
    # ========================================================

    top_casos = (
        df_incidencia[
            df_incidencia[
                "MUNICIPIO_NAME"
            ].notna()
        ]
        .sort_values(
            "CASOS",
            ascending=False
        )
        .head(top_n)
    )

    evidencias_top_casos = []

    for _, linha in top_casos.iterrows():

        evidencias_top_casos.append(
            {
                "codigo_municipio": str(
                    linha[
                        "CODIGO_MUNICIPIO_7"
                    ]
                ),
                "municipio": str(
                    linha[
                        "MUNICIPIO_NAME"
                    ]
                ),
                "uf": str(
                    linha[
                        "UF_RESIDENCIA"
                    ]
                ),
                "casos": int(
                    linha[
                        "CASOS"
                    ]
                ),
                "populacao_2022":
                    normalizar_valor(
                        linha[
                            "POPULACAO_2022"
                        ]
                    ),
                "incidencia_100mil":
                    normalizar_valor(
                        linha[
                            "INCIDENCIA_100MIL"
                        ]
                    ),
                "grau_criticidade": str(
                    linha[
                        "GRAU_CRITICIDADE"
                    ]
                )
            }
        )

    # ========================================================
    # TOP MUNICÍPIOS POR INCIDÊNCIA
    # ========================================================

    top_incidencia = (
        df_incidencia[
            df_incidencia[
                "INCIDENCIA_100MIL"
            ].notna()
            &
            df_incidencia[
                "MUNICIPIO_NAME"
            ].notna()
        ]
        .sort_values(
            "INCIDENCIA_100MIL",
            ascending=False
        )
        .head(top_n)
    )

    evidencias_top_incidencia = []

    for _, linha in top_incidencia.iterrows():

        evidencias_top_incidencia.append(
            {
                "codigo_municipio": str(
                    linha[
                        "CODIGO_MUNICIPIO_7"
                    ]
                ),
                "municipio": str(
                    linha[
                        "MUNICIPIO_NAME"
                    ]
                ),
                "uf": str(
                    linha[
                        "UF_RESIDENCIA"
                    ]
                ),
                "casos": int(
                    linha[
                        "CASOS"
                    ]
                ),
                "populacao_2022":
                    normalizar_valor(
                        linha[
                            "POPULACAO_2022"
                        ]
                    ),
                "incidencia_100mil": float(
                    linha[
                        "INCIDENCIA_100MIL"
                    ]
                ),
                "grau_criticidade": str(
                    linha[
                        "GRAU_CRITICIDADE"
                    ]
                )
            }
        )

    # ========================================================
    # MUNICÍPIO COM MAIS REGISTROS
    # ========================================================

    municipio_maior_casos = (
        evidencias_top_casos[0]
        if evidencias_top_casos
        else None
    )

    # ========================================================
    # MUNICÍPIO COM MAIOR INCIDÊNCIA
    # ========================================================

    municipio_maior_incidencia = (
        evidencias_top_incidencia[0]
        if evidencias_top_incidencia
        else None
    )

    # ========================================================
    # DISTRIBUIÇÃO NACIONAL DA CRITICIDADE
    # ========================================================

    criticidade_nacional = (
        df_criticidade
        .groupby(
            "GRAU_CRITICIDADE",
            as_index=False
        )["TOTAL_MUNICIPIOS"]
        .sum()
        .sort_values(
            "TOTAL_MUNICIPIOS",
            ascending=False
        )
    )

    evidencias_criticidade = []

    for _, linha in criticidade_nacional.iterrows():

        evidencias_criticidade.append(
            {
                "grau_criticidade": str(
                    linha[
                        "GRAU_CRITICIDADE"
                    ]
                ),
                "total_municipios": int(
                    linha[
                        "TOTAL_MUNICIPIOS"
                    ]
                )
            }
        )

    # ========================================================
    # DISTRIBUIÇÃO POR UF DE RESIDÊNCIA
    # ========================================================

    evidencias_residencia = []

    for _, linha in (
        df_casos_residencia
        .sort_values(
            "TOTAL_CASOS",
            ascending=False
        )
        .iterrows()
    ):

        evidencias_residencia.append(
            {
                "codigo_uf": str(
                    linha[
                        "SG_UF"
                    ]
                ),
                "uf": str(
                    linha[
                        "UF_RESIDENCIA"
                    ]
                ),
                "total_casos": int(
                    linha[
                        "TOTAL_CASOS"
                    ]
                )
            }
        )

    # ========================================================
    # DISTRIBUIÇÃO POR UF DE NOTIFICAÇÃO
    # ========================================================

    evidencias_notificacao = []

    for _, linha in (
        df_casos_notificacao
        .sort_values(
            "TOTAL_CASOS",
            ascending=False
        )
        .iterrows()
    ):

        evidencias_notificacao.append(
            {
                "codigo_uf": str(
                    linha[
                        "SG_UF_NOT"
                    ]
                ),
                "uf": str(
                    linha[
                        "UF_NAME"
                    ]
                ),
                "total_casos": int(
                    linha[
                        "TOTAL_CASOS"
                    ]
                )
            }
        )

    # ========================================================
    # INTERPRETAÇÃO
    # ========================================================

    interpretacao = (
        f"No período analisado, {uf_maior_residencia} apresentou "
        f"o maior número de registros segundo a UF de residência, "
        f"com {total_uf_maior_residencia:,} registros. "
        f"Considerando a UF de notificação, "
        f"{uf_maior_notificacao} apresentou o maior total, "
        f"com {total_uf_maior_notificacao:,} registros."
    )

    if municipio_maior_casos is not None:

        interpretacao += (
            f" No nível municipal, "
            f"{municipio_maior_casos['municipio']} "
            f"({municipio_maior_casos['uf']}) apresentou "
            f"o maior número absoluto entre os municípios "
            f"analisados, com "
            f"{municipio_maior_casos['casos']:,} registros."
        )

    if municipio_maior_incidencia is not None:

        interpretacao += (
            f" A maior incidência municipal foi observada em "
            f"{municipio_maior_incidencia['municipio']} "
            f"({municipio_maior_incidencia['uf']}), com "
            f"{municipio_maior_incidencia['incidencia_100mil']:,.2f} "
            f"registros por 100 mil habitantes."
        )

    # ========================================================
    # DOCUMENTO SEMÂNTICO
    # ========================================================

    documento = criar_documento_semantico(

        document_id=gerar_document_id(
            fonte="SINAN",
            doenca="DENGUE",
            ano=ano,
            dominio="GEOGRAFICO",
            localizacao="BRASIL"
        ),

        titulo=(
            f"Panorama geográfico dos registros "
            f"de dengue no Brasil — {ano}"
        ),

        tipo_documento=(
            "panorama_geografico_nacional"
        ),

        dominio="geografico",

        sintese=(
            f"Este documento apresenta um panorama nacional "
            f"da distribuição geográfica dos registros de dengue "
            f"no SINAN em {ano}, considerando as semanas "
            f"epidemiológicas {semana_inicial} a {semana_final}, "
            f"as UFs de residência e de notificação, a distribuição "
            f"municipal, a incidência por 100 mil habitantes e a "
            f"classificação de criticidade."
        ),

        indicadores={
            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final - semana_inicial + 1,

            "total_casos_residencia":
                total_casos_residencia,

            "total_casos_notificacao":
                total_casos_notificacao,

            "total_ufs_residencia":
                total_ufs_residencia,

            "total_ufs_notificacao":
                total_ufs_notificacao,

            "total_municipios_analisados":
                total_municipios_analisados,

            "uf_maior_numero_residencia":
                uf_maior_residencia,

            "codigo_uf_maior_numero_residencia":
                codigo_uf_maior_residencia,

            "total_maior_uf_residencia":
                total_uf_maior_residencia,

            "uf_maior_numero_notificacao":
                uf_maior_notificacao,

            "codigo_uf_maior_numero_notificacao":
                codigo_uf_maior_notificacao,

            "total_maior_uf_notificacao":
                total_uf_maior_notificacao,

            "municipio_maior_numero_casos": (
                municipio_maior_casos[
                    "municipio"
                ]
                if municipio_maior_casos
                else None
            ),

            "maior_numero_casos_municipio": (
                municipio_maior_casos[
                    "casos"
                ]
                if municipio_maior_casos
                else None
            ),

            "municipio_maior_incidencia": (
                municipio_maior_incidencia[
                    "municipio"
                ]
                if municipio_maior_incidencia
                else None
            ),

            "maior_incidencia_100mil": (
                municipio_maior_incidencia[
                    "incidencia_100mil"
                ]
                if municipio_maior_incidencia
                else None
            )
        },

        evidencias={

            "distribuicao_por_uf_residencia":
                evidencias_residencia,

            "distribuicao_por_uf_notificacao":
                evidencias_notificacao,

            "criticidade_municipal_nacional":
                evidencias_criticidade,

            "municipios_maior_numero_registros":
                evidencias_top_casos,

            "municipios_maior_incidencia":
                evidencias_top_incidencia
        },

        interpretacao=interpretacao,

        escopo={
            "pais": "Brasil",
            "ano": ano,
            "semana_inicial":
                semana_inicial,
            "semana_final":
                semana_final,
            "total_municipios_analisados":
                total_municipios_analisados
        },

        observacoes_dados=[
            (
                f"Os resultados correspondem ao período disponível "
                f"entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}."
            ),
            (
                "Os totais segundo UF de residência e UF de "
                "notificação correspondem a perspectivas "
                "geográficas distintas."
            ),
            (
                "A incidência municipal é calculada em relação "
                "à população utilizada como referência no "
                "processo de enriquecimento dos dados."
            ),
            (
                "Municípios com populações pequenas podem "
                "apresentar elevada incidência mesmo quando "
                "o número absoluto de registros é relativamente baixo."
            ),
            (
                f"As listas municipais apresentadas neste panorama "
                f"estão limitadas aos {top_n} maiores valores. "
                f"A informação municipal completa está preservada "
                f"nos documentos geográficos específicos das UFs."
            )
        ],

        conceitos_semanticos=[
            "Dengue",
            "Brasil",
            "Município",
            "Unidade Federativa",
            "Residência",
            "Local de notificação",
            "Incidência",
            "População",
            "Criticidade"
        ],

        palavras_chave=[
            "dengue",
            "SINAN",
            "Brasil",
            "município",
            "UF",
            "incidência",
            "casos",
            "residência",
            "notificação",
            "criticidade",
            "distribuição geográfica"
        ],

        fonte="SINAN",
        doenca="dengue",
        ano=ano
    )

    return documento

## 2. Gerar o documento

In [108]:
# ============================================================
# GERAR PANORAMA GEOGRÁFICO DO BRASIL
# ============================================================

documento_geografico_brasil = (
    criar_documento_panorama_geografico_brasil(
        df_incidencia=df_incidencia_municipio,
        df_criticidade=df_criticidade_uf,
        df_casos_residencia=df_casos_uf_residencia,
        df_casos_notificacao=df_casos_uf_notificacao,
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO,
        top_n=10
    )
)

## 3. Antes de salvar, faça uma validação rápida

In [109]:
print(
    "Document ID:",
    documento_geografico_brasil[
        "document_id"
    ]
)

print(
    "\nTotal por residência:",
    documento_geografico_brasil[
        "indicadores"
    ][
        "total_casos_residencia"
    ]
)

print(
    "Total por notificação:",
    documento_geografico_brasil[
        "indicadores"
    ][
        "total_casos_notificacao"
    ]
)

print(
    "\nUF com maior total por residência:",
    documento_geografico_brasil[
        "indicadores"
    ][
        "uf_maior_numero_residencia"
    ]
)

print(
    "UF com maior total por notificação:",
    documento_geografico_brasil[
        "indicadores"
    ][
        "uf_maior_numero_notificacao"
    ]
)

print(
    "\nMunicípio com maior número de registros:",
    documento_geografico_brasil[
        "indicadores"
    ][
        "municipio_maior_numero_casos"
    ]
)

print(
    "Município com maior incidência:",
    documento_geografico_brasil[
        "indicadores"
    ][
        "municipio_maior_incidencia"
    ]
)

Document ID: SINAN_DENGUE_2026_GEOGRAFICO_BRASIL

Total por residência: 444266
Total por notificação: 444266

UF com maior total por residência: GO
UF com maior total por notificação: Goiás

Município com maior número de registros: Goiânia
Município com maior incidência: Turvânia


Também visualize o Markdown:

In [110]:
print(
    documento_para_markdown(
        documento_geografico_brasil
    )
)

# Panorama geográfico dos registros de dengue no Brasil — 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_GEOGRAFICO_BRASIL
- **Tipo de documento:** panorama_geografico_nacional
- **Domínio:** geografico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Pais:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total municipios analisados:** 4711

## Síntese epidemiológica

Este documento apresenta um panorama nacional da distribuição geográfica dos registros de dengue no SINAN em 2026, considerando as semanas epidemiológicas 1 a 34, as UFs de residência e de notificação, a distribuição municipal, a incidência por 100 mil habitantes e a classificação de criticidade.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total casos residencia:** 444266
- **Total casos notificacao:** 444266
- **Total ufs residencia:** 27
- **Total ufs notificacao:** 27
- **Total municipios analisados:** 47

## Salvar

In [111]:
# ============================================================
# SALVAR PANORAMA GEOGRÁFICO NACIONAL
# ============================================================

arquivos_panorama_geografico = (
    salvar_documento_semantico(
        documento=documento_geografico_brasil,
        pasta_saida=PASTA_DOCS_PANORAMA_GEOGRAFICO
    )
)

print("=" * 70)
print("PANORAMA GEOGRÁFICO NACIONAL SALVO")
print("=" * 70)

print(
    f"JSON:\n"
    f"{arquivos_panorama_geografico['json']}"
)

print(
    f"\nMarkdown:\n"
    f"{arquivos_panorama_geografico['markdown']}"
)

print("\n" + "=" * 70)

print(
    "JSON existe:",
    arquivos_panorama_geografico[
        "json"
    ].exists()
)

print(
    "Markdown existe:",
    arquivos_panorama_geografico[
        "markdown"
    ].exists()
)

PANORAMA GEOGRÁFICO NACIONAL SALVO
JSON:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/geograficos/panorama_geografico/sinan_dengue_2026_geografico_brasil.json

Markdown:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/geograficos/panorama_geografico/sinan_dengue_2026_geografico_brasil.md

JSON existe: True
Markdown existe: True


Verificação final do domínio geográfico

Vale fechar essa etapa com uma célula de auditoria:

In [112]:
# ============================================================
# VERIFICAR DOCUMENTOS DO DOMÍNIO GEOGRÁFICO
# ============================================================

json_uf = list(
    PASTA_DOCS_GEOGRAFICOS_UF.glob("*.json")
)

markdown_uf = list(
    PASTA_DOCS_GEOGRAFICOS_UF.glob("*.md")
)

json_panorama = list(
    PASTA_DOCS_PANORAMA_GEOGRAFICO.glob("*.json")
)

markdown_panorama = list(
    PASTA_DOCS_PANORAMA_GEOGRAFICO.glob("*.md")
)

total_documentos = (
    len(json_uf)
    + len(json_panorama)
)

total_arquivos = (
    len(json_uf)
    + len(markdown_uf)
    + len(json_panorama)
    + len(markdown_panorama)
)

print("=" * 70)
print("RESUMO DO DOMÍNIO GEOGRÁFICO")
print("=" * 70)

print(
    f"Documentos por UF: {len(json_uf)}"
)

print(
    f"Panoramas nacionais: {len(json_panorama)}"
)

print(
    f"Total de documentos semânticos: {total_documentos}"
)

print(
    f"Arquivos JSON: "
    f"{len(json_uf) + len(json_panorama)}"
)

print(
    f"Arquivos Markdown: "
    f"{len(markdown_uf) + len(markdown_panorama)}"
)

print(
    f"Total de arquivos físicos: {total_arquivos}"
)

if (
    len(json_uf) == 27
    and len(markdown_uf) == 27
    and len(json_panorama) == 1
    and len(markdown_panorama) == 1
):
    print(
        "\n✓ Domínio geográfico concluído corretamente."
    )
else:
    print(
        "\n⚠ Verifique os arquivos do domínio geográfico."
    )

RESUMO DO DOMÍNIO GEOGRÁFICO
Documentos por UF: 27
Panoramas nacionais: 1
Total de documentos semânticos: 28
Arquivos JSON: 28
Arquivos Markdown: 28
Total de arquivos físicos: 56

✓ Domínio geográfico concluído corretamente.


## 8. Domínio clínico


In [113]:
# ============================================================
# INSPECIONAR PRODUTOS ANALÍTICOS DO DOMÍNIO CLÍNICO
# ============================================================

dataframes_clinicos = {
    "sintomas_brasil":
        df_sintomas_brasil,

    "sintomas_uf":
        df_sintomas_uf,

    "doencas_preexistentes_brasil":
        df_doencas_preexistentes_brasil,

    "doencas_preexistentes_uf":
        df_doencas_preexistentes_uf,
}

for nome, df in dataframes_clinicos.items():

    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    print(
        f"Dimensão: "
        f"{df.shape[0]:,} linhas x "
        f"{df.shape[1]} colunas"
    )

    print(
        "Colunas:",
        df.columns.tolist()
    )

    display(
        df.head(10)
    )


sintomas_brasil
Dimensão: 14 linhas x 9 colunas
Colunas: ['VARIAVEL', 'SINAL_CLINICO', 'TOTAL_REGISTROS', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,VARIAVEL,SINAL_CLINICO,TOTAL_REGISTROS,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,FEBRE_DECODED,Febre,444266,428673,378134,50539,15593,88.21,3.51
1,MIALGIA_DECODED,Mialgia,444266,428673,339154,89519,15593,79.12,3.51
2,CEFALEIA_DECODED,Cefaleia,444266,428673,337325,91348,15593,78.69,3.51
3,NAUSEA_DECODED,Náusea,444266,428673,176360,252313,15593,41.14,3.51
4,VOMITO_DECODED,Vômito,444266,428673,123997,304676,15593,28.93,3.51
5,DOR_RETRO_DECODED,Dor retro-orbital,444266,428673,122819,305854,15593,28.65,3.51
6,DOR_COSTAS_DECODED,Dor nas costas,444266,428673,107302,321371,15593,25.03,3.51
7,ARTRALGIA_DECODED,Artralgia,444266,428673,78961,349712,15593,18.42,3.51
8,EXANTEMA_DECODED,Exantema,444266,428673,42524,386149,15593,9.92,3.51
9,ARTRITE_DECODED,Artrite,444266,428673,36240,392433,15593,8.45,3.51



sintomas_uf
Dimensão: 378 linhas x 11 colunas
Colunas: ['SG_UF_NOT', 'UF_NAME', 'VARIAVEL', 'SINAL_CLINICO', 'TOTAL_REGISTROS', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,SG_UF_NOT,UF_NAME,VARIAVEL,SINAL_CLINICO,TOTAL_REGISTROS,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,11,Rondônia,FEBRE_DECODED,Febre,1354,1103,1009,94,251,91.48,18.54
1,11,Rondônia,MIALGIA_DECODED,Mialgia,1354,1103,778,325,251,70.53,18.54
2,11,Rondônia,CEFALEIA_DECODED,Cefaleia,1354,1103,849,254,251,76.97,18.54
3,11,Rondônia,EXANTEMA_DECODED,Exantema,1354,1103,110,993,251,9.97,18.54
4,11,Rondônia,VOMITO_DECODED,Vômito,1354,1103,299,804,251,27.11,18.54
5,11,Rondônia,NAUSEA_DECODED,Náusea,1354,1103,417,686,251,37.81,18.54
6,11,Rondônia,DOR_COSTAS_DECODED,Dor nas costas,1354,1103,320,783,251,29.01,18.54
7,11,Rondônia,CONJUNTVIT_DECODED,Conjuntivite,1354,1103,28,1075,251,2.54,18.54
8,11,Rondônia,ARTRITE_DECODED,Artrite,1354,1103,171,932,251,15.50,18.54
9,11,Rondônia,ARTRALGIA_DECODED,Artralgia,1354,1103,277,826,251,25.11,18.54



doencas_preexistentes_brasil
Dimensão: 7 linhas x 9 colunas
Colunas: ['VARIAVEL', 'DOENCA_PREEXISTENTE', 'TOTAL_REGISTROS', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,VARIAVEL,DOENCA_PREEXISTENTE,TOTAL_REGISTROS,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,HIPERTENSA_DECODED,Hipertensão arterial,444266,428667,33179,395488,15599,7.74,3.51
1,DIABETES_DECODED,Diabetes,444266,428668,15374,413294,15598,3.59,3.51
2,AUTO_IMUNE_DECODED,Doença autoimune,444266,428668,2761,425907,15598,0.64,3.51
3,HEMATOLOG_DECODED,Doença hematológica,444266,428669,2306,426363,15597,0.54,3.51
4,HEPATOPAT_DECODED,Hepatopatia,444266,428668,2259,426409,15598,0.53,3.51
5,RENAL_DECODED,Doença renal,444266,428668,2177,426491,15598,0.51,3.51
6,ACIDO_PEPT_DECODED,Doença ácido-péptica,444266,428668,2139,426529,15598,0.50,3.51



doencas_preexistentes_uf
Dimensão: 189 linhas x 11 colunas
Colunas: ['SG_UF_NOT', 'UF_NAME', 'VARIAVEL', 'DOENCA_PREEXISTENTE', 'TOTAL_REGISTROS', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,SG_UF_NOT,UF_NAME,VARIAVEL,DOENCA_PREEXISTENTE,TOTAL_REGISTROS,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,11,Rondônia,DIABETES_DECODED,Diabetes,1354,1103,32,1071,251,2.90,18.54
1,11,Rondônia,HEMATOLOG_DECODED,Doença hematológica,1354,1103,8,1095,251,0.73,18.54
2,11,Rondônia,HEPATOPAT_DECODED,Hepatopatia,1354,1103,7,1096,251,0.63,18.54
3,11,Rondônia,RENAL_DECODED,Doença renal,1354,1103,8,1095,251,0.73,18.54
4,11,Rondônia,HIPERTENSA_DECODED,Hipertensão arterial,1354,1103,54,1049,251,4.90,18.54
5,11,Rondônia,ACIDO_PEPT_DECODED,Doença ácido-péptica,1354,1103,6,1097,251,0.54,18.54
6,11,Rondônia,AUTO_IMUNE_DECODED,Doença autoimune,1354,1103,7,1096,251,0.63,18.54
7,12,Acre,DIABETES_DECODED,Diabetes,1832,1645,43,1602,187,2.61,10.21
8,12,Acre,HEMATOLOG_DECODED,Doença hematológica,1832,1645,6,1639,187,0.36,10.21
9,12,Acre,HEPATOPAT_DECODED,Hepatopatia,1832,1645,7,1638,187,0.43,10.21


In [114]:
# ============================================================
# VERIFICAR COBERTURA GEOGRÁFICA DOS PRODUTOS CLÍNICOS
# ============================================================

print("=" * 70)
print("COBERTURA DOS PRODUTOS CLÍNICOS")
print("=" * 70)

for nome, df in {
    "Sintomas": df_sintomas_uf,
    "Doenças preexistentes":
        df_doencas_preexistentes_uf
}.items():

    print(f"\n{nome}")

    print(
        "Total de linhas:",
        len(df)
    )

    if "SG_UF_NOT" in df.columns:

        print(
            "Quantidade de UFs:",
            df["SG_UF_NOT"]
            .dropna()
            .nunique()
        )

        print(
            "Códigos:",
            sorted(
                df["SG_UF_NOT"]
                .dropna()
                .astype(str)
                .unique()
            )
        )

COBERTURA DOS PRODUTOS CLÍNICOS

Sintomas
Total de linhas: 378
Quantidade de UFs: 27
Códigos: ['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26', '27', '28', '29', '31', '32', '33', '35', '41', '42', '43', '50', '51', '52', '53']

Doenças preexistentes
Total de linhas: 189
Quantidade de UFs: 27
Códigos: ['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26', '27', '28', '29', '31', '32', '33', '35', '41', '42', '43', '50', '51', '52', '53']


In [115]:
print("=" * 70)
print("ESCOPO TEMPORAL")
print("=" * 70)

print("Ano:", ANO)
print("Semana inicial:", SEMANA_INICIAL)
print("Semana final:", SEMANA_FINAL)
print("Total de semanas:", TOTAL_SEMANAS)

ESCOPO TEMPORAL
Ano: 2026
Semana inicial: 1
Semana final: 34
Total de semanas: 34


A saída confirma que os produtos clínicos estão bem estruturados para o modelo que planejamos. Temos 14 sinais clínicos no nível nacional e 378 linhas no nível UF, compatíveis com 14 sinais × 27 UFs. Também temos 7 doenças preexistentes nacionalmente e 189 linhas por UF, isto é, 7 × 27 UFs.

Isso permite criar um único documento clínico por UF, com duas categorias de evidências:
```
Perfil clínico da UF
│
├── Sinais clínicos
│   ├── Febre
│   ├── Mialgia
│   ├── Cefaleia
│   └── ...
│
└── Doenças preexistentes
    ├── Hipertensão
    ├── Diabetes
    ├── Doença renal
    └── ...
```

## 1. Função de interpretação clínica

A interpretação deve ser descritiva. Não devemos afirmar, por exemplo, que hipertensão "aumentou o risco" de dengue, pois esses DataFrames não permitem estabelecer essa relação.

In [116]:
# ============================================================
# INTERPRETAÇÃO DO PERFIL CLÍNICO POR UF
# ============================================================

def interpretar_clinico_uf(
    uf_nome,
    total_registros,
    sinal_mais_frequente,
    percentual_sinal,
    doenca_mais_frequente,
    percentual_doenca,
    semana_inicial,
    semana_final,
    ano
):

    return (
        f"Na UF {uf_nome}, foram considerados "
        f"{total_registros:,} registros de dengue entre "
        f"as semanas epidemiológicas {semana_inicial} e "
        f"{semana_final} de {ano}. "
        f"Entre os sinais clínicos avaliados, "
        f"{sinal_mais_frequente} apresentou a maior "
        f"frequência de registros com resposta positiva, "
        f"correspondendo a {percentual_sinal:.2f}% dos "
        f"registros avaliáveis para essa variável. "
        f"Entre as doenças preexistentes avaliadas, "
        f"{doenca_mais_frequente} apresentou a maior "
        f"frequência de resposta positiva, correspondendo "
        f"a {percentual_doenca:.2f}% dos registros "
        f"avaliáveis para essa variável."
    )

A expressão "dos registros avaliáveis para essa variável" é importante. Por exemplo, nacionalmente, febre tem 425.929 registros totais, mas 409.648 avaliáveis; os 88,03% são calculados sobre os avaliáveis.

## 2. Gerador dos 27 documentos clínicos

In [117]:
# ============================================================
# CRIAR DOCUMENTOS CLÍNICOS POR UF
# ============================================================

def criar_documentos_clinicos_por_uf(
    df_sintomas,
    df_doencas,
    semana_inicial,
    semana_final,
    ano=ANO
):

    documentos = []

    # ========================================================
    # UNIVERSO DAS UFs
    # ========================================================

    ufs = (
        df_sintomas[
            [
                "SG_UF_NOT",
                "UF_NAME"
            ]
        ]
        .drop_duplicates()
        .sort_values("SG_UF_NOT")
    )

    for _, linha_uf in ufs.iterrows():

        codigo_uf = str(
            linha_uf["SG_UF_NOT"]
        )

        nome_uf = linha_uf[
            "UF_NAME"
        ]

        # ====================================================
        # SINAIS CLÍNICOS DA UF
        # ====================================================

        sintomas_uf = (
            df_sintomas[
                df_sintomas["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        sintomas_uf = (
            sintomas_uf
            .sort_values(
                "PERCENTUAL_SIM",
                ascending=False
            )
        )

        # ====================================================
        # DOENÇAS PREEXISTENTES DA UF
        # ====================================================

        doencas_uf = (
            df_doencas[
                df_doencas["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        doencas_uf = (
            doencas_uf
            .sort_values(
                "PERCENTUAL_SIM",
                ascending=False
            )
        )

        # ====================================================
        # TOTAL DE REGISTROS
        # ====================================================

        total_registros = int(
            sintomas_uf[
                "TOTAL_REGISTROS"
            ]
            .iloc[0]
        )

        # ====================================================
        # PRINCIPAIS INDICADORES
        # ====================================================

        sinal_mais_frequente = (
            sintomas_uf.iloc[0][
                "SINAL_CLINICO"
            ]
        )

        percentual_sinal = float(
            sintomas_uf.iloc[0][
                "PERCENTUAL_SIM"
            ]
        )

        doenca_mais_frequente = (
            doencas_uf.iloc[0][
                "DOENCA_PREEXISTENTE"
            ]
        )

        percentual_doenca = float(
            doencas_uf.iloc[0][
                "PERCENTUAL_SIM"
            ]
        )

        # ====================================================
        # EVIDÊNCIAS - SINAIS CLÍNICOS
        # ====================================================

        evidencias_sintomas = []

        for _, linha in sintomas_uf.iterrows():

            evidencias_sintomas.append(
                {
                    "variavel":
                        linha["VARIAVEL"],

                    "sinal_clinico":
                        linha["SINAL_CLINICO"],

                    "total_registros":
                        int(
                            linha[
                                "TOTAL_REGISTROS"
                            ]
                        ),

                    "avaliaveis":
                        int(
                            linha["AVALIAVEIS"]
                        ),

                    "sim":
                        int(
                            linha["SIM"]
                        ),

                    "nao":
                        int(
                            linha["NAO"]
                        ),

                    "ausentes":
                        int(
                            linha["AUSENTES"]
                        ),

                    "percentual_sim":
                        float(
                            linha[
                                "PERCENTUAL_SIM"
                            ]
                        ),

                    "percentual_ausentes":
                        float(
                            linha[
                                "PERCENTUAL_AUSENTES"
                            ]
                        )
                }
            )

        # ====================================================
        # EVIDÊNCIAS - DOENÇAS PREEXISTENTES
        # ====================================================

        evidencias_doencas = []

        for _, linha in doencas_uf.iterrows():

            evidencias_doencas.append(
                {
                    "variavel":
                        linha["VARIAVEL"],

                    "doenca_preexistente":
                        linha[
                            "DOENCA_PREEXISTENTE"
                        ],

                    "total_registros":
                        int(
                            linha[
                                "TOTAL_REGISTROS"
                            ]
                        ),

                    "avaliaveis":
                        int(
                            linha["AVALIAVEIS"]
                        ),

                    "sim":
                        int(
                            linha["SIM"]
                        ),

                    "nao":
                        int(
                            linha["NAO"]
                        ),

                    "ausentes":
                        int(
                            linha["AUSENTES"]
                        ),

                    "percentual_sim":
                        float(
                            linha[
                                "PERCENTUAL_SIM"
                            ]
                        ),

                    "percentual_ausentes":
                        float(
                            linha[
                                "PERCENTUAL_AUSENTES"
                            ]
                        )
                }
            )

        # ====================================================
        # INTERPRETAÇÃO
        # ====================================================

        interpretacao = (
            interpretar_clinico_uf(
                uf_nome=nome_uf,
                total_registros=total_registros,
                sinal_mais_frequente=(
                    sinal_mais_frequente
                ),
                percentual_sinal=(
                    percentual_sinal
                ),
                doenca_mais_frequente=(
                    doenca_mais_frequente
                ),
                percentual_doenca=(
                    percentual_doenca
                ),
                semana_inicial=semana_inicial,
                semana_final=semana_final,
                ano=ano
            )
        )

        # ====================================================
        # DOCUMENTO SEMÂNTICO
        # ====================================================

        documento = criar_documento_semantico(

            document_id=gerar_document_id(
                fonte="SINAN",
                doenca="dengue",
                ano=ano,
                dominio="clinico",
                localizacao=codigo_uf
            ),

            titulo=(
                f"Perfil clínico dos registros de dengue "
                f"em {nome_uf} - {ano}"
            ),

            tipo_documento="perfil_clinico_uf",

            dominio="clinico",

            sintese=(
                f"Este documento apresenta o perfil clínico "
                f"dos registros de dengue no SINAN em "
                f"{nome_uf}, considerando as semanas "
                f"epidemiológicas {semana_inicial} a "
                f"{semana_final} de {ano}. "
                f"São apresentadas informações sobre sinais "
                f"clínicos e doenças preexistentes."
            ),

            indicadores={
                "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1,

                "total_registros":
                    total_registros,

                "sinal_clinico_maior_frequencia":
                    sinal_mais_frequente,

                "percentual_sinal_clinico":
                    percentual_sinal,

                "doenca_preexistente_maior_frequencia":
                    doenca_mais_frequente,

                "percentual_doenca_preexistente":
                    percentual_doenca
            },

            evidencias={
                "sinais_clinicos":
                    evidencias_sintomas,

                "doencas_preexistentes":
                    evidencias_doencas
            },

            interpretacao=interpretacao,

            escopo={
                "codigo_uf":
                    codigo_uf,

                "uf_nome":
                    nome_uf,

                "ano":
                    ano,

                "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1
            },

            observacoes_dados=[
                (
                    f"Os resultados correspondem aos "
                    f"registros disponíveis entre as semanas "
                    f"epidemiológicas {semana_inicial} e "
                    f"{semana_final} de {ano}."
                ),
                (
                    "Os percentuais de resposta positiva "
                    "são calculados sobre os registros "
                    "avaliáveis de cada variável."
                ),
                (
                    "A ausência de preenchimento é "
                    "apresentada separadamente para cada "
                    "sinal clínico e doença preexistente."
                ),
                (
                    "As frequências apresentadas possuem "
                    "caráter descritivo e não estabelecem "
                    "relações causais entre condições "
                    "preexistentes e dengue."
                )
            ],

            conceitos_semanticos=[
                "dengue",
                "sinal clínico",
                "sintoma",
                "doença preexistente",
                "perfil clínico",
                "semana epidemiológica",
                "unidade federativa",
                "vigilância epidemiológica"
            ],

            palavras_chave=[
                "dengue",
                "sinais clínicos",
                "sintomas",
                "doenças preexistentes",
                "perfil clínico",
                nome_uf,
                codigo_uf,
                str(ano),
                "SINAN"
            ],

            ano=ano
        )

        documentos.append(
            documento
        )

    return documentos

## 3. Gerar os documentos

Agora usando os nomes independentes do ano:

In [118]:
documentos_clinicos_uf = (
    criar_documentos_clinicos_por_uf(
        df_sintomas=df_sintomas_uf,
        df_doencas=(
            df_doencas_preexistentes_uf
        ),
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO
    )
)

print(
    "Total de documentos clínicos gerados:",
    len(documentos_clinicos_uf)
)

Total de documentos clínicos gerados: 27


## 4. Não salvar ainda

Primeiro vamos verificar um documento inteiro:

In [119]:
print(
    documento_para_markdown(
        documentos_clinicos_uf[0]
    )
)

# Perfil clínico dos registros de dengue em Rondônia - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_CLINICO_11
- **Tipo de documento:** perfil_clinico_uf
- **Domínio:** clinico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 11
- **Uf nome:** Rondônia
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34

## Síntese epidemiológica

Este documento apresenta o perfil clínico dos registros de dengue no SINAN em Rondônia, considerando as semanas epidemiológicas 1 a 34 de 2026. São apresentadas informações sobre sinais clínicos e doenças preexistentes.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros:** 1354
- **Sinal clinico maior frequencia:** Febre
- **Percentual sinal clinico:** 91.48
- **Doenca preexistente maior frequencia:** Hipertensão arterial
- **Percentual doenca preexistente:** 4.9

## Evidências

### Sinais clinicos

- **Variavel:*

# Salvar

In [120]:
# ============================================================
# DIRETÓRIOS DOS DOCUMENTOS CLÍNICOS
# ============================================================

PASTA_DOCS_CLINICOS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
    / "clinicas"
)

PASTA_DOCS_CLINICOS_UF = (
    PASTA_DOCS_CLINICOS
    / "perfil_clinico_por_uf"
)

PASTA_DOCS_PANORAMA_CLINICO = (
    PASTA_DOCS_CLINICOS
    / "panorama_clinico"
)

PASTA_DOCS_CLINICOS_UF.mkdir(
    parents=True,
    exist_ok=True
)

PASTA_DOCS_PANORAMA_CLINICO.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Pasta dos documentos por UF:",
    PASTA_DOCS_CLINICOS_UF
)

print(
    "Pasta do panorama nacional:",
    PASTA_DOCS_PANORAMA_CLINICO
)

Pasta dos documentos por UF: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/clinicas/perfil_clinico_por_uf
Pasta do panorama nacional: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/clinicas/panorama_clinico


In [121]:
# ============================================================
# SALVAR DOCUMENTOS CLÍNICOS POR UF
# ============================================================

arquivos_clinicos_uf = []

for documento in documentos_clinicos_uf:

    caminhos = salvar_documento_semantico(
        documento=documento,
        pasta_saida=PASTA_DOCS_CLINICOS_UF
    )

    arquivos_clinicos_uf.append(
        {
            "document_id":
                documento["document_id"],

            "json":
                caminhos["json"],

            "markdown":
                caminhos["markdown"]
        }
    )

print("=" * 70)
print("DOCUMENTOS CLÍNICOS POR UF")
print("=" * 70)

print(
    "Documentos salvos:",
    len(arquivos_clinicos_uf)
)

DOCUMENTOS CLÍNICOS POR UF
Documentos salvos: 27


## Panorama clínico nacional

Para o panorama, vamos usar diretamente:

- df_sintomas_brasil
- df_doencas_preexistentes_brasil

Os produtos nacionais contêm 14 sinais clínicos e 7 doenças preexistentes, com totais, avaliáveis, respostas positivas/negativas, ausentes e percentuais.

Crie a função:

In [122]:
# ============================================================
# CRIAR PANORAMA CLÍNICO NACIONAL
# ============================================================

def criar_documento_panorama_clinico_brasil(
    df_sintomas,
    df_doencas,
    semana_inicial,
    semana_final,
    ano=ANO
):

    # ========================================================
    # ORDENAR POR FREQUÊNCIA
    # ========================================================

    sintomas = (
        df_sintomas
        .sort_values(
            "PERCENTUAL_SIM",
            ascending=False
        )
        .copy()
    )

    doencas = (
        df_doencas
        .sort_values(
            "PERCENTUAL_SIM",
            ascending=False
        )
        .copy()
    )

    # ========================================================
    # TOTAL DE REGISTROS
    # ========================================================

    total_registros = int(
        sintomas[
            "TOTAL_REGISTROS"
        ]
        .iloc[0]
    )

    # ========================================================
    # PRINCIPAIS INDICADORES
    # ========================================================

    sinal_mais_frequente = (
        sintomas.iloc[0][
            "SINAL_CLINICO"
        ]
    )

    percentual_sinal = float(
        sintomas.iloc[0][
            "PERCENTUAL_SIM"
        ]
    )

    doenca_mais_frequente = (
        doencas.iloc[0][
            "DOENCA_PREEXISTENTE"
        ]
    )

    percentual_doenca = float(
        doencas.iloc[0][
            "PERCENTUAL_SIM"
        ]
    )

    # ========================================================
    # EVIDÊNCIAS - SINAIS CLÍNICOS
    # ========================================================

    evidencias_sintomas = []

    for _, linha in sintomas.iterrows():

        evidencias_sintomas.append(
            {
                "variavel":
                    linha["VARIAVEL"],

                "sinal_clinico":
                    linha["SINAL_CLINICO"],

                "total_registros":
                    int(
                        linha["TOTAL_REGISTROS"]
                    ),

                "avaliaveis":
                    int(
                        linha["AVALIAVEIS"]
                    ),

                "sim":
                    int(
                        linha["SIM"]
                    ),

                "nao":
                    int(
                        linha["NAO"]
                    ),

                "ausentes":
                    int(
                        linha["AUSENTES"]
                    ),

                "percentual_sim":
                    float(
                        linha["PERCENTUAL_SIM"]
                    ),

                "percentual_ausentes":
                    float(
                        linha[
                            "PERCENTUAL_AUSENTES"
                        ]
                    )
            }
        )

    # ========================================================
    # EVIDÊNCIAS - DOENÇAS PREEXISTENTES
    # ========================================================

    evidencias_doencas = []

    for _, linha in doencas.iterrows():

        evidencias_doencas.append(
            {
                "variavel":
                    linha["VARIAVEL"],

                "doenca_preexistente":
                    linha[
                        "DOENCA_PREEXISTENTE"
                    ],

                "total_registros":
                    int(
                        linha["TOTAL_REGISTROS"]
                    ),

                "avaliaveis":
                    int(
                        linha["AVALIAVEIS"]
                    ),

                "sim":
                    int(
                        linha["SIM"]
                    ),

                "nao":
                    int(
                        linha["NAO"]
                    ),

                "ausentes":
                    int(
                        linha["AUSENTES"]
                    ),

                "percentual_sim":
                    float(
                        linha["PERCENTUAL_SIM"]
                    ),

                "percentual_ausentes":
                    float(
                        linha[
                            "PERCENTUAL_AUSENTES"
                        ]
                    )
            }
        )

    # ========================================================
    # INTERPRETAÇÃO
    # ========================================================

    interpretacao = (
        f"No Brasil, foram considerados "
        f"{total_registros:,} registros de dengue "
        f"entre as semanas epidemiológicas "
        f"{semana_inicial} e {semana_final} de {ano}. "
        f"Entre os sinais clínicos avaliados, "
        f"{sinal_mais_frequente} apresentou a maior "
        f"frequência de respostas positivas, com "
        f"{percentual_sinal:.2f}% dos registros avaliáveis "
        f"para essa variável. "
        f"Entre as doenças preexistentes avaliadas, "
        f"{doenca_mais_frequente} apresentou a maior "
        f"frequência de respostas positivas, com "
        f"{percentual_doenca:.2f}% dos registros avaliáveis "
        f"para essa variável."
    )

    # ========================================================
    # DOCUMENTO
    # ========================================================

    documento = criar_documento_semantico(

        document_id=gerar_document_id(
            fonte="SINAN",
            doenca="dengue",
            ano=ano,
            dominio="clinico",
            localizacao="BRASIL"
        ),

        titulo=(
            f"Panorama clínico dos registros "
            f"de dengue no Brasil - {ano}"
        ),

        tipo_documento=(
            "panorama_clinico_nacional"
        ),

        dominio="clinico",

        sintese=(
            f"Este documento apresenta o panorama "
            f"nacional do perfil clínico dos registros "
            f"de dengue no SINAN entre as semanas "
            f"epidemiológicas {semana_inicial} e "
            f"{semana_final} de {ano}. "
            f"São apresentadas informações agregadas "
            f"sobre sinais clínicos e doenças "
            f"preexistentes."
        ),

        indicadores={
            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final
                - semana_inicial
                + 1,

            "total_registros":
                total_registros,

            "sinal_clinico_maior_frequencia":
                sinal_mais_frequente,

            "percentual_sinal_clinico":
                percentual_sinal,

            "doenca_preexistente_maior_frequencia":
                doenca_mais_frequente,

            "percentual_doenca_preexistente":
                percentual_doenca,

            "total_sinais_clinicos_avaliados":
                len(sintomas),

            "total_doencas_preexistentes_avaliadas":
                len(doencas)
        },

        evidencias={
            "sinais_clinicos":
                evidencias_sintomas,

            "doencas_preexistentes":
                evidencias_doencas
        },

        interpretacao=interpretacao,

        escopo={
            "pais":
                "Brasil",

            "ano":
                ano,

            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final
                - semana_inicial
                + 1
        },

        observacoes_dados=[
            (
                f"Os resultados correspondem aos registros "
                f"disponíveis entre as semanas "
                f"epidemiológicas {semana_inicial} e "
                f"{semana_final} de {ano}."
            ),
            (
                "Os percentuais de resposta positiva são "
                "calculados sobre os registros avaliáveis "
                "de cada variável."
            ),
            (
                "A ausência de preenchimento é apresentada "
                "separadamente para cada sinal clínico e "
                "doença preexistente."
            ),
            (
                "As frequências apresentadas possuem "
                "caráter descritivo e não estabelecem "
                "relações causais entre condições "
                "preexistentes e dengue."
            )
        ],

        conceitos_semanticos=[
            "dengue",
            "sinal clínico",
            "sintoma",
            "doença preexistente",
            "perfil clínico",
            "semana epidemiológica",
            "Brasil",
            "vigilância epidemiológica"
        ],

        palavras_chave=[
            "dengue",
            "sinais clínicos",
            "sintomas",
            "doenças preexistentes",
            "perfil clínico",
            "Brasil",
            str(ano),
            "SINAN"
        ],

        ano=ano
    )

    return documento

In [123]:
# ============================================================
# GERAR PANORAMA CLÍNICO NACIONAL
# ============================================================

documento_clinico_brasil = (
    criar_documento_panorama_clinico_brasil(
        df_sintomas=df_sintomas_brasil,
        df_doencas=(
            df_doencas_preexistentes_brasil
        ),
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO
    )
)

print(
    documento_para_markdown(
        documento_clinico_brasil
    )
)

# Panorama clínico dos registros de dengue no Brasil - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_CLINICO_BRASIL
- **Tipo de documento:** panorama_clinico_nacional
- **Domínio:** clinico
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Pais:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34

## Síntese epidemiológica

Este documento apresenta o panorama nacional do perfil clínico dos registros de dengue no SINAN entre as semanas epidemiológicas 1 e 34 de 2026. São apresentadas informações agregadas sobre sinais clínicos e doenças preexistentes.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros:** 444266
- **Sinal clinico maior frequencia:** Febre
- **Percentual sinal clinico:** 88.21
- **Doenca preexistente maior frequencia:** Hipertensão arterial
- **Percentual doenca preexistente:** 7.74
- **Total sinais clinicos avaliados:** 14
- **Tota

In [124]:
# ============================================================
# SALVAR PANORAMA CLÍNICO NACIONAL
# ============================================================

arquivo_panorama_clinico = (
    salvar_documento_semantico(
        documento=documento_clinico_brasil,
        pasta_saida=PASTA_DOCS_PANORAMA_CLINICO
    )
)

print("=" * 70)
print("PANORAMA CLÍNICO NACIONAL")
print("=" * 70)

print(
    "JSON:",
    arquivo_panorama_clinico["json"]
)

print(
    "Markdown:",
    arquivo_panorama_clinico["markdown"]
)

PANORAMA CLÍNICO NACIONAL
JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/clinicas/panorama_clinico/sinan_dengue_2026_clinico_brasil.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/clinicas/panorama_clinico/sinan_dengue_2026_clinico_brasil.md


In [125]:
#Depois eu faria uma auditoria final do domínio clínico:
# ============================================================
# AUDITORIA FINAL DO DOMÍNIO CLÍNICO
# ============================================================

json_uf = list(
    PASTA_DOCS_CLINICOS_UF.glob("*.json")
)

md_uf = list(
    PASTA_DOCS_CLINICOS_UF.glob("*.md")
)

json_panorama = list(
    PASTA_DOCS_PANORAMA_CLINICO.glob("*.json")
)

md_panorama = list(
    PASTA_DOCS_PANORAMA_CLINICO.glob("*.md")
)

total_documentos = (
    len(json_uf)
    + len(json_panorama)
)

total_arquivos = (
    len(json_uf)
    + len(md_uf)
    + len(json_panorama)
    + len(md_panorama)
)

print("=" * 70)
print("RESUMO DO DOMÍNIO CLÍNICO")
print("=" * 70)

print(
    f"Documentos por UF: "
    f"{len(json_uf)}"
)

print(
    f"Panoramas nacionais: "
    f"{len(json_panorama)}"
)

print(
    f"Total de documentos semânticos: "
    f"{total_documentos}"
)

print(
    f"Arquivos JSON: "
    f"{len(json_uf) + len(json_panorama)}"
)

print(
    f"Arquivos Markdown: "
    f"{len(md_uf) + len(md_panorama)}"
)

print(
    f"Total de arquivos físicos: "
    f"{total_arquivos}"
)

if (
    len(json_uf) == 27
    and len(md_uf) == 27
    and len(json_panorama) == 1
    and len(md_panorama) == 1
):
    print(
        "\n✓ Domínio clínico concluído corretamente."
    )
else:
    print(
        "\n⚠ Verifique a quantidade de arquivos."
    )

RESUMO DO DOMÍNIO CLÍNICO
Documentos por UF: 27
Panoramas nacionais: 1
Total de documentos semânticos: 28
Arquivos JSON: 28
Arquivos Markdown: 28
Total de arquivos físicos: 56

✓ Domínio clínico concluído corretamente.


## 9. Domínio desfechos


In [126]:
# ============================================================
# INSPECIONAR PRODUTOS ANALÍTICOS DO DOMÍNIO DESFECHOS
# ============================================================

print("=" * 70)
print("PRODUTOS ANALÍTICOS - DESFECHOS")
print("=" * 70)

for nome_variavel, info in dataframes_analytics.items():

    if info["dominio"] == "desfechos":

        df = info["dataframe"]

        print("\n" + "=" * 70)
        print(nome_variavel)
        print("=" * 70)

        # Mostra produto de origem somente se existir
        produto_origem = info.get(
            "arquivo_origem",
            info.get("produto", "não informado")
        )

        print(
            f"Produto de origem: "
            f"{produto_origem}"
        )

        print(
            f"Dimensão: "
            f"{df.shape[0]:,} linhas x "
            f"{df.shape[1]} colunas"
        )

        print(
            "Colunas:",
            df.columns.tolist()
        )

        display(
            df.head(10)
        )

PRODUTOS ANALÍTICOS - DESFECHOS

df_doencas_preexistentes_obitos_brasil
Produto de origem: doencas_preexistentes_obitos_brasil
Dimensão: 7 linhas x 9 colunas
Colunas: ['VARIAVEL', 'DOENCA_PREEXISTENTE', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,VARIAVEL,DOENCA_PREEXISTENTE,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,DIABETES_DECODED,Diabetes,559,559,112,447,0,20.04,0.0
1,HEMATOLOG_DECODED,Doença hematológica,559,559,17,542,0,3.04,0.0
2,HEPATOPAT_DECODED,Hepatopatia,559,559,24,535,0,4.29,0.0
3,RENAL_DECODED,Doença renal,559,559,35,524,0,6.26,0.0
4,HIPERTENSA_DECODED,Hipertensão arterial,559,559,207,352,0,37.03,0.0
5,ACIDO_PEPT_DECODED,Doença ácido-péptica,559,559,4,555,0,0.72,0.0
6,AUTO_IMUNE_DECODED,Doença autoimune,559,559,20,539,0,3.58,0.0



df_doencas_preexistentes_obitos_uf
Produto de origem: doencas_preexistentes_obitos_uf
Dimensão: 182 linhas x 11 colunas
Colunas: ['SG_UF_NOT', 'UF_NAME', 'VARIAVEL', 'DOENCA_PREEXISTENTE', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,SG_UF_NOT,UF_NAME,VARIAVEL,DOENCA_PREEXISTENTE,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,11,Rondônia,DIABETES_DECODED,Diabetes,1,1,0,1,0,0.0,0.0
1,11,Rondônia,HEMATOLOG_DECODED,Doença hematológica,1,1,0,1,0,0.0,0.0
2,11,Rondônia,HEPATOPAT_DECODED,Hepatopatia,1,1,0,1,0,0.0,0.0
3,11,Rondônia,RENAL_DECODED,Doença renal,1,1,0,1,0,0.0,0.0
4,11,Rondônia,HIPERTENSA_DECODED,Hipertensão arterial,1,1,0,1,0,0.0,0.0
5,11,Rondônia,ACIDO_PEPT_DECODED,Doença ácido-péptica,1,1,0,1,0,0.0,0.0
6,11,Rondônia,AUTO_IMUNE_DECODED,Doença autoimune,1,1,0,1,0,0.0,0.0
7,12,Acre,DIABETES_DECODED,Diabetes,2,2,1,1,0,50.0,0.0
8,12,Acre,HEMATOLOG_DECODED,Doença hematológica,2,2,0,2,0,0.0,0.0
9,12,Acre,HEPATOPAT_DECODED,Hepatopatia,2,2,1,1,0,50.0,0.0



df_evolucao_brasil
Produto de origem: evolucao_brasil
Dimensão: 6 linhas x 4 colunas
Colunas: ['EVOLUCAO', 'TOTAL_REGISTROS', 'PERCENTUAL_TOTAL', 'PERCENTUAL_AVALIAVEIS']


...

,SG_UF_NOT,UF_NAME,TOTAL_REGISTROS_OBITO,TOTAL_REGISTROS_UF,PERCENTUAL_OBITOS_TOTAL,TOTAL_EVOLUCAO_AVALIAVEL,PERCENTUAL_OBITOS_AVALIAVEIS,PERCENTUAL_REGISTROS_OBITO_TOTAL,PERCENTUAL_REGISTROS_OBITO_AVALIAVEIS
0,11,Rondônia,1,1354,0.0739,804,0.1244,0.0739,0.1244
1,12,Acre,2,1832,0.1092,1146,0.1745,0.1092,0.1745
2,13,Amazonas,1,1399,0.0715,1000,0.1000,0.0715,0.1000
3,14,Roraima,2,556,0.3597,322,0.6211,0.3597,0.6211
4,15,Pará,21,9323,0.2252,6774,0.3100,0.2252,0.3100
5,16,Amapá,1,513,0.1949,286,0.3497,0.1949,0.3497
6,17,Tocantins,19,15430,0.1231,10099,0.1881,0.1231,0.1881
7,21,Maranhão,22,12282,0.1791,7023,0.3133,0.1791,0.3133
8,22,Piauí,20,17383,0.1151,8574,0.2333,0.1151,0.2333
9,23,Ceará,26,17337,0.1500,13045,0.1993,0.1500,0.1993



df_sintomas_obitos_brasil
Produto de origem: sintomas_obitos_brasil
Dimensão: 14 linhas x 9 colunas
Colunas: ['VARIAVEL', 'SINAL_CLINICO', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,VARIAVEL,SINAL_CLINICO,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,FEBRE_DECODED,Febre,559,559,444,115,0,79.43,0.0
1,MIALGIA_DECODED,Mialgia,559,559,380,179,0,67.98,0.0
2,CEFALEIA_DECODED,Cefaleia,559,559,305,254,0,54.56,0.0
3,EXANTEMA_DECODED,Exantema,559,559,58,501,0,10.38,0.0
4,VOMITO_DECODED,Vômito,559,559,273,286,0,48.84,0.0
5,NAUSEA_DECODED,Náusea,559,559,266,293,0,47.58,0.0
6,DOR_COSTAS_DECODED,Dor nas costas,559,559,136,423,0,24.33,0.0
7,CONJUNTVIT_DECODED,Conjuntivite,559,559,16,543,0,2.86,0.0
8,ARTRITE_DECODED,Artrite,559,559,47,512,0,8.41,0.0
9,ARTRALGIA_DECODED,Artralgia,559,559,96,463,0,17.17,0.0



df_sintomas_obitos_uf
Produto de origem: sintomas_obitos_uf
Dimensão: 364 linhas x 11 colunas
Colunas: ['SG_UF_NOT', 'UF_NAME', 'VARIAVEL', 'SINAL_CLINICO', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,SG_UF_NOT,UF_NAME,VARIAVEL,SINAL_CLINICO,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,11,Rondônia,FEBRE_DECODED,Febre,1,1,1,0,0,100.0,0.0
1,11,Rondônia,MIALGIA_DECODED,Mialgia,1,1,0,1,0,0.0,0.0
2,11,Rondônia,CEFALEIA_DECODED,Cefaleia,1,1,1,0,0,100.0,0.0
3,11,Rondônia,EXANTEMA_DECODED,Exantema,1,1,0,1,0,0.0,0.0
4,11,Rondônia,VOMITO_DECODED,Vômito,1,1,0,1,0,0.0,0.0
5,11,Rondônia,NAUSEA_DECODED,Náusea,1,1,1,0,0,100.0,0.0
6,11,Rondônia,DOR_COSTAS_DECODED,Dor nas costas,1,1,0,1,0,0.0,0.0
7,11,Rondônia,CONJUNTVIT_DECODED,Conjuntivite,1,1,0,1,0,0.0,0.0
8,11,Rondônia,ARTRITE_DECODED,Artrite,1,1,0,1,0,0.0,0.0
9,11,Rondônia,ARTRALGIA_DECODED,Artralgia,1,1,0,1,0,0.0,0.0


In [127]:
#Antes disso, podemos conferir quais chaves realmente existem no primeiro item:
# ============================================================
# VERIFICAR ESTRUTURA DO DICIONÁRIO
# ============================================================

for nome_variavel, info in dataframes_analytics.items():

    print(
        nome_variavel,
        "->",
        info.keys()
    )

    break

df_doencas_preexistentes_brasil -> dict_keys(['ano', 'dominio', 'produto', 'arquivo_origem', 'dataframe'])


In [128]:
# ============================================================
# CRIAR ALIASES LÓGICOS INDEPENDENTES DO ANO
# ============================================================

import re

dataframes_analytics = {}

for dominio, resultados in resultados_analytics.items():

    for nome_arquivo, df in resultados.items():

        # Ignorar inventários
        if nome_arquivo.startswith(
            "inventario_produtos"
        ):
            continue

        # Remove somente o ano no final do nome
        nome_logico = re.sub(
            rf"_{ANO}$",
            "",
            nome_arquivo
        )

        nome_variavel = (
            f"df_{nome_logico}"
        )

        # Cria variável dinâmica
        globals()[nome_variavel] = df

        # Registra metadados
        dataframes_analytics[
            nome_variavel
        ] = {
            "ano":
                ANO,

            "dominio":
                dominio,

            "produto":
                nome_logico,

            "arquivo_origem":
                nome_arquivo,

            "dataframe":
                df
        }

In [129]:
# ============================================================
# TESTAR ESTRUTURA
# ============================================================

for nome_variavel, info in dataframes_analytics.items():

    print(nome_variavel)

    print(
        info.keys()
    )

    break

df_doencas_preexistentes_brasil
dict_keys(['ano', 'dominio', 'produto', 'arquivo_origem', 'dataframe'])


In [130]:
# ============================================================
# INSPECIONAR PRODUTOS ANALÍTICOS DO DOMÍNIO DESFECHOS
# ============================================================

print("=" * 70)
print("PRODUTOS ANALÍTICOS - DESFECHOS")
print("=" * 70)

for nome_variavel, info in dataframes_analytics.items():

    if info["dominio"] == "desfechos":

        df = info["dataframe"]

        print("\n" + "=" * 70)
        print(nome_variavel)
        print("=" * 70)

        print(
            f"Produto de origem: "
            f"{info['arquivo_origem']}"
        )

        print(
            f"Dimensão: "
            f"{df.shape[0]:,} linhas x "
            f"{df.shape[1]} colunas"
        )

        print(
            "Colunas:",
            df.columns.tolist()
        )

        display(
            df.head(10)
        )

PRODUTOS ANALÍTICOS - DESFECHOS

df_doencas_preexistentes_obitos_brasil
Produto de origem: doencas_preexistentes_obitos_brasil
Dimensão: 7 linhas x 9 colunas
Colunas: ['VARIAVEL', 'DOENCA_PREEXISTENTE', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,VARIAVEL,DOENCA_PREEXISTENTE,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,DIABETES_DECODED,Diabetes,559,559,112,447,0,20.04,0.0
1,HEMATOLOG_DECODED,Doença hematológica,559,559,17,542,0,3.04,0.0
2,HEPATOPAT_DECODED,Hepatopatia,559,559,24,535,0,4.29,0.0
3,RENAL_DECODED,Doença renal,559,559,35,524,0,6.26,0.0
4,HIPERTENSA_DECODED,Hipertensão arterial,559,559,207,352,0,37.03,0.0
5,ACIDO_PEPT_DECODED,Doença ácido-péptica,559,559,4,555,0,0.72,0.0
6,AUTO_IMUNE_DECODED,Doença autoimune,559,559,20,539,0,3.58,0.0



df_doencas_preexistentes_obitos_uf
Produto de origem: doencas_preexistentes_obitos_uf
Dimensão: 182 linhas x 11 colunas
Colunas: ['SG_UF_NOT', 'UF_NAME', 'VARIAVEL', 'DOENCA_PREEXISTENTE', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,SG_UF_NOT,UF_NAME,VARIAVEL,DOENCA_PREEXISTENTE,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,11,Rondônia,DIABETES_DECODED,Diabetes,1,1,0,1,0,0.0,0.0
1,11,Rondônia,HEMATOLOG_DECODED,Doença hematológica,1,1,0,1,0,0.0,0.0
2,11,Rondônia,HEPATOPAT_DECODED,Hepatopatia,1,1,0,1,0,0.0,0.0
3,11,Rondônia,RENAL_DECODED,Doença renal,1,1,0,1,0,0.0,0.0
4,11,Rondônia,HIPERTENSA_DECODED,Hipertensão arterial,1,1,0,1,0,0.0,0.0
5,11,Rondônia,ACIDO_PEPT_DECODED,Doença ácido-péptica,1,1,0,1,0,0.0,0.0
6,11,Rondônia,AUTO_IMUNE_DECODED,Doença autoimune,1,1,0,1,0,0.0,0.0
7,12,Acre,DIABETES_DECODED,Diabetes,2,2,1,1,0,50.0,0.0
8,12,Acre,HEMATOLOG_DECODED,Doença hematológica,2,2,0,2,0,0.0,0.0
9,12,Acre,HEPATOPAT_DECODED,Hepatopatia,2,2,1,1,0,50.0,0.0



df_evolucao_brasil
Produto de origem: evolucao_brasil
Dimensão: 6 linhas x 4 colunas
Colunas: ['EVOLUCAO', 'TOTAL_REGISTROS', 'PERCENTUAL_TOTAL', 'PERCENTUAL_AVALIAVEIS']


...

,SG_UF_NOT,UF_NAME,TOTAL_REGISTROS_OBITO,TOTAL_REGISTROS_UF,PERCENTUAL_OBITOS_TOTAL,TOTAL_EVOLUCAO_AVALIAVEL,PERCENTUAL_OBITOS_AVALIAVEIS,PERCENTUAL_REGISTROS_OBITO_TOTAL,PERCENTUAL_REGISTROS_OBITO_AVALIAVEIS
0,11,Rondônia,1,1354,0.0739,804,0.1244,0.0739,0.1244
1,12,Acre,2,1832,0.1092,1146,0.1745,0.1092,0.1745
2,13,Amazonas,1,1399,0.0715,1000,0.1000,0.0715,0.1000
3,14,Roraima,2,556,0.3597,322,0.6211,0.3597,0.6211
4,15,Pará,21,9323,0.2252,6774,0.3100,0.2252,0.3100
5,16,Amapá,1,513,0.1949,286,0.3497,0.1949,0.3497
6,17,Tocantins,19,15430,0.1231,10099,0.1881,0.1231,0.1881
7,21,Maranhão,22,12282,0.1791,7023,0.3133,0.1791,0.3133
8,22,Piauí,20,17383,0.1151,8574,0.2333,0.1151,0.2333
9,23,Ceará,26,17337,0.1500,13045,0.1993,0.1500,0.1993



df_sintomas_obitos_brasil
Produto de origem: sintomas_obitos_brasil
Dimensão: 14 linhas x 9 colunas
Colunas: ['VARIAVEL', 'SINAL_CLINICO', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,VARIAVEL,SINAL_CLINICO,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,FEBRE_DECODED,Febre,559,559,444,115,0,79.43,0.0
1,MIALGIA_DECODED,Mialgia,559,559,380,179,0,67.98,0.0
2,CEFALEIA_DECODED,Cefaleia,559,559,305,254,0,54.56,0.0
3,EXANTEMA_DECODED,Exantema,559,559,58,501,0,10.38,0.0
4,VOMITO_DECODED,Vômito,559,559,273,286,0,48.84,0.0
5,NAUSEA_DECODED,Náusea,559,559,266,293,0,47.58,0.0
6,DOR_COSTAS_DECODED,Dor nas costas,559,559,136,423,0,24.33,0.0
7,CONJUNTVIT_DECODED,Conjuntivite,559,559,16,543,0,2.86,0.0
8,ARTRITE_DECODED,Artrite,559,559,47,512,0,8.41,0.0
9,ARTRALGIA_DECODED,Artralgia,559,559,96,463,0,17.17,0.0



df_sintomas_obitos_uf
Produto de origem: sintomas_obitos_uf
Dimensão: 364 linhas x 11 colunas
Colunas: ['SG_UF_NOT', 'UF_NAME', 'VARIAVEL', 'SINAL_CLINICO', 'TOTAL_REGISTROS_OBITO', 'AVALIAVEIS', 'SIM', 'NAO', 'AUSENTES', 'PERCENTUAL_SIM', 'PERCENTUAL_AUSENTES']


,SG_UF_NOT,UF_NAME,VARIAVEL,SINAL_CLINICO,TOTAL_REGISTROS_OBITO,AVALIAVEIS,SIM,NAO,AUSENTES,PERCENTUAL_SIM,PERCENTUAL_AUSENTES
0,11,Rondônia,FEBRE_DECODED,Febre,1,1,1,0,0,100.0,0.0
1,11,Rondônia,MIALGIA_DECODED,Mialgia,1,1,0,1,0,0.0,0.0
2,11,Rondônia,CEFALEIA_DECODED,Cefaleia,1,1,1,0,0,100.0,0.0
3,11,Rondônia,EXANTEMA_DECODED,Exantema,1,1,0,1,0,0.0,0.0
4,11,Rondônia,VOMITO_DECODED,Vômito,1,1,0,1,0,0.0,0.0
5,11,Rondônia,NAUSEA_DECODED,Náusea,1,1,1,0,0,100.0,0.0
6,11,Rondônia,DOR_COSTAS_DECODED,Dor nas costas,1,1,0,1,0,0.0,0.0
7,11,Rondônia,CONJUNTVIT_DECODED,Conjuntivite,1,1,0,1,0,0.0,0.0
8,11,Rondônia,ARTRITE_DECODED,Artrite,1,1,0,1,0,0.0,0.0
9,11,Rondônia,ARTRALGIA_DECODED,Artralgia,1,1,0,1,0,0.0,0.0


In [131]:
# ============================================================
# LISTAR NOMES LÓGICOS DOS PRODUTOS DE DESFECHOS
# ============================================================

produtos_desfechos = [
    nome
    for nome, info
    in dataframes_analytics.items()
    if info["dominio"] == "desfechos"
]

print(
    "Quantidade de produtos:",
    len(produtos_desfechos)
)

for produto in produtos_desfechos:
    print("-", produto)

Quantidade de produtos: 13
- df_doencas_preexistentes_obitos_brasil
- df_doencas_preexistentes_obitos_uf
- df_evolucao_brasil
- df_evolucao_uf
- df_hospitalizacao_brasil
- df_hospitalizacao_uf
- df_obitos_brasil
- df_obitos_sorotipo_brasil
- df_obitos_sorotipo_uf
- df_obitos_tipo_uf
- df_obitos_uf
- df_sintomas_obitos_brasil
- df_sintomas_obitos_uf


O domínio desfechos é consideravelmente mais rico que os anteriores: não temos apenas hospitalização e evolução; temos também uma caracterização específica dos registros classificados como óbito.

Os produtos mostram hospitalização nacional e por UF, evolução nacional e por UF, óbitos por tipo, óbitos por UF, sorotipo entre óbitos e perfil clínico/preexistências dos óbitos.

Eu faria uma pequena mudança em relação ao plano inicial: não colocaria tudo em um único documento por UF.

Estrutura proposta
```
desfechos/
│
├── perfil_desfechos_por_uf/
│   └── 27 documentos
│       ├── hospitalização
│       └── evolução
│
├── perfil_obitos_por_uf/
│   └── 27 documentos
│       ├── quantidade/tipo de óbito
│       ├── sorotipos entre óbitos
│       ├── sinais clínicos entre óbitos
│       └── doenças preexistentes entre óbitos
│
├── panorama_desfechos/
│   └── 1 documento Brasil
│
└── panorama_obitos/
    └── 1 documento Brasil
```

Isso daria 56 documentos semânticos no domínio de desfechos.

A separação é importante porque uma pergunta como "Qual percentual de registros foi hospitalizado em Rondônia?" tem uma intenção informacional bastante diferente de "Quais sintomas foram mais frequentes entre os óbitos no Brasil?". Misturar tudo em um documento muito grande pode prejudicar a granularidade da recuperação do RAG.

Além disso, há um cuidado essencial: o conjunto de óbitos considerado nesses produtos possui 546 registros, enquanto a evolução nacional mostra 276 óbitos pelo agravo, 175 em investigação e 95 por outras causas — exatamente 546 quando somados.

## Primeiro: documentos gerais de desfecho por UF

Vamos começar somente com hospitalização + evolução. Depois fazemos o bloco específico de óbitos.

1. Função de interpretação

In [132]:
# ============================================================
# INTERPRETAÇÃO DOS DESFECHOS POR UF
# ============================================================

def interpretar_desfechos_uf(
    uf_nome,
    total_registros,
    hospitalizados,
    percentual_hospitalizados,
    evolucao_predominante,
    total_evolucao_predominante,
    percentual_evolucao_predominante,
    semana_inicial,
    semana_final,
    ano
):

    return (
        f"Na UF {uf_nome}, foram considerados "
        f"{total_registros:,} registros de dengue entre "
        f"as semanas epidemiológicas {semana_inicial} e "
        f"{semana_final} de {ano}. "
        f"Entre os registros avaliáveis para hospitalização, "
        f"{hospitalizados:,} apresentaram hospitalização, "
        f"correspondendo a {percentual_hospitalizados:.2f}%. "
        f"Entre as categorias avaliáveis de evolução, "
        f"{evolucao_predominante} foi a mais frequente, "
        f"com {total_evolucao_predominante:,} registros "
        f"e {percentual_evolucao_predominante:.2f}% dos "
        f"registros com evolução avaliável."
    )

Essa distinção de denominadores é importante. No produto de hospitalização, por exemplo, Rondônia tem 1.270 registros, mas somente 650 avaliáveis para hospitalização; os 11,85% hospitalizados são relativos aos avaliáveis.

## 2. Criar os documentos por UF

In [133]:
# ============================================================
# CRIAR DOCUMENTOS DE DESFECHOS POR UF
# Hospitalização + Evolução
# ============================================================

def criar_documentos_desfechos_por_uf(
    df_hospitalizacao,
    df_evolucao,
    semana_inicial,
    semana_final,
    ano=ANO
):

    documentos = []

    # Hospitalização possui exatamente uma linha por UF
    hospitalizacao = (
        df_hospitalizacao
        .sort_values("SG_UF_NOT")
        .copy()
    )

    for _, linha_hosp in hospitalizacao.iterrows():

        codigo_uf = str(
            linha_hosp["SG_UF_NOT"]
        )

        nome_uf = linha_hosp[
            "UF_NAME"
        ]

        # ====================================================
        # HOSPITALIZAÇÃO
        # ====================================================

        total_registros = int(
            linha_hosp["TOTAL_REGISTROS"]
        )

        avaliaveis_hospitalizacao = int(
            linha_hosp["AVALIAVEIS"]
        )

        hospitalizados = int(
            linha_hosp["HOSPITALIZADOS"]
        )

        nao_hospitalizados = int(
            linha_hosp["NAO_HOSPITALIZADOS"]
        )

        ignorados_hospitalizacao = int(
            linha_hosp["IGNORADOS"]
        )

        ausentes_hospitalizacao = int(
            linha_hosp["AUSENTES"]
        )

        percentual_hospitalizados = float(
            linha_hosp[
                "PERCENTUAL_HOSPITALIZADOS"
            ]
        )

        # ====================================================
        # EVOLUÇÃO
        # ====================================================

        evolucao_uf = (
            df_evolucao[
                df_evolucao["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        # Categorias consideradas avaliáveis
        evolucao_avaliavel = (
            evolucao_uf[
                evolucao_uf["AVALIAVEL"] == True
            ]
            .copy()
        )

        # Maior frequência entre categorias avaliáveis
        evolucao_predominante_linha = (
            evolucao_avaliavel
            .sort_values(
                "TOTAL_REGISTROS",
                ascending=False
            )
            .iloc[0]
        )

        evolucao_predominante = (
            evolucao_predominante_linha[
                "EVOLUCAO_DECODED"
            ]
        )

        total_evolucao_predominante = int(
            evolucao_predominante_linha[
                "TOTAL_REGISTROS"
            ]
        )

        percentual_evolucao_predominante = float(
            evolucao_predominante_linha[
                "PERCENTUAL_AVALIAVEIS"
            ]
        )

        total_evolucao_avaliavel = int(
            evolucao_predominante_linha[
                "TOTAL_AVALIAVEIS"
            ]
        )

        # ====================================================
        # EVIDÊNCIA DE HOSPITALIZAÇÃO
        # ====================================================

        evidencia_hospitalizacao = {
            "total_registros":
                total_registros,

            "avaliaveis":
                avaliaveis_hospitalizacao,

            "hospitalizados":
                hospitalizados,

            "nao_hospitalizados":
                nao_hospitalizados,

            "ignorados":
                ignorados_hospitalizacao,

            "ausentes":
                ausentes_hospitalizacao,

            "percentual_hospitalizados":
                percentual_hospitalizados,

            "percentual_nao_hospitalizados":
                float(
                    linha_hosp[
                        "PERCENTUAL_NAO_HOSPITALIZADOS"
                    ]
                ),

            "percentual_ignorados":
                float(
                    linha_hosp[
                        "PERCENTUAL_IGNORADOS"
                    ]
                ),

            "percentual_ausentes":
                float(
                    linha_hosp[
                        "PERCENTUAL_AUSENTES"
                    ]
                )
        }

        # ====================================================
        # EVIDÊNCIAS DE EVOLUÇÃO
        # ====================================================

        evidencias_evolucao = []

        for _, linha in evolucao_uf.iterrows():

            evolucao_nome = (
                linha["EVOLUCAO_DECODED"]
            )

            if pd.isna(evolucao_nome):
                evolucao_nome = "Ausente"

            evidencias_evolucao.append(
                {
                    "evolucao":
                        evolucao_nome,

                    "total_registros":
                        int(
                            linha["TOTAL_REGISTROS"]
                        ),

                    "percentual_total":
                        float(
                            linha["PERCENTUAL_TOTAL"]
                        ),

                    "avaliavel":
                        bool(
                            linha["AVALIAVEL"]
                        ),

                    "total_avaliaveis":
                        int(
                            linha["TOTAL_AVALIAVEIS"]
                        ),

                    "percentual_avaliaveis":
                        (
                            None
                            if pd.isna(
                                linha[
                                    "PERCENTUAL_AVALIAVEIS"
                                ]
                            )
                            else float(
                                linha[
                                    "PERCENTUAL_AVALIAVEIS"
                                ]
                            )
                        )
                }
            )

        # ====================================================
        # INTERPRETAÇÃO
        # ====================================================

        interpretacao = (
            interpretar_desfechos_uf(
                uf_nome=nome_uf,
                total_registros=total_registros,
                hospitalizados=hospitalizados,
                percentual_hospitalizados=(
                    percentual_hospitalizados
                ),
                evolucao_predominante=(
                    evolucao_predominante
                ),
                total_evolucao_predominante=(
                    total_evolucao_predominante
                ),
                percentual_evolucao_predominante=(
                    percentual_evolucao_predominante
                ),
                semana_inicial=semana_inicial,
                semana_final=semana_final,
                ano=ano
            )
        )

        # ====================================================
        # DOCUMENTO
        # ====================================================

        documento = criar_documento_semantico(

            document_id=gerar_document_id(
                fonte="SINAN",
                doenca="dengue",
                ano=ano,
                dominio="desfechos",
                localizacao=codigo_uf
            ),

            titulo=(
                f"Desfechos dos registros de dengue "
                f"em {nome_uf} - {ano}"
            ),

            tipo_documento="desfechos_uf",

            dominio="desfechos",

            sintese=(
                f"Este documento apresenta informações "
                f"sobre hospitalização e evolução dos "
                f"registros de dengue no SINAN em "
                f"{nome_uf}, entre as semanas "
                f"epidemiológicas {semana_inicial} e "
                f"{semana_final} de {ano}."
            ),

            indicadores={
                "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1,

                "total_registros":
                    total_registros,

                "avaliaveis_hospitalizacao":
                    avaliaveis_hospitalizacao,

                "hospitalizados":
                    hospitalizados,

                "percentual_hospitalizados":
                    percentual_hospitalizados,

                "total_evolucao_avaliavel":
                    total_evolucao_avaliavel,

                "evolucao_predominante":
                    evolucao_predominante,

                "total_evolucao_predominante":
                    total_evolucao_predominante,

                "percentual_evolucao_predominante":
                    percentual_evolucao_predominante
            },


            evidencias={
                "hospitalizacao":
                    [evidencia_hospitalizacao],

                "evolucao":
                    evidencias_evolucao
            },

            interpretacao=interpretacao,

            escopo={
                "codigo_uf":
                    codigo_uf,

                "uf_nome":
                    nome_uf,

                "ano":
                    ano,

                "semana_inicial":
                    semana_inicial,

                "semana_final":
                    semana_final,

                "total_semanas":
                    semana_final
                    - semana_inicial
                    + 1
            },

            observacoes_dados=[
                (
                    f"Os resultados correspondem aos "
                    f"registros disponíveis entre as "
                    f"semanas epidemiológicas "
                    f"{semana_inicial} e {semana_final} "
                    f"de {ano}."
                ),
                (
                    "Os percentuais de hospitalização são "
                    "calculados sobre os registros "
                    "avaliáveis para hospitalização."
                ),
                (
                    "As categorias ausente e ignorado são "
                    "mantidas separadamente das categorias "
                    "avaliáveis de evolução."
                ),
                (
                    "Os resultados possuem caráter "
                    "descritivo e não estabelecem relações "
                    "causais."
                )
            ],

            conceitos_semanticos=[
                "dengue",
                "hospitalização",
                "evolução clínica",
                "cura",
                "óbito",
                "desfecho",
                "semana epidemiológica",
                "unidade federativa",
                "vigilância epidemiológica"
            ],

            palavras_chave=[
                "dengue",
                "hospitalização",
                "evolução",
                "cura",
                "óbito",
                "desfechos",
                nome_uf,
                codigo_uf,
                str(ano),
                "SINAN"
            ],

            ano=ano
        )

        documentos.append(
            documento
        )

    return documentos

## 3. Gerar sem salvar

In [134]:
# ============================================================
# GERAR DOCUMENTOS DE DESFECHOS POR UF
# ============================================================

documentos_desfechos_uf = (
    criar_documentos_desfechos_por_uf(
        df_hospitalizacao=(
            df_hospitalizacao_uf
        ),
        df_evolucao=(
            df_evolucao_uf
        ),
        semana_inicial=SEMANA_INICIAL,
        semana_final=SEMANA_FINAL,
        ano=ANO
    )
)

print(
    "Total de documentos gerados:",
    len(documentos_desfechos_uf)
)

Total de documentos gerados: 27


In [135]:
print(
    documento_para_markdown(
        documentos_desfechos_uf[0]
    )
)

# Desfechos dos registros de dengue em Rondônia - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_DESFECHOS_11
- **Tipo de documento:** desfechos_uf
- **Domínio:** desfechos
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 11
- **Uf nome:** Rondônia
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34

## Síntese epidemiológica

Este documento apresenta informações sobre hospitalização e evolução dos registros de dengue no SINAN em Rondônia, entre as semanas epidemiológicas 1 e 34 de 2026.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros:** 1354
- **Avaliaveis hospitalizacao:** 687
- **Hospitalizados:** 87
- **Percentual hospitalizados:** 12.66
- **Total evolucao avaliavel:** 804
- **Evolucao predominante:** Cura
- **Total evolucao predominante:** 803
- **Percentual evolucao predominante:** 99.88

## Evidências

### Hospitalizacao

- **Total 

In [136]:
# ============================================================
# DIRETÓRIOS - DOMÍNIO DESFECHOS
# ============================================================

PASTA_DOCS_DESFECHOS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
    / "desfechos"
)

PASTA_DOCS_DESFECHOS_UF = (
    PASTA_DOCS_DESFECHOS
    / "desfechos_por_uf"
)

PASTA_DOCS_DESFECHOS_UF.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Pasta:",
    PASTA_DOCS_DESFECHOS_UF
)

Pasta: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/desfechos_por_uf


In [137]:
# ============================================================
# SALVAR DOCUMENTOS DE DESFECHOS POR UF
# ============================================================

arquivos_desfechos_uf = []

for documento in documentos_desfechos_uf:

    caminhos = salvar_documento_semantico(
        documento=documento,
        pasta_saida=PASTA_DOCS_DESFECHOS_UF
    )

    arquivos_desfechos_uf.append(
        {
            "document_id":
                documento["document_id"],

            "json":
                caminhos["json"],

            "markdown":
                caminhos["markdown"]
        }
    )

print("=" * 70)
print("DOCUMENTOS DE DESFECHOS POR UF")
print("=" * 70)

print(
    "Documentos salvos:",
    len(arquivos_desfechos_uf)
)

print(
    "Arquivos físicos:",
    len(arquivos_desfechos_uf) * 2
)

DOCUMENTOS DE DESFECHOS POR UF
Documentos salvos: 27
Arquivos físicos: 54


Próximo bloco: perfil dos óbitos

Aqui eu manteria um documento separado porque estamos mudando a população analisada.

Nos documentos anteriores, a população é:

registros de dengue.

Agora será:

registros classificados em categorias de evolução relacionadas a óbito.

E o Notebook 05 mostra que são 546 registros de óbito, distribuídos em 276 óbitos pelo agravo, 175 óbitos em investigação e 95 óbitos por outras causas.

Para cada UF, podemos estruturar:
```
Perfil de óbitos
│
├── Indicadores
│   ├── total de registros da UF
│   ├── total de registros de óbito
│   ├── evolução avaliável
│   └── percentual de óbitos
│
├── Evidências
│   ├── tipos de óbito
│   │   ├── óbito pelo agravo
│   │   ├── óbito em investigação
│   │   └── óbito por outras causas
│   │
│   ├── sorotipos entre os óbitos
│   │
│   ├── sinais clínicos entre os óbitos
│   │
│   └── doenças preexistentes entre os óbitos
│
└── Observações sobre os dados
```

Temos produtos específicos para cada uma dessas dimensões: df_obitos_uf, df_obitos_tipo_uf, df_obitos_sorotipo_uf, df_sintomas_obitos_uf e df_doencas_preexistentes_obitos_uf.

Há ainda uma vantagem metodológica nessa separação: não confundiremos o perfil clínico geral com o perfil clínico dos registros de óbito. Por exemplo, no panorama clínico geral a febre apareceu em 88,03% dos avaliáveis; entre os 546 registros de óbito, aparece em 76,92%. São populações analíticas distintas.

Portanto, agora eu partiria para construir os 27 documentos perfil_obitos_uf, sem ainda salvar, e validaríamos primeiro Rondônia como fizemos nos outros domínios.

In [138]:
# ============================================================
# INTERPRETAÇÃO DO PERFIL DE ÓBITOS POR UF
# ============================================================

def interpretar_perfil_obitos_uf(
    uf_nome,
    total_registros_obito,
    total_registros_uf,
    percentual_obitos_total,
    tipo_obito_predominante,
    total_tipo_obito_predominante,
    semana_inicial,
    semana_final,
    ano
):

    if total_registros_obito == 1:
        descricao_quantidade = (
            "foi identificado 1 registro classificado"
        )
    else:
        descricao_quantidade = (
            f"foram identificados "
            f"{total_registros_obito:,} registros classificados"
        )

    termo_tipo = (
        "registro"
        if total_tipo_obito_predominante == 1
        else "registros"
    )

    return (
        f"Na UF {uf_nome}, {descricao_quantidade} "
        f"em categorias de evolução relacionadas a óbito, "
        f"entre as semanas epidemiológicas {semana_inicial} "
        f"e {semana_final} de {ano}. "
        f"Esse conjunto corresponde a "
        f"{percentual_obitos_total:.4f}% do total de "
        f"{total_registros_uf:,} registros de dengue da UF. "
        f"Entre os tipos de óbito registrados, "
        f"{tipo_obito_predominante} apresentou a maior "
        f"frequência, com "
        f"{total_tipo_obito_predominante:,} {termo_tipo}."
    )

In [139]:
# ============================================================
# CRIAR DOCUMENTOS DE PERFIL DE ÓBITOS POR UF
# ============================================================

def criar_documentos_perfil_obitos_por_uf(
    df_obitos,
    df_tipos_obito,
    df_sorotipos_obito,
    df_sintomas_obito,
    df_doencas_obito,
    semana_inicial,
    semana_final,
    ano=ANO
):

    documentos = []

    # Base: uma linha por UF
    base_ufs = (
        df_obitos
        .sort_values("SG_UF_NOT")
        .copy()
    )

    for _, linha_obito in base_ufs.iterrows():

        codigo_uf = str(
            linha_obito["SG_UF_NOT"]
        )

        nome_uf = linha_obito[
            "UF_NAME"
        ]

        # ====================================================
        # INDICADORES PRINCIPAIS
        # ====================================================

        total_registros_obito = int(
            linha_obito[
                "TOTAL_REGISTROS_OBITO"
            ]
        )
        LIMIAR_PEQUENO_NUMERO_OBITOS = 10
        pequeno_numero_obitos = (
            total_registros_obito
            < LIMIAR_PEQUENO_NUMERO_OBITOS
        )


        total_registros_uf = int(
            linha_obito[
                "TOTAL_REGISTROS_UF"
            ]
        )

        total_evolucao_avaliavel = int(
            linha_obito[
                "TOTAL_EVOLUCAO_AVALIAVEL"
            ]
        )

        percentual_obitos_total = float(
            linha_obito[
                "PERCENTUAL_OBITOS_TOTAL"
            ]
        )

        percentual_obitos_avaliaveis = float(
            linha_obito[
                "PERCENTUAL_OBITOS_AVALIAVEIS"
            ]
        )

        # ====================================================
        # TIPOS DE ÓBITO
        # ====================================================

        tipos_uf = (
            df_tipos_obito[
                df_tipos_obito["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        tipos_uf = (
            tipos_uf
            .sort_values(
                "TOTAL_OBITOS",
                ascending=False
            )
        )

        evidencias_tipos_obito = []

        for _, linha in tipos_uf.iterrows():

            evidencias_tipos_obito.append(
                {
                    "tipo_obito":
                        linha[
                            "EVOLUCAO_DECODED"
                        ],

                    "total_obitos":
                        int(
                            linha[
                                "TOTAL_OBITOS"
                            ]
                        )
                }
            )

        # Tipo predominante
        if len(tipos_uf) > 0:

            tipo_obito_predominante = (
                tipos_uf.iloc[0][
                    "EVOLUCAO_DECODED"
                ]
            )

            total_tipo_obito_predominante = int(
                tipos_uf.iloc[0][
                    "TOTAL_OBITOS"
                ]
            )

        else:

            tipo_obito_predominante = (
                "Sem informação"
            )

            total_tipo_obito_predominante = 0

        # ====================================================
        # SOROTIPOS ENTRE ÓBITOS
        # ====================================================

        sorotipos_uf = (
            df_sorotipos_obito[
                df_sorotipos_obito["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        evidencias_sorotipos = []

        for _, linha in sorotipos_uf.iterrows():

            sorotipo = (
                linha[
                    "SOROTIPO_DECODED"
                ]
            )

            if pd.isna(sorotipo):
                sorotipo = "Ausente"

            evidencias_sorotipos.append(
                {
                    "sorotipo":
                        sorotipo,

                    "total_registros_obito":
                        int(
                            linha[
                                "TOTAL_REGISTROS_OBITO"
                            ]
                        ),

                    "total_obitos_uf":
                        int(
                            linha[
                                "TOTAL_OBITOS_UF"
                            ]
                        ),

                    "percentual_total_obitos_uf":
                        float(
                            linha[
                                "PERCENTUAL_TOTAL_OBITOS_UF"
                            ]
                        ),

                    "total_sorotipo_informado_uf":
                        int(
                            linha[
                                "TOTAL_SOROTIPO_INFORMADO_UF"
                            ]
                        ),

                    "percentual_entre_sorotipos_informados":
                        (
                            None
                            if pd.isna(
                                linha[
                                    "PERCENTUAL_ENTRE_SOROTIPOS_INFORMADOS"
                                ]
                            )
                            else float(
                                linha[
                                    "PERCENTUAL_ENTRE_SOROTIPOS_INFORMADOS"
                                ]
                            )
                        )
                }
            )

        # ====================================================
        # SINAIS CLÍNICOS ENTRE ÓBITOS
        # ====================================================

        sintomas_uf = (
            df_sintomas_obito[
                df_sintomas_obito["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        sintomas_uf = (
            sintomas_uf
            .sort_values(
                "PERCENTUAL_SIM",
                ascending=False
            )
        )

        evidencias_sintomas = []

        for _, linha in sintomas_uf.iterrows():

            evidencias_sintomas.append(
                {
                    "variavel":
                        linha["VARIAVEL"],

                    "sinal_clinico":
                        linha["SINAL_CLINICO"],

                    "total_registros_obito":
                        int(
                            linha[
                                "TOTAL_REGISTROS_OBITO"
                            ]
                        ),

                    "avaliaveis":
                        int(
                            linha["AVALIAVEIS"]
                        ),

                    "sim":
                        int(
                            linha["SIM"]
                        ),

                    "nao":
                        int(
                            linha["NAO"]
                        ),

                    "ausentes":
                        int(
                            linha["AUSENTES"]
                        ),

                    "percentual_sim":
                        float(
                            linha[
                                "PERCENTUAL_SIM"
                            ]
                        ),

                    "percentual_ausentes":
                        float(
                            linha[
                                "PERCENTUAL_AUSENTES"
                            ]
                        )
                }
            )

        # ====================================================
        # DOENÇAS PREEXISTENTES ENTRE ÓBITOS
        # ====================================================

        doencas_uf = (
            df_doencas_obito[
                df_doencas_obito["SG_UF_NOT"]
                .astype(str)
                == codigo_uf
            ]
            .copy()
        )

        doencas_uf = (
            doencas_uf
            .sort_values(
                "PERCENTUAL_SIM",
                ascending=False
            )
        )

        evidencias_doencas = []

        for _, linha in doencas_uf.iterrows():

            evidencias_doencas.append(
                {
                    "variavel":
                        linha["VARIAVEL"],

                    "doenca_preexistente":
                        linha[
                            "DOENCA_PREEXISTENTE"
                        ],

                    "total_registros_obito":
                        int(
                            linha[
                                "TOTAL_REGISTROS_OBITO"
                            ]
                        ),

                    "avaliaveis":
                        int(
                            linha["AVALIAVEIS"]
                        ),

                    "sim":
                        int(
                            linha["SIM"]
                        ),

                    "nao":
                        int(
                            linha["NAO"]
                        ),

                    "ausentes":
                        int(
                            linha["AUSENTES"]
                        ),

                    "percentual_sim":
                        float(
                            linha[
                                "PERCENTUAL_SIM"
                            ]
                        ),

                    "percentual_ausentes":
                        float(
                            linha[
                                "PERCENTUAL_AUSENTES"
                            ]
                        )
                }
            )

        # ============================================================
        # SINAL CLÍNICO MAIS FREQUENTE
        # ============================================================

        if (
            len(sintomas_uf) > 0
            and sintomas_uf["SIM"].sum() > 0
        ):

            sintoma_mais_frequente = (
                sintomas_uf.iloc[0][
                    "SINAL_CLINICO"
                ]
            )

            percentual_sintoma = float(
                sintomas_uf.iloc[0][
                    "PERCENTUAL_SIM"
                ]
            )

        else:

            sintoma_mais_frequente = (
                "Nenhum registrado"
            )

            percentual_sintoma = 0.0

        # ============================================================
        # DOENÇA PREEXISTENTE MAIS FREQUENTE
        # ============================================================

        if (
            len(doencas_uf) > 0
            and doencas_uf["SIM"].sum() > 0
        ):

            doenca_mais_frequente = (
                doencas_uf.iloc[0][
                    "DOENCA_PREEXISTENTE"
                ]
            )

            percentual_doenca = float(
                doencas_uf.iloc[0][
                    "PERCENTUAL_SIM"
                ]
            )

        else:

            doenca_mais_frequente = (
                "Nenhuma registrada"
            )

            percentual_doenca = 0.0

        # ====================================================
        # INTERPRETAÇÃO
        # ====================================================

        interpretacao = (
            interpretar_perfil_obitos_uf(
                uf_nome=nome_uf,
                total_registros_obito=(
                    total_registros_obito
                ),
                total_registros_uf=(
                    total_registros_uf
                ),
                percentual_obitos_total=(
                    percentual_obitos_total
                ),
                tipo_obito_predominante=(
                    tipo_obito_predominante
                ),
                total_tipo_obito_predominante=(
                    total_tipo_obito_predominante
                ),
                semana_inicial=(
                    semana_inicial
                ),
                semana_final=(
                    semana_final
                ),
                ano=ano
            )
        )
        # ============================================================
        # OBSERVAÇÕES SOBRE OS DADOS
        # ============================================================

        observacoes = [
            (
                f"Os resultados correspondem aos registros "
                f"disponíveis entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}."
            ),
            (
                "Os registros de óbito incluem as categorias "
                "óbito pelo agravo, óbito em investigação e "
                "óbito por outras causas."
            ),
            (
                "Os percentuais de sinais clínicos e doenças "
                "preexistentes são calculados sobre os registros "
                "avaliáveis de cada variável."
            ),
            (
                "A ausência de informação de sorotipo é mantida "
                "explicitamente nas evidências."
            ),
            (
                "Os resultados possuem caráter descritivo e não "
                "estabelecem relações causais entre sinais clínicos, "
                "doenças preexistentes, sorotipo e óbito."
            )
        ]

        if pequeno_numero_obitos:

          if total_registros_obito == 1:
              texto_quantidade_obitos = (
                  "1 registro classificado"
              )
          else:
              texto_quantidade_obitos = (
                  f"{total_registros_obito} registros classificados"
              )

          observacoes.append(
              (
                  f"A UF possui apenas {texto_quantidade_obitos} "
                  f"em categorias de óbito. "
                  f"Percentuais calculados sobre esse subconjunto "
                  f"devem ser interpretados com cautela, pois "
                  f"pequenas frequências absolutas podem resultar "
                  f"em percentuais elevados."
              )
          )

        # ============================================================
        # DOCUMENTO SEMÂNTICO
        # ============================================================

        documento = criar_documento_semantico(

            document_id=gerar_document_id(
                fonte="SINAN",
                doenca="dengue",
                ano=ano,
                dominio="obitos",
                localizacao=codigo_uf
            ),

            titulo=(
                f"Perfil dos registros classificados "
                f"em categorias de óbito em {nome_uf} - {ano}"
            ),

            tipo_documento="perfil_obitos_uf",

            dominio="desfechos",

            sintese=(
                f"Este documento apresenta o perfil dos registros "
                f"classificados em categorias de evolução relacionadas "
                f"a óbito no SINAN em {nome_uf}, entre as semanas "
                f"epidemiológicas {semana_inicial} e "
                f"{semana_final} de {ano}. São apresentadas "
                f"informações sobre tipo de óbito, sorotipo, "
                f"sinais clínicos e doenças preexistentes."
            ),

            indicadores={
                "semana_inicial": semana_inicial,
                "semana_final": semana_final,
                "total_semanas": semana_final - semana_inicial + 1,
                "total_registros_uf": total_registros_uf,
                "total_registros_obito": total_registros_obito,
                "pequeno_numero_registros_obito": pequeno_numero_obitos,
                "total_evolucao_avaliavel": total_evolucao_avaliavel,
                "percentual_obitos_total": percentual_obitos_total,
                "percentual_obitos_avaliaveis": percentual_obitos_avaliaveis,
                "tipo_obito_predominante": tipo_obito_predominante,
                "total_tipo_obito_predominante": total_tipo_obito_predominante,
                "sinal_clinico_mais_frequente": sintoma_mais_frequente,
                "percentual_sinal_clinico": percentual_sintoma,
                "doenca_preexistente_mais_frequente": doenca_mais_frequente,
                "percentual_doenca_preexistente": percentual_doenca
            },

            evidencias={
                "tipos_de_obito": evidencias_tipos_obito,
                "sorotipos_entre_obitos": evidencias_sorotipos,
                "sinais_clinicos_entre_obitos": evidencias_sintomas,
                "doencas_preexistentes_entre_obitos": evidencias_doencas
            },

            interpretacao=interpretacao,

            escopo={
                "codigo_uf": codigo_uf,
                "uf_nome": nome_uf,
                "ano": ano,
                "semana_inicial": semana_inicial,
                "semana_final": semana_final,
                "total_semanas": semana_final - semana_inicial + 1,
                "populacao_analitica": (
                    "registros classificados em categorias "
                    "de evolução relacionadas a óbito"
                )
            },

            observacoes_dados=observacoes,

            conceitos_semanticos=[
                "dengue",
                "óbito",
                "evolução clínica",
                "óbito pelo agravo",
                "óbito em investigação",
                "óbito por outras causas",
                "sorotipo",
                "sinal clínico",
                "doença preexistente",
                "semana epidemiológica",
                "unidade federativa",
                "vigilância epidemiológica"
            ],

            palavras_chave=[
                "dengue",
                "óbito",
                "mortalidade",
                "evolução",
                "sorotipo",
                "sinais clínicos",
                "doenças preexistentes",
                nome_uf,
                codigo_uf,
                str(ano),
                "SINAN"
            ],

            ano=ano
        )

        documentos.append(
            documento
        )

    return documentos

In [140]:
# ============================================================
# GERAR DOCUMENTOS DE PERFIL DE ÓBITOS POR UF
# ============================================================

documentos_obitos_uf = (
    criar_documentos_perfil_obitos_por_uf(
        df_obitos=
            df_obitos_uf,

        df_tipos_obito=
            df_obitos_tipo_uf,

        df_sorotipos_obito=
            df_obitos_sorotipo_uf,

        df_sintomas_obito=
            df_sintomas_obitos_uf,

        df_doencas_obito=
            df_doencas_preexistentes_obitos_uf,

        semana_inicial=
            SEMANA_INICIAL,

        semana_final=
            SEMANA_FINAL,

        ano=ANO
    )
)

print(
    "Total de documentos gerados:",
    len(documentos_obitos_uf)
)

Total de documentos gerados: 26


In [141]:
print(
    documento_para_markdown(
        documentos_obitos_uf[0]
    )
)

# Perfil dos registros classificados em categorias de óbito em Rondônia - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_OBITOS_11
- **Tipo de documento:** perfil_obitos_uf
- **Domínio:** desfechos
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Codigo uf:** 11
- **Uf nome:** Rondônia
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Populacao analitica:** registros classificados em categorias de evolução relacionadas a óbito

## Síntese epidemiológica

Este documento apresenta o perfil dos registros classificados em categorias de evolução relacionadas a óbito no SINAN em Rondônia, entre as semanas epidemiológicas 1 e 34 de 2026. São apresentadas informações sobre tipo de óbito, sorotipo, sinais clínicos e doenças preexistentes.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros uf:** 1354
- **Total registros obito:** 1
- **Pequeno numero registros

Salvar os 27 documentos de perfil de óbitos por UF

Use:

In [142]:
# ============================================================
# DIRETÓRIO DE SAÍDA - PERFIL DE ÓBITOS POR UF
# ============================================================

PASTA_DOCS_PERFIL_OBITOS_UF = (
    PASTA_DOCS_DESFECHOS
    / "perfil_obitos_por_uf"
)

PASTA_DOCS_PERFIL_OBITOS_UF.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Pasta de saída: "
    f"{PASTA_DOCS_PERFIL_OBITOS_UF}"
)

Pasta de saída: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf


In [143]:
# ============================================================
# SALVAR DOCUMENTOS - PERFIL DE ÓBITOS POR UF
# ============================================================

arquivos_perfil_obitos_uf = []

for documento in documentos_obitos_uf:

    caminhos = salvar_documento_semantico(
        documento=documento,
        pasta_saida=PASTA_DOCS_PERFIL_OBITOS_UF
    )

    arquivos_perfil_obitos_uf.append(
        {
            "document_id":
                documento["document_id"],

            "json":
                caminhos["json"],

            "markdown":
                caminhos["markdown"]
        }
    )


print(
    "Documentos semânticos salvos:",
    len(arquivos_perfil_obitos_uf)
)

print(
    "Arquivos físicos gerados:",
    len(arquivos_perfil_obitos_uf) * 2
)

Documentos semânticos salvos: 26
Arquivos físicos gerados: 52


In [144]:
for item in arquivos_perfil_obitos_uf[:3]:

    print(
        "\nID:",
        item["document_id"]
    )

    print(
        "JSON:",
        item["json"]
    )

    print(
        "Markdown:",
        item["markdown"]
    )


ID: SINAN_DENGUE_2026_OBITOS_11
JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf/sinan_dengue_2026_obitos_11.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf/sinan_dengue_2026_obitos_11.md

ID: SINAN_DENGUE_2026_OBITOS_12
JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf/sinan_dengue_2026_obitos_12.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf/sinan_dengue_2026_obitos_12.md

ID: SINAN_DENGUE_2026_OBITOS_13
JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf/sinan_dengue_2026_obitos_13.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/perfil_obitos_por_uf/sinan_dengue_2026_obitos_13.md


## 2. Agora: panorama nacional de desfechos

Aqui eu manteria separado do panorama nacional de óbitos.

O panorama nacional de desfechos deve responder principalmente por:

hospitalização;
não hospitalização;
ausência/ignorado em hospitalização;
evolução;
cura;
categorias de óbito;
ausência/ignorado em evolução.

Ele usa apenas:
```
df_hospitalizacao_brasil
df_evolucao_brasil
```

In [145]:
# ============================================================
# INTERPRETAÇÃO - PANORAMA NACIONAL DE DESFECHOS
# ============================================================

def interpretar_panorama_desfechos(
    total_registros,
    hospitalizados,
    percentual_hospitalizados,
    evolucao_predominante,
    total_evolucao_predominante,
    percentual_evolucao_predominante,
    semana_inicial,
    semana_final,
    ano
):

    return (
        f"No Brasil, foram considerados "
        f"{total_registros:,} registros de dengue entre "
        f"as semanas epidemiológicas {semana_inicial} e "
        f"{semana_final} de {ano}. "
        f"Entre os registros avaliáveis para hospitalização, "
        f"{hospitalizados:,} apresentaram hospitalização, "
        f"correspondendo a {percentual_hospitalizados:.2f}%. "
        f"Entre os registros com evolução avaliável, "
        f"{evolucao_predominante} foi a categoria mais "
        f"frequente, com "
        f"{total_evolucao_predominante:,} registros "
        f"e {percentual_evolucao_predominante:.2f}% "
        f"dos registros avaliáveis."
    )

In [146]:
# Função de criação do documento
# ============================================================
# CRIAR PANORAMA NACIONAL DE DESFECHOS
# ============================================================

def criar_panorama_desfechos_brasil(
    df_hospitalizacao,
    df_evolucao,
    semana_inicial,
    semana_final,
    ano=ANO
):

    # ========================================================
    # HOSPITALIZAÇÃO
    # ========================================================

    linha_hosp = df_hospitalizacao.iloc[0]

    total_registros = int(
        linha_hosp["TOTAL_REGISTROS"]
    )

    hospitalizacao_avaliaveis = int(
        linha_hosp["AVALIAVEIS"]
    )

    hospitalizados = int(
        linha_hosp["HOSPITALIZADOS"]
    )

    nao_hospitalizados = int(
        linha_hosp["NAO_HOSPITALIZADOS"]
    )

    ignorados_hospitalizacao = int(
        linha_hosp["IGNORADOS"]
    )

    ausentes_hospitalizacao = int(
        linha_hosp["AUSENTES"]
    )

    percentual_hospitalizados = float(
        linha_hosp["PERCENTUAL_HOSPITALIZADOS"]
    )

    percentual_nao_hospitalizados = float(
        linha_hosp["PERCENTUAL_NAO_HOSPITALIZADOS"]
    )

    percentual_ignorados_hospitalizacao = float(
        linha_hosp["PERCENTUAL_IGNORADOS"]
    )

    percentual_ausentes_hospitalizacao = float(
        linha_hosp["PERCENTUAL_AUSENTES"]
    )

    # ========================================================
    # EVOLUÇÃO
    # ========================================================

    evolucao = df_evolucao.copy()

    # ========================================================
    # EVOLUÇÃO
    # ========================================================

    evolucao = df_evolucao.copy()

    # ========================================================
    # TOTAL DE REGISTROS COM EVOLUÇÃO AVALIÁVEL
    # ========================================================

    total_evolucao_avaliavel = int(
        evolucao.loc[
            evolucao["PERCENTUAL_AVALIAVEIS"].notna(),
            "TOTAL_REGISTROS"
        ].sum()
    )

    # Categorias consideradas avaliáveis:
    # excluímos apenas a categoria explicitamente Ausente.
    evolucao_avaliavel = evolucao[
        evolucao["EVOLUCAO"] != "Ausente"
    ].copy()

    # Para identificar o desfecho clínico predominante,
    # excluímos também Ignorado.
    evolucao_informada = evolucao[
        ~evolucao["EVOLUCAO"].isin(
            [
                "Ausente",
                "Ignorado"
            ]
        )
    ].copy()

    # Categorias consideradas avaliáveis:
    # excluímos apenas a categoria explicitamente Ausente.
    evolucao_avaliavel = evolucao[
        evolucao["EVOLUCAO"] != "Ausente"
    ].copy()

    # Para identificar o desfecho clínico predominante,
    # excluímos também Ignorado.
    evolucao_informada = evolucao[
        ~evolucao["EVOLUCAO"].isin(
            [
                "Ausente",
                "Ignorado"
            ]
        )
    ].copy()

    evolucao_informada = (
        evolucao_informada
        .sort_values(
            "TOTAL_REGISTROS",
            ascending=False
        )
    )

    if len(evolucao_informada) > 0:

        linha_predominante = (
            evolucao_informada.iloc[0]
        )

        evolucao_predominante = (
            linha_predominante["EVOLUCAO"]
        )

        total_evolucao_predominante = int(
            linha_predominante[
                "TOTAL_REGISTROS"
            ]
        )

        percentual_evolucao_predominante = float(
            linha_predominante[
                "PERCENTUAL_AVALIAVEIS"
            ]
        )

    else:

        evolucao_predominante = (
            "Sem informação"
        )

        total_evolucao_predominante = 0

        percentual_evolucao_predominante = 0.0

    # ========================================================
    # EVIDÊNCIAS - HOSPITALIZAÇÃO
    # ========================================================

    evidencia_hospitalizacao = {
        "total_registros":
            total_registros,

        "avaliaveis":
            hospitalizacao_avaliaveis,

        "hospitalizados":
            hospitalizados,

        "nao_hospitalizados":
            nao_hospitalizados,

        "ignorados":
            ignorados_hospitalizacao,

        "ausentes":
            ausentes_hospitalizacao,

        "percentual_hospitalizados":
            percentual_hospitalizados,

        "percentual_nao_hospitalizados":
            percentual_nao_hospitalizados,

        "percentual_ignorados":
            percentual_ignorados_hospitalizacao,

        "percentual_ausentes":
            percentual_ausentes_hospitalizacao
    }

    # ========================================================
    # EVIDÊNCIAS - EVOLUÇÃO
    # ========================================================

    evidencias_evolucao = []

    for _, linha in evolucao.iterrows():

        evidencias_evolucao.append(
            {
                "evolucao":
                    linha["EVOLUCAO"],

                "total_registros":
                    int(
                        linha[
                            "TOTAL_REGISTROS"
                        ]
                    ),

                "percentual_total":
                    float(
                        linha[
                            "PERCENTUAL_TOTAL"
                        ]
                    ),

                "percentual_avaliaveis":
                    (
                        None
                        if pd.isna(
                            linha[
                                "PERCENTUAL_AVALIAVEIS"
                            ]
                        )
                        else float(
                            linha[
                                "PERCENTUAL_AVALIAVEIS"
                            ]
                        )
                    )
            }
        )

    # ========================================================
    # INTERPRETAÇÃO
    # ========================================================

    interpretacao = (
        interpretar_panorama_desfechos(
            total_registros=
                total_registros,

            hospitalizados=
                hospitalizados,

            percentual_hospitalizados=
                percentual_hospitalizados,

            evolucao_predominante=
                evolucao_predominante,

            total_evolucao_predominante=
                total_evolucao_predominante,

            percentual_evolucao_predominante=
                percentual_evolucao_predominante,

            semana_inicial=
                semana_inicial,

            semana_final=
                semana_final,

            ano=ano
        )
    )

    # ========================================================
    # DOCUMENTO
    # ========================================================

    documento = criar_documento_semantico(

        document_id=gerar_document_id(
            fonte="SINAN",
            doenca="dengue",
            ano=ano,
            dominio="desfechos",
            localizacao="BRASIL"
        ),

        titulo=(
            f"Panorama nacional dos desfechos "
            f"das notificações de dengue - {ano}"
        ),

        tipo_documento=
            "panorama_desfechos_nacional",

        dominio=
            "desfechos",

        sintese=(
            f"Este documento apresenta um panorama nacional "
            f"dos desfechos registrados nas notificações de "
            f"dengue do SINAN entre as semanas epidemiológicas "
            f"{semana_inicial} e {semana_final} de {ano}. "
            f"São apresentadas informações sobre "
            f"hospitalização e evolução dos registros."
        ),

        indicadores={
            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final
                - semana_inicial
                + 1,

            "total_registros":
                total_registros,

            "hospitalizacao_avaliaveis":
                hospitalizacao_avaliaveis,

            "hospitalizados":
                hospitalizados,

            "percentual_hospitalizados":
                percentual_hospitalizados,

            "nao_hospitalizados":
                nao_hospitalizados,

            "percentual_nao_hospitalizados":
                percentual_nao_hospitalizados,

            "total_evolucao_avaliavel":
                total_evolucao_avaliavel,

            "evolucao_predominante":
                evolucao_predominante,

            "evolucao_predominante":
                evolucao_predominante,

            "total_evolucao_predominante":
                total_evolucao_predominante,

            "percentual_evolucao_predominante":
                percentual_evolucao_predominante
        },

        evidencias={
            "hospitalizacao": [
                evidencia_hospitalizacao
            ],

            "evolucao":
                evidencias_evolucao
        },

        interpretacao=
            interpretacao,

        escopo={
            "localizacao":
                "Brasil",

            "ano":
                ano,

            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final
                - semana_inicial
                + 1,

            "populacao_analitica":
                (
                    "registros de dengue "
                    "notificados no SINAN"
                )
        },

        observacoes_dados=[
            (
                f"Os resultados correspondem aos registros "
                f"disponíveis entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}."
            ),
            (
                "Hospitalização e evolução possuem campos "
                "distintos no SINAN e, portanto, podem apresentar "
                "diferentes quantidades de registros avaliáveis."
            ),
            (
                "Valores ausentes e ignorados são mantidos "
                "explicitamente para preservar a informação "
                "sobre completude dos dados."
            ),
            (
                "Os percentuais devem ser interpretados "
                "considerando o denominador específico "
                "utilizado em cada indicador."
            ),
            (
                "Os resultados possuem caráter descritivo "
                "e não representam inferência causal."
            )
        ],

        conceitos_semanticos=[
            "dengue",
            "hospitalização",
            "evolução clínica",
            "cura",
            "óbito",
            "semana epidemiológica",
            "vigilância epidemiológica",
            "Brasil",
            "SINAN"
        ],

        palavras_chave=[
            "dengue",
            "hospitalização",
            "evolução",
            "cura",
            "óbito",
            "desfecho",
            "Brasil",
            str(ano),
            "SINAN"
        ],

        ano=ano
    )

    return documento

## 3. Gerar e inspecionar antes de salvar

In [147]:
documento_panorama_desfechos = (
    criar_panorama_desfechos_brasil(
        df_hospitalizacao=
            df_hospitalizacao_brasil,

        df_evolucao=
            df_evolucao_brasil,

        semana_inicial=
            SEMANA_INICIAL,

        semana_final=
            SEMANA_FINAL,

        ano=ANO
    )
)

print(
    documento_para_markdown(
        documento_panorama_desfechos
    )
)

# Panorama nacional dos desfechos das notificações de dengue - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_DESFECHOS_BRASIL
- **Tipo de documento:** panorama_desfechos_nacional
- **Domínio:** desfechos
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Localizacao:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Populacao analitica:** registros de dengue notificados no SINAN

## Síntese epidemiológica

Este documento apresenta um panorama nacional dos desfechos registrados nas notificações de dengue do SINAN entre as semanas epidemiológicas 1 e 34 de 2026. São apresentadas informações sobre hospitalização e evolução dos registros.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros:** 444266
- **Hospitalizacao avaliaveis:** 325931
- **Hospitalizados:** 27589
- **Percentual hospitalizados:** 8.46
- **Nao hospitalizados:** 298342
- **Percen

## 1. Salvar o panorama nacional de desfechos

Se ainda não criou a pasta:

In [148]:
# ============================================================
# DIRETÓRIO - PANORAMA NACIONAL DE DESFECHOS
# ============================================================

PASTA_DOCS_PANORAMA_DESFECHOS = (
    PASTA_DOCS_DESFECHOS
    / "panorama_desfechos"
)

PASTA_DOCS_PANORAMA_DESFECHOS.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Pasta de saída: "
    f"{PASTA_DOCS_PANORAMA_DESFECHOS}"
)

Pasta de saída: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/panorama_desfechos


In [149]:
# ============================================================
# SALVAR PANORAMA NACIONAL DE DESFECHOS
# ============================================================

arquivos_panorama_desfechos = (
    salvar_documento_semantico(
        documento=documento_panorama_desfechos,
        pasta_saida=PASTA_DOCS_PANORAMA_DESFECHOS
    )
)

print(
    "JSON:",
    arquivos_panorama_desfechos["json"]
)

print(
    "Markdown:",
    arquivos_panorama_desfechos["markdown"]
)

JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/panorama_desfechos/sinan_dengue_2026_desfechos_brasil.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/panorama_desfechos/sinan_dengue_2026_desfechos_brasil.md


## 2. Falta agora o panorama nacional das categorias de óbito

Eu manteria esse documento separado do panorama que acabamos de salvar. Ele terá outro objetivo analítico.

Usaremos os quatro produtos nacionais já calculados no Notebook 05:
```
df_obitos_brasil
df_obitos_sorotipo_brasil
df_sintomas_obitos_brasil
df_doencas_preexistentes_obitos_brasil
```
O documento deverá reunir quatro grupos de evidências:
```
Panorama nacional dos registros classificados
em categorias de óbito
│
├── Tipos de óbito
│   ├── Óbito pelo agravo
│   ├── Óbito em investigação
│   └── Óbito por outras causas
│
├── Sorotipos
│   ├── DENV-1
│   ├── DENV-2
│   ├── DENV-3
│   ├── DENV-4
│   └── Ausente
│
├── Sinais clínicos
│   ├── Febre
│   ├── Mialgia
│   ├── Cefaleia
│   └── ...
│
└── Doenças preexistentes
    ├── Hipertensão
    ├── Diabetes
    └── ...
```
Há uma distinção metodológica importante: o universo desse segundo panorama será de 546 registros classificados nas categorias de óbito, e não os 425.929 registros gerais.

Esses 546 são:
276  Óbito pelo agravo
175  Óbito em investigação
 95  Óbito por outras causas

546  registros

In [150]:
# ============================================================
# INTERPRETAÇÃO - PANORAMA NACIONAL DE CATEGORIAS DE ÓBITO
# ============================================================

def interpretar_panorama_obitos_brasil(
    total_registros_obito,
    tipo_obito_predominante,
    total_tipo_obito_predominante,
    percentual_tipo_obito_predominante,
    sintoma_mais_frequente,
    percentual_sintoma,
    doenca_mais_frequente,
    percentual_doenca,
    total_sorotipo_informado,
    sorotipo_predominante,
    percentual_sorotipo_predominante,
    semana_inicial,
    semana_final,
    ano
):

    texto = (
        f"No Brasil, foram identificados "
        f"{total_registros_obito:,} registros classificados "
        f"em categorias de evolução relacionadas a óbito "
        f"entre as semanas epidemiológicas {semana_inicial} "
        f"e {semana_final} de {ano}. "
        f"Entre essas categorias, {tipo_obito_predominante} "
        f"foi a mais frequente, com "
        f"{total_tipo_obito_predominante:,} registros "
        f"({percentual_tipo_obito_predominante:.2f}% "
        f"desse subconjunto). "
        f"O sinal clínico mais frequentemente registrado "
        f"como presente foi {sintoma_mais_frequente}, "
        f"com {percentual_sintoma:.2f}%, enquanto "
        f"{doenca_mais_frequente} foi a doença preexistente "
        f"mais frequentemente registrada, com "
        f"{percentual_doenca:.2f}%."
    )

    if total_sorotipo_informado > 0:

        texto += (
            f" Entre os {total_sorotipo_informado:,} registros "
            f"com sorotipo informado, "
            f"{sorotipo_predominante} foi o mais frequente, "
            f"representando "
            f"{percentual_sorotipo_predominante:.2f}% "
            f"dos sorotipos informados."
        )

    else:

        texto += (
            " Não havia registros com sorotipo informado "
            "nesse subconjunto."
        )

    return texto

In [151]:
# ============================================================
# CRIAR PANORAMA NACIONAL DOS REGISTROS EM
# CATEGORIAS DE ÓBITO
# ============================================================

def criar_panorama_obitos_brasil(
    df_obitos,
    df_sorotipos_obito,
    df_sintomas_obito,
    df_doencas_obito,
    semana_inicial,
    semana_final,
    ano=ANO
):

    # ========================================================
    # TIPOS DE ÓBITO
    # ========================================================

    obitos = (
        df_obitos
        .copy()
        .sort_values(
            "TOTAL_OBITOS",
            ascending=False
        )
    )

    total_registros_obito = int(
        obitos["TOTAL_OBITOS"].sum()
    )

    evidencias_tipos_obito = []

    for _, linha in obitos.iterrows():

        evidencias_tipos_obito.append(
            {
                "tipo_obito":
                    linha["EVOLUCAO_DECODED"],

                "total_obitos":
                    int(
                        linha["TOTAL_OBITOS"]
                    ),

                "percentual_entre_obitos":
                    float(
                        linha[
                            "PERCENTUAL_ENTRE_OBITOS"
                        ]
                    )
            }
        )

    if len(obitos) > 0:

        linha_tipo_predominante = (
            obitos.iloc[0]
        )

        tipo_obito_predominante = (
            linha_tipo_predominante[
                "EVOLUCAO_DECODED"
            ]
        )

        total_tipo_obito_predominante = int(
            linha_tipo_predominante[
                "TOTAL_OBITOS"
            ]
        )

        percentual_tipo_obito_predominante = float(
            linha_tipo_predominante[
                "PERCENTUAL_ENTRE_OBITOS"
            ]
        )

    else:

        tipo_obito_predominante = (
            "Sem informação"
        )

        total_tipo_obito_predominante = 0
        percentual_tipo_obito_predominante = 0.0

    # ========================================================
    # SOROTIPOS ENTRE REGISTROS DE ÓBITO
    # ========================================================

    sorotipos = df_sorotipos_obito.copy()

    evidencias_sorotipos = []

    for _, linha in sorotipos.iterrows():

        sorotipo = linha["SOROTIPO"]

        if pd.isna(sorotipo):
            sorotipo = "Ausente"

        evidencias_sorotipos.append(
            {
                "sorotipo":
                    sorotipo,

                "total_registros_obito":
                    int(
                        linha[
                            "TOTAL_REGISTROS_OBITO"
                        ]
                    ),

                "percentual_total_obitos":
                    float(
                        linha[
                            "PERCENTUAL_TOTAL_OBITOS"
                        ]
                    ),

                "percentual_entre_sorotipos_informados":
                    (
                        None
                        if pd.isna(
                            linha[
                                "PERCENTUAL_ENTRE_SOROTIPOS_INFORMADOS"
                            ]
                        )
                        else float(
                            linha[
                                "PERCENTUAL_ENTRE_SOROTIPOS_INFORMADOS"
                            ]
                        )
                    )
            }
        )

    # Sorotipos efetivamente informados
    sorotipos_informados = (
        sorotipos[
            sorotipos["SOROTIPO"] != "Ausente"
        ]
        .copy()
    )

    sorotipos_informados = (
        sorotipos_informados[
            sorotipos_informados[
                "PERCENTUAL_ENTRE_SOROTIPOS_INFORMADOS"
            ].notna()
        ]
    )

    total_sorotipo_informado = int(
        sorotipos_informados[
            "TOTAL_REGISTROS_OBITO"
        ].sum()
    )

    if len(sorotipos_informados) > 0:

        sorotipos_informados = (
            sorotipos_informados
            .sort_values(
                "TOTAL_REGISTROS_OBITO",
                ascending=False
            )
        )

        linha_sorotipo = (
            sorotipos_informados.iloc[0]
        )

        sorotipo_predominante = (
            linha_sorotipo["SOROTIPO"]
        )

        percentual_sorotipo_predominante = float(
            linha_sorotipo[
                "PERCENTUAL_ENTRE_SOROTIPOS_INFORMADOS"
            ]
        )

    else:

        sorotipo_predominante = (
            "Sem informação"
        )

        percentual_sorotipo_predominante = 0.0

    # ========================================================
    # SINAIS CLÍNICOS
    # ========================================================

    sintomas = (
        df_sintomas_obito
        .copy()
        .sort_values(
            "PERCENTUAL_SIM",
            ascending=False
        )
    )

    evidencias_sintomas = []

    for _, linha in sintomas.iterrows():

        evidencias_sintomas.append(
            {
                "variavel":
                    linha["VARIAVEL"],

                "sinal_clinico":
                    linha["SINAL_CLINICO"],

                "total_registros_obito":
                    int(
                        linha[
                            "TOTAL_REGISTROS_OBITO"
                        ]
                    ),

                "avaliaveis":
                    int(
                        linha["AVALIAVEIS"]
                    ),

                "sim":
                    int(
                        linha["SIM"]
                    ),

                "nao":
                    int(
                        linha["NAO"]
                    ),

                "ausentes":
                    int(
                        linha["AUSENTES"]
                    ),

                "percentual_sim":
                    float(
                        linha["PERCENTUAL_SIM"]
                    ),

                "percentual_ausentes":
                    float(
                        linha[
                            "PERCENTUAL_AUSENTES"
                        ]
                    )
            }
        )

    if (
        len(sintomas) > 0
        and sintomas["SIM"].sum() > 0
    ):

        sintoma_mais_frequente = (
            sintomas.iloc[0][
                "SINAL_CLINICO"
            ]
        )

        percentual_sintoma = float(
            sintomas.iloc[0][
                "PERCENTUAL_SIM"
            ]
        )

    else:

        sintoma_mais_frequente = (
            "Nenhum registrado"
        )

        percentual_sintoma = 0.0

    # ========================================================
    # DOENÇAS PREEXISTENTES
    # ========================================================

    doencas = (
        df_doencas_obito
        .copy()
        .sort_values(
            "PERCENTUAL_SIM",
            ascending=False
        )
    )

    evidencias_doencas = []

    for _, linha in doencas.iterrows():

        evidencias_doencas.append(
            {
                "variavel":
                    linha["VARIAVEL"],

                "doenca_preexistente":
                    linha[
                        "DOENCA_PREEXISTENTE"
                    ],

                "total_registros_obito":
                    int(
                        linha[
                            "TOTAL_REGISTROS_OBITO"
                        ]
                    ),

                "avaliaveis":
                    int(
                        linha["AVALIAVEIS"]
                    ),

                "sim":
                    int(
                        linha["SIM"]
                    ),

                "nao":
                    int(
                        linha["NAO"]
                    ),

                "ausentes":
                    int(
                        linha["AUSENTES"]
                    ),

                "percentual_sim":
                    float(
                        linha["PERCENTUAL_SIM"]
                    ),

                "percentual_ausentes":
                    float(
                        linha[
                            "PERCENTUAL_AUSENTES"
                        ]
                    )
            }
        )

    if (
        len(doencas) > 0
        and doencas["SIM"].sum() > 0
    ):

        doenca_mais_frequente = (
            doencas.iloc[0][
                "DOENCA_PREEXISTENTE"
            ]
        )

        percentual_doenca = float(
            doencas.iloc[0][
                "PERCENTUAL_SIM"
            ]
        )

    else:

        doenca_mais_frequente = (
            "Nenhuma registrada"
        )

        percentual_doenca = 0.0

    # ========================================================
    # INTERPRETAÇÃO
    # ========================================================

    interpretacao = (
        interpretar_panorama_obitos_brasil(
            total_registros_obito=
                total_registros_obito,

            tipo_obito_predominante=
                tipo_obito_predominante,

            total_tipo_obito_predominante=
                total_tipo_obito_predominante,

            percentual_tipo_obito_predominante=
                percentual_tipo_obito_predominante,

            sintoma_mais_frequente=
                sintoma_mais_frequente,

            percentual_sintoma=
                percentual_sintoma,

            doenca_mais_frequente=
                doenca_mais_frequente,

            percentual_doenca=
                percentual_doenca,

            total_sorotipo_informado=
                total_sorotipo_informado,

            sorotipo_predominante=
                sorotipo_predominante,

            percentual_sorotipo_predominante=
                percentual_sorotipo_predominante,

            semana_inicial=
                semana_inicial,

            semana_final=
                semana_final,

            ano=ano
        )
    )

    # ========================================================
    # DOCUMENTO SEMÂNTICO
    # ========================================================

    documento = criar_documento_semantico(

        document_id=gerar_document_id(
            fonte="SINAN",
            doenca="dengue",
            ano=ano,
            dominio="obitos",
            localizacao="BRASIL"
        ),

        titulo=(
            f"Panorama nacional dos registros classificados "
            f"em categorias de óbito - {ano}"
        ),

        tipo_documento=
            "panorama_obitos_nacional",

        dominio=
            "desfechos",

        sintese=(
            f"Este documento apresenta o panorama nacional "
            f"dos registros classificados em categorias de "
            f"evolução relacionadas a óbito no SINAN, entre "
            f"as semanas epidemiológicas {semana_inicial} e "
            f"{semana_final} de {ano}. São apresentados os "
            f"tipos de óbito registrados, a distribuição de "
            f"sorotipos, os sinais clínicos e as doenças "
            f"preexistentes desse subconjunto."
        ),

        indicadores={
            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final
                - semana_inicial
                + 1,

            "total_registros_obito":
                total_registros_obito,

            "tipo_obito_predominante":
                tipo_obito_predominante,

            "total_tipo_obito_predominante":
                total_tipo_obito_predominante,

            "percentual_tipo_obito_predominante":
                percentual_tipo_obito_predominante,

            "total_sorotipo_informado":
                total_sorotipo_informado,

            "sorotipo_predominante_informado":
                sorotipo_predominante,

            "percentual_sorotipo_predominante":
                percentual_sorotipo_predominante,

            "sinal_clinico_mais_frequente":
                sintoma_mais_frequente,

            "percentual_sinal_clinico":
                percentual_sintoma,

            "doenca_preexistente_mais_frequente":
                doenca_mais_frequente,

            "percentual_doenca_preexistente":
                percentual_doenca
        },

        evidencias={
            "tipos_de_obito":
                evidencias_tipos_obito,

            "sorotipos_entre_obitos":
                evidencias_sorotipos,

            "sinais_clinicos_entre_obitos":
                evidencias_sintomas,

            "doencas_preexistentes_entre_obitos":
                evidencias_doencas
        },

        interpretacao=
            interpretacao,

        escopo={
            "localizacao":
                "Brasil",

            "ano":
                ano,

            "semana_inicial":
                semana_inicial,

            "semana_final":
                semana_final,

            "total_semanas":
                semana_final
                - semana_inicial
                + 1,

            "populacao_analitica":
                (
                    "registros classificados em categorias "
                    "de evolução relacionadas a óbito"
                )
        },

        observacoes_dados=[
            (
                f"Os resultados correspondem aos registros "
                f"disponíveis entre as semanas epidemiológicas "
                f"{semana_inicial} e {semana_final} de {ano}."
            ),
            (
                "O subconjunto inclui registros classificados "
                "como óbito pelo agravo, óbito em investigação "
                "e óbito por outras causas."
            ),
            (
                "Por esse motivo, o total apresentado não deve "
                "ser interpretado exclusivamente como número de "
                "óbitos causados pela dengue."
            ),
            (
                "Os percentuais de sinais clínicos e doenças "
                "preexistentes consideram os registros avaliáveis "
                "de cada variável."
            ),
            (
                "A ausência de informação de sorotipo é mantida "
                "explicitamente nas evidências."
            ),
            (
                "Os resultados possuem caráter descritivo e não "
                "estabelecem relações causais entre sorotipo, "
                "sinais clínicos, doenças preexistentes e óbito."
            )
        ],

        conceitos_semanticos=[
            "dengue",
            "óbito",
            "evolução clínica",
            "óbito pelo agravo",
            "óbito em investigação",
            "óbito por outras causas",
            "sorotipo",
            "sinal clínico",
            "doença preexistente",
            "semana epidemiológica",
            "Brasil",
            "vigilância epidemiológica",
            "SINAN"
        ],

        palavras_chave=[
            "dengue",
            "óbito",
            "mortalidade",
            "evolução",
            "sorotipo",
            "sinais clínicos",
            "doenças preexistentes",
            "Brasil",
            str(ano),
            "SINAN"
        ],

        ano=ano
    )

    return documento

In [152]:
# ============================================================
# GERAR PANORAMA NACIONAL DAS CATEGORIAS DE ÓBITO
# ============================================================

documento_panorama_obitos = (
    criar_panorama_obitos_brasil(
        df_obitos=
            df_obitos_brasil,

        df_sorotipos_obito=
            df_obitos_sorotipo_brasil,

        df_sintomas_obito=
            df_sintomas_obitos_brasil,

        df_doencas_obito=
            df_doencas_preexistentes_obitos_brasil,

        semana_inicial=
            SEMANA_INICIAL,

        semana_final=
            SEMANA_FINAL,

        ano=ANO
    )
)

print(
    documento_para_markdown(
        documento_panorama_obitos
    )
)

# Panorama nacional dos registros classificados em categorias de óbito - 2026

## Identificação

- **ID do documento:** SINAN_DENGUE_2026_OBITOS_BRASIL
- **Tipo de documento:** panorama_obitos_nacional
- **Domínio:** desfechos
- **Fonte:** SINAN
- **Doença:** dengue
- **Ano:** 2026

## Escopo

- **Localizacao:** Brasil
- **Ano:** 2026
- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Populacao analitica:** registros classificados em categorias de evolução relacionadas a óbito

## Síntese epidemiológica

Este documento apresenta o panorama nacional dos registros classificados em categorias de evolução relacionadas a óbito no SINAN, entre as semanas epidemiológicas 1 e 34 de 2026. São apresentados os tipos de óbito registrados, a distribuição de sorotipos, os sinais clínicos e as doenças preexistentes desse subconjunto.

## Indicadores

- **Semana inicial:** 1
- **Semana final:** 34
- **Total semanas:** 34
- **Total registros obito:** 559
- **Tipo obito predomina

Essa saída ficou boa e, metodologicamente, está consistente. Eu validaria esse panorama nacional.

Os pontos principais estão corretos:

- 546 registros no subconjunto;
- 276 classificados como óbito pelo agravo, correspondendo a 50,55%;
- 100 registros com sorotipo informado;
- DENV-2 predominante entre os sorotipos informados, com 69%;
- Febre como sinal clínico mais frequente, com 76,92%;
- Hipertensão arterial como doença preexistente mais frequente, com 35,90%;
a observação deixa claro que os 546 não representam necessariamente óbitos causados pela dengue.

Agora você pode salvar esse último documento do domínio desfechos.

In [153]:
# ============================================================
# DIRETÓRIO - PANORAMA NACIONAL DE ÓBITOS
# ============================================================

PASTA_DOCS_PANORAMA_OBITOS = (
    PASTA_DOCS_DESFECHOS
    / "panorama_obitos"
)

PASTA_DOCS_PANORAMA_OBITOS.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Pasta de saída: "
    f"{PASTA_DOCS_PANORAMA_OBITOS}"
)

Pasta de saída: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/panorama_obitos


In [154]:
# ============================================================
# SALVAR PANORAMA NACIONAL DE ÓBITOS
# ============================================================

arquivos_panorama_obitos = (
    salvar_documento_semantico(
        documento=documento_panorama_obitos,
        pasta_saida=PASTA_DOCS_PANORAMA_OBITOS
    )
)

print(
    "JSON:",
    arquivos_panorama_obitos["json"]
)

print(
    "Markdown:",
    arquivos_panorama_obitos["markdown"]
)

JSON: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/panorama_obitos/sinan_dengue_2026_obitos_brasil.json
Markdown: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/desfechos/panorama_obitos/sinan_dengue_2026_obitos_brasil.md


Com isso, o domínio desfechos fica concluído com esta estrutura:
```
desfechos/
├── desfechos_por_uf/
│   └── 27 documentos
├── perfil_obitos_por_uf/
│   └── 27 documentos
├── panorama_desfechos/
│   └── 1 documento
└── panorama_obitos/
    └── 1 documento
```

Total do domínio:

- 56 documentos semânticos

- 112 arquivos físicos

## 10. Verificação estrutural e inventário final

A etapa final valida o contrato mínimo dos JSON, a correspondência com Markdown, a unicidade dos identificadores e o uso do vocabulário controlado de domínios e tipos de documento.


In [155]:
# ============================================================
# GERAR INVENTÁRIO DOS DOCUMENTOS SEMÂNTICOS
# ============================================================

def gerar_inventario_documentos(
    pasta_docs
):
    registros = []
    ignorados = []
    erros = []

    campos_obrigatorios = {
        "document_id",
        "titulo",
        "tipo_documento",
        "dominio",
        "fonte",
        "escopo",
        "sintese",
        "indicadores",
        "evidencias",
        "interpretacao",
        "observacoes_dados",
        "conceitos_semanticos",
        "palavras_chave",
    }

    for arquivo_json in sorted(
        pasta_docs.rglob("*.json")
    ):
        if "_inventario" in arquivo_json.parts:
            continue

        try:
            with open(
                arquivo_json,
                "r",
                encoding="utf-8"
            ) as arquivo:
                documento = json.load(arquivo)

            if not isinstance(documento, dict):
                ignorados.append(
                    {
                        "ARQUIVO": arquivo_json.name,
                        "CAMINHO": str(
                            arquivo_json.relative_to(
                                pasta_docs
                            )
                        ),
                        "MOTIVO": (
                            "Raiz JSON não é "
                            "objeto/dicionário"
                        ),
                    }
                )
                continue

            ausentes = sorted(
                campos_obrigatorios
                - set(documento.keys())
            )

            if ausentes:
                ignorados.append(
                    {
                        "ARQUIVO": arquivo_json.name,
                        "CAMINHO": str(
                            arquivo_json.relative_to(
                                pasta_docs
                            )
                        ),
                        "MOTIVO": (
                            "Campos obrigatórios ausentes: "
                            + ", ".join(ausentes)
                        ),
                    }
                )
                continue

            fonte = documento.get("fonte", {})
            escopo = documento.get("escopo", {})

            if not isinstance(fonte, dict):
                fonte = {}

            if not isinstance(escopo, dict):
                escopo = {}

            caminho_markdown = (
                arquivo_json.with_suffix(".md")
            )

            registros.append(
                {
                    "DOCUMENT_ID":
                        documento.get("document_id"),
                    "TITULO":
                        documento.get("titulo"),
                    "DOMINIO":
                        documento.get("dominio"),
                    "TIPO_DOCUMENTO":
                        documento.get("tipo_documento"),
                    "FONTE":
                        fonte.get("sistema"),
                    "DOENCA":
                        fonte.get("doenca"),
                    "ANO":
                        fonte.get("ano"),
                    "CODIGO_UF":
                        escopo.get("codigo_uf"),
                    "UF_NOME":
                        escopo.get("uf_nome"),
                    "LOCALIZACAO":
                        escopo.get("localizacao"),
                    "SEMANA_INICIAL":
                        escopo.get("semana_inicial"),
                    "SEMANA_FINAL":
                        escopo.get("semana_final"),
                    "TOTAL_SEMANAS":
                        escopo.get("total_semanas"),
                    "ARQUIVO_JSON":
                        str(
                            arquivo_json.relative_to(
                                pasta_docs
                            )
                        ),
                    "ARQUIVO_MARKDOWN":
                        str(
                            caminho_markdown.relative_to(
                                pasta_docs
                            )
                        ),
                    "MARKDOWN_EXISTE":
                        caminho_markdown.exists(),
                }
            )

        except Exception as erro:
            erros.append(
                {
                    "ARQUIVO": arquivo_json.name,
                    "CAMINHO": str(
                        arquivo_json.relative_to(
                            pasta_docs
                        )
                    ),
                    "ERRO": str(erro),
                }
            )

    return (
        pd.DataFrame(registros),
        pd.DataFrame(ignorados),
        pd.DataFrame(erros),
    )


(
    df_inventario_documentos,
    df_arquivos_ignorados,
    df_arquivos_com_erro,
) = gerar_inventario_documentos(
    PASTA_DOCS
)


In [156]:
# ============================================================
# VERIFICAÇÕES FINAIS E VOCABULÁRIO CONTROLADO
# ============================================================

documentos_sem_markdown = (
    df_inventario_documentos[
        ~df_inventario_documentos[
            "MARKDOWN_EXISTE"
        ]
    ]
)

ids_duplicados = (
    df_inventario_documentos[
        df_inventario_documentos[
            "DOCUMENT_ID"
        ].duplicated(
            keep=False
        )
    ]
    .sort_values("DOCUMENT_ID")
)

dominios_encontrados = set(
    df_inventario_documentos[
        "DOMINIO"
    ].dropna()
)

tipos_encontrados = set(
    df_inventario_documentos[
        "TIPO_DOCUMENTO"
    ].dropna()
)

dominios_nao_padronizados = (
    dominios_encontrados
    - DOMINIOS_VALIDOS
)

tipos_nao_padronizados = (
    tipos_encontrados
    - TIPOS_DOCUMENTO_VALIDOS
)

resumo_dominios = (
    df_inventario_documentos
    .groupby(
        "DOMINIO",
        dropna=False
    )
    .size()
    .reset_index(
        name="TOTAL_DOCUMENTOS"
    )
    .sort_values("DOMINIO")
)

resumo_tipos_documentos = (
    df_inventario_documentos
    .groupby(
        [
            "DOMINIO",
            "TIPO_DOCUMENTO",
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="TOTAL_DOCUMENTOS"
    )
    .sort_values(
        [
            "DOMINIO",
            "TIPO_DOCUMENTO",
        ]
    )
)

print("=" * 60)
print("INVENTÁRIO FINAL DOS DOCUMENTOS SEMÂNTICOS")
print("=" * 60)
print(
    "Documentos semânticos válidos:",
    len(df_inventario_documentos)
)
print(
    "JSON ignorados:",
    len(df_arquivos_ignorados)
)
print(
    "JSON com erro:",
    len(df_arquivos_com_erro)
)
print(
    "Documentos sem Markdown:",
    len(documentos_sem_markdown)
)
print(
    "IDs duplicados:",
    len(ids_duplicados)
)
print(
    "Domínios não padronizados:",
    sorted(dominios_nao_padronizados)
)
print(
    "Tipos não padronizados:",
    sorted(tipos_nao_padronizados)
)

print("\nDocumentos por domínio:")
display(resumo_dominios)

print("\nDocumentos por tipo:")
display(resumo_tipos_documentos)


INVENTÁRIO FINAL DOS DOCUMENTOS SEMÂNTICOS
Documentos semânticos válidos: 167
JSON ignorados: 0
JSON com erro: 0
Documentos sem Markdown: 0
IDs duplicados: 0
Domínios não padronizados: []
Tipos não padronizados: []

Documentos por domínio:


,DOMINIO,TOTAL_DOCUMENTOS
0,clinico,28
1,desfechos,55
2,geografico,28
3,temporal,28
4,virologico,28



Documentos por tipo:


,DOMINIO,TIPO_DOCUMENTO,TOTAL_DOCUMENTOS
0,clinico,panorama_clinico_nacional,1
1,clinico,perfil_clinico_uf,27
2,desfechos,desfechos_uf,27
3,desfechos,panorama_desfechos_nacional,1
4,desfechos,panorama_obitos_nacional,1
5,desfechos,perfil_obitos_uf,26
6,geografico,distribuicao_geografica_uf,27
7,geografico,panorama_geografico_nacional,1
8,temporal,panorama_temporal_nacional,1
9,temporal,perfil_temporal_uf,27


In [157]:
# ============================================================
# SALVAR INVENTÁRIO E RELATÓRIOS DE AUDITORIA
# ============================================================

PASTA_INVENTARIO = (
    PASTA_DOCS
    / "_inventario"
)

PASTA_INVENTARIO.mkdir(
    parents=True,
    exist_ok=True
)

df_inventario_documentos.to_csv(
    PASTA_INVENTARIO
    / f"inventario_documentos_semanticos_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_dominios.to_csv(
    PASTA_INVENTARIO
    / f"resumo_documentos_por_dominio_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_tipos_documentos.to_csv(
    PASTA_INVENTARIO
    / f"resumo_documentos_por_tipo_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

df_arquivos_ignorados.to_csv(
    PASTA_INVENTARIO
    / f"arquivos_ignorados_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

df_arquivos_com_erro.to_csv(
    PASTA_INVENTARIO
    / f"arquivos_com_erro_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

ids_duplicados.to_csv(
    PASTA_INVENTARIO
    / f"document_ids_duplicados_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

documentos_sem_markdown.to_csv(
    PASTA_INVENTARIO
    / f"documentos_sem_markdown_{ANO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Inventário e auditoria salvos em: "
    f"{PASTA_INVENTARIO}"
)


Inventário e auditoria salvos em: /content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/_inventario


In [158]:
# ============================================================
# ENCERRAMENTO DO NOTEBOOK 06
# ============================================================

consistente = (
    len(df_arquivos_ignorados) == 0
    and len(df_arquivos_com_erro) == 0
    and len(documentos_sem_markdown) == 0
    and len(ids_duplicados) == 0
    and len(dominios_nao_padronizados) == 0
    and len(tipos_nao_padronizados) == 0
)

print("=" * 60)
print("NOTEBOOK 06 - DOCUMENTOS SEMÂNTICOS")
print("=" * 60)
print(f"Ano analisado: {ANO}")
print(
    "Documentos semânticos válidos:",
    len(df_inventario_documentos)
)
print(
    "Status geral:",
    (
        "OK - conjunto estruturalmente consistente."
        if consistente
        else "ATENÇÃO - revisar inconsistências."
    )
)


NOTEBOOK 06 - DOCUMENTOS SEMÂNTICOS
Ano analisado: 2026
Documentos semânticos válidos: 167
Status geral: OK - conjunto estruturalmente consistente.
